# BA-NeuS — ULTRRA `va_split` aerial (geo-grounded, raw)

Raw **Bundle-Adjusting NeuS** on **va_split** aerial/satellite views — no MV-SAM3D Stage-1 prior.

**ULTRRA `jsons/` are position-only** (`lat/lon/alt`, no `omega/phi/kappa`, no intrinsics, **no depth maps**).
We therefore use **DA3** for orientations, intrinsics, and depth, then **Sim(3)-align** DA3 camera centers to the geodetic ENU positions from `jsons/`.

**Default workflow:** `§1` → `§2` → `§2.5` → `§8` (DA3 + geo align) → **skip §9** → `§11` → `§12` → `§13` → `§14`.

**Input (Drive):**
```
va_split/
  satellite/images/ + satellite/masks/
  jsons/<stem>.json     ← same level as satellite/ and ground/
```

**Output (Drive):** `MyDrive/SatelliteFuse/BAneus/<UTC>/<CASE>/` (raw geo BA-NeuS, **no** Stage-1). Distinct from `SatelliteFuse/SamBAneus/`.

> BlendedMVS Cathedral GT (`cams/` + `rendered_depth_maps/`) is a different format — see `baneus_cathedral_gt.ipynb`.


## 0 — Pipeline overview

| Phase | Source | Role |
|-------|--------|------|
| Masks | `satellite/masks/` | per-view building mask (pre-made) |
| Geo positions | `jsons/*.json` | RTK GPS camera locations (`lat/lon/alt`) |
| Pose / depth | **DA3** | orientations, intrinsics, depth maps (ULTRRA has no GT depth) |
| Geo align | Umeyama Sim(3) | anchor DA3 translations to JSON ENU |
| Refinement | **BA-NeuS** | RGB + DA3 depth + pose BA (`learn_pose=True`, no Stage-1 prior) |

**Why DA3?** ULTRRA train-style JSONs lack rotation and intrinsics. COLMAP/SfM is the official baseline path, but DA3 is more practical on sparse near-nadir satellite views.

**Runtime:** A100 80 GB recommended. Masks must exist under `satellite/masks/` (skip §2.5 SAM3).


## 1 — Environment & install (self-contained, fresh Colab A100-80GB)

This section is **fully self-contained** — run it top to bottom in a brand-new
Colab session. It is the exact, known-working install sequence from
`mvsam3d_example_current.ipynb`.

**Before you start:**
1. **Runtime → Change runtime type → A100 GPU** (80 GB). The SLAT backbone + DA3
   on building imagery need the headroom.
2. **Add your Hugging Face token to Colab Secrets** as `HF_TOKEN` (sidebar key
   icon, toggle notebook access). `facebook/sam-3d-objects` is a **gated** repo —
   request access on its HF page first, or checkpoint download will 401.

**Restart gotcha (important):** Cell *1.2 (Step 0)* pins `torch==2.5.1+cu121`. On
a fresh runtime Colab ships a newer torch, so this cell will downgrade it and
print *"Restart the runtime now!"*. When you see that: **Runtime → Restart
session**, then re-run **from the clone cell (1.3) downward** (skip re-running
Step 0 — torch is already correct, and it must not be re-pinned).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, subprocess, sys, os

assert torch.cuda.is_available(), "No GPU — enable a GPU runtime first!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:     {gpu_name}")
print(f"VRAM:    {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"Python:  {sys.version}")

if vram_gb < 30:
    print("\n⚠  Less than 32 GB VRAM — inference may OOM on large batches. "
          "Consider switching to an A100 runtime.")

# SAM 3D Objects targets torch 2.5.1 + CUDA 12.1.
# Colab's default torch is often too new for pre-built wheels of
# flash-attn, kaolin, spconv, etc.  Pin the correct stack now.
TARGET_TORCH = "2.5.1"
if not torch.__version__.startswith(TARGET_TORCH):
    print(f"\n▶ Downgrading torch {torch.__version__} → {TARGET_TORCH}+cu121 …")
    !pip install -q \
        torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
        --index-url https://download.pytorch.org/whl/cu121
    !pip install -q xformers==0.0.28.post3
    print("✓ Torch stack pinned.  ⚠ Restart the runtime now!")
    print("   Go to Runtime → Restart session, then re-run from this cell.")
else:
    print(f"\n✓ torch {torch.__version__} is the target version")

> **If Step 0 above printed "Restart the runtime now!":** do
> **Runtime → Restart session**, then continue from this cell (1.3). Do **not**
> re-run Step 0 after the restart. If it printed *"torch 2.5.1 is the target
> version"*, just keep going — no restart needed.

In [ ]:
# 1.1 — Define paths and clone the three repos (same remotes as the working notebook)
from pathlib import Path
import os, sys

ROOT         = Path("/content")
SAM3D_ROOT   = ROOT / "sam-3d-objects"
DA3_ROOT     = ROOT / "Depth-Anything-3"
MVSAM3D_ROOT = ROOT / "MV-SAM3D"

# Pin numpy AND scipy for EVERY pip install in this session. SAM3D + DA3 require
# numpy<2, but kaolin/scipy/etc. try to pull numpy 2.x — and upgrading numpy while it
# is already imported corrupts the live kernel ("No module named 'numpy.char' / fails
# sanity checks"). PIP_CONSTRAINT makes pip refuse to change them.
# scipy is pinned too: modern Colab ships a numpy-2-only scipy whose array-API shim
# does `from numpy import strings` (numpy>=2 only), so under numpy 1.26 `import scipy`
# explodes with "No module named 'numpy.strings'" — which crashes `import kaolin`.
# scipy 1.13.1 supports numpy 1.26 and has no such code path.
# NOTE: env vars do NOT survive a runtime restart — this cell re-sets it, so always
# re-run it (it's the first cell after Step 0) before the install cells.
_np_pin = ROOT / "_numpy_pin.txt"
_np_pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_np_pin)
print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"], "(numpy 1.26.4 + scipy 1.13.1 locked)")

# Force-align scipy with numpy 1.26.4 NOW, before anything imports scipy this session.
# (PIP_CONSTRAINT prevents future upgrades but won't downgrade the scipy Colab preinstalled.)
# Doing it here — on a fresh kernel, before any `import scipy` — avoids stale broken
# modules; otherwise kaolin's import later dies on "No module named 'numpy.strings'".
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
import importlib, scipy, numpy
importlib.reload(scipy)
print(f"✓ numpy {numpy.__version__}  scipy {scipy.__version__}")

%cd /content
REPOS = {
    "sam-3d-objects":   "https://github.com/facebookresearch/sam-3d-objects.git",
    "Depth-Anything-3": "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
    "MV-SAM3D":         "https://github.com/devinli123/MV-SAM3D.git",
}
for name, url in REPOS.items():
    if os.path.isdir(name):
        print(f"OK   {name}/ already exists")
    else:
        !git clone --depth 1 {url}
        print(f"cloned {name}")


In [ ]:
# xformers must match the installed torch+cuda version.
# Colab usually ships a compatible pair, but let's be explicit.
!pip install -q xformers

# Install DA3 as an editable package (pulls in its own deps)
!pip install -q -e /content/Depth-Anything-3

print("\n✓ Depth Anything 3 installed")

In [ ]:
import torch

TORCH_VERSION = torch.__version__.split("+")[0]   # e.g. "2.10.0"
CUDA_TAG      = torch.version.cuda.replace(".", "")  # e.g. "128"
print(f"Targeting torch {TORCH_VERSION}  CUDA {torch.version.cuda}")

# ── 4a: Install the sam3d_objects package (code only, no deps) ──
!pip install -q --no-deps -e /content/sam-3d-objects

# ── 4b: Runtime deps (open3d and utils3d conflict on numpy, install separately) ──
!pip install -q \
    hydra-core==1.3.2 \
    omegaconf \
    loguru \
    opencv-python-headless \
    trimesh \
    roma \
    seaborn \
    safetensors \
    tqdm \
    einops \
    astor \
    lightning \
    xatlas \
    pyvista \
    pymeshfix \
    igraph \
    "numpy<2" \
    "huggingface_hub[cli]" \
    gradio

# open3d and utils3d have conflicting numpy pins — install each alone
!pip install -q open3d
!pip install -q --no-deps "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900"

# ── 4c: MoGe (pointmap geometry utilities used by sam3d_objects) ──
!pip install -q --no-deps "MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b"

# The long list of "sam3d-objects requires X" warnings above is expected —
# we intentionally skipped those with --no-deps. They are dev/training deps
# not needed for inference.

print("\n✓ Core SAM 3D deps installed")

In [ ]:
import torch, subprocess, sys, os

TORCH_VER = torch.__version__.split("+")[0]            # "2.5.1"
CUDA_VER  = "cu" + torch.version.cuda.replace(".", "") # "cu121"
PY_VER    = f"{sys.version_info.major}{sys.version_info.minor}"

print(f"torch {TORCH_VER}  CUDA {torch.version.cuda}  Python {PY_VER}")
assert TORCH_VER.startswith("2.5"), (
    f"Expected torch 2.5.x but got {TORCH_VER}. "
    "Re-run Step 0 to pin the correct version, then restart the runtime."
)

# ── Re-assert the numpy/scipy pin for THIS cell (kaolin is the usual offender) ──
# Every `!pip install` below must honor numpy<2. If this cell runs on a kernel where
# PIP_CONSTRAINT isn't set (e.g. cell 1.1 wasn't re-run after a restart), `pip install
# kaolin` pulls numpy 2.x as a dependency and overwrites the numpy already loaded in
# this kernel -> 'No module named numpy.char' / 'fails sanity checks'. Recreate it here.
from pathlib import Path as _Path
_pin = _Path(os.environ.get("PIP_CONSTRAINT") or "")
if not (_pin and _pin.exists()):
    _pin = _Path("/content/_numpy_pin.txt")
    _pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_pin)
print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"], "(numpy 1.26.4 + scipy 1.13.1 locked)")

!pip install -q ninja packaging

# ── Safety check: scipy must be numpy-1.26 compatible before kaolin imports it ──
# (cell 1.1 already downgrades scipy to 1.13.1; this just verifies / repairs.)
import scipy as _scipy
if not _scipy.__version__.startswith(("1.10", "1.11", "1.12", "1.13")):
    print(f"scipy {_scipy.__version__} is too new for numpy 1.26 — fixing …")
    !pip install -q "scipy==1.13.1"
    print("scipy downgraded. If `import kaolin` below still fails, RESTART the runtime")
    print("and re-run from cell 1.1 (scipy was already imported in this kernel).")
else:
    print(f"✓ scipy {_scipy.__version__} (numpy-1.26 compatible)")

# ── PyTorch3D ──
try:
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")
except ImportError:
    print("Installing PyTorch3D …")
    ret = subprocess.run(
        f"pip install --no-index --no-cache-dir pytorch3d "
        f"-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/"
        f"py{PY_VER}_{CUDA_VER}_pyt{TORCH_VER.replace('.','')}/download.html",
        shell=True,
    )
    if ret.returncode != 0:
        print("Pre-built wheel not found — building from source (5-10 min) …")
        !pip install "pytorch3d @ git+https://github.com/facebookresearch/pytorch3d.git"
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")

# ── flash-attn (pre-built wheel exists for torch 2.5.1 + cu121) ──
try:
    import flash_attn
    print(f"✓ flash_attn {flash_attn.__version__}")
except ImportError:
    print("Installing flash-attn …")
    !pip install -q flash-attn --no-build-isolation
    import flash_attn
    print(f"✓ flash_attn {flash_attn.__version__}")

# ── kaolin (NVIDIA publishes wheels indexed by torch+cuda) ──
os.environ["PIP_FIND_LINKS"] = (
    f"https://nvidia-kaolin.s3.us-east-2.amazonaws.com/"
    f"torch-{TORCH_VER}_{CUDA_VER}.html"
)
try:
    import kaolin
    print(f"✓ kaolin {kaolin.__version__}")
except ImportError:
    print("Installing kaolin …")
    # --constraint is passed explicitly (belt-and-suspenders with PIP_CONSTRAINT) so the
    # kaolin dependency resolver can NEVER pull numpy 2.x into this live kernel.
    !pip install -q kaolin --constraint "$PIP_CONSTRAINT"
    # REPAIR numpy after kaolin's deps. Even with the version pinned to 1.26.4, kaolin's
    # dependency install can leave a MIXED numpy on disk: 1.26 metadata (so a version
    # check passes) but a numpy.random C-extension rebuilt against numpy 2.x. Importing
    # it then dies with 'numpy.dtype size changed ... Expected 96 ... got 88'. A clean
    # --force-reinstall rewrites EVERY numpy binary to a consistent 1.26.4 before import.
    !pip install -q --force-reinstall --no-deps "numpy==1.26.4"
    try:
        import kaolin
        print(f"✓ kaolin {kaolin.__version__}")
    except (ImportError, ValueError) as _e:
        raise RuntimeError(
            f"kaolin import still failing after numpy repair ({type(_e).__name__}: {_e}).\n"
            "This kernel already loaded a broken numpy, so it cannot recover in place.\n"
            "FIX: Runtime > Restart session, then run cell 1.1 FIRST, then re-run this cell.\n"
            "(The disk is now repaired, so the next run will import kaolin cleanly.)") from _e

# ── gsplat (Gaussian splatting renderer) ──
try:
    import gsplat
    print(f"✓ gsplat {getattr(gsplat, '__version__', 'ok')}")
except ImportError:
    print("Installing gsplat …")
    !pip install -q --no-build-isolation \
        "gsplat @ git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70"
    import gsplat
    print(f"✓ gsplat installed")

# ── spconv (sparse convolution, used by the 3D backbone) ──
# Two failure modes we have to defeat here, in order:
#  (1) ABI mismatch: spconv-cu121==2.3.8 is compiled against cumm 0.7.x and pins
#      cumm-cu121<0.8.0. Letting pip pull the *latest* cumm (>=0.8.0) gives a
#      core_cc that fails with 'could not convert default argument workspace:
#      tv::Tensor ... type not registered yet'. Fix: pin BOTH in one resolve.
#  (2) Bogus JIT recompile: on Colab, cumm misdetects an 'editable' install and
#      tries to recompile core_cc at import. That compile fails because cumm
#      probes /usr/local/lib/python3.12/dist-packages/include (exists but empty)
#      as its header root instead of cumm/include, so it can't find
#      tensorview/*.h. Fix: CUMM_DISABLE_JIT=1 / SPCONV_DISABLE_JIT=1 force use
#      of the prebuilt core_cc shipped in the wheel — no compile at all.
# These env vars are read once at first import of cumm.constants, so they MUST be
# set before anything imports cumm/spconv (true on a fresh kernel).
os.environ["CUMM_DISABLE_JIT"]   = "1"
os.environ["SPCONV_DISABLE_JIT"] = "1"
# Probe spconv.pytorch (NOT bare `import spconv`) — bare import is lazy and never
# loads the C extension, so it can't detect a broken binary.
try:
    import spconv.pytorch  # noqa: F401 — this is what actually loads core_cc
    print("✓ spconv installed")
except Exception:
    print("Installing spconv (pinned pair: spconv-cu121==2.3.8 + cumm-cu121==0.7.11) …")
    !pip install -q --force-reinstall "spconv-cu121==2.3.8" "cumm-cu121==0.7.11"
    # Drop any half-loaded spconv/cumm modules so the freshly installed binary is
    # picked up; if core_cc was never loaded this lets us recover without a restart.
    import sys as _sys
    for _m in [k for k in list(_sys.modules) if k == "spconv" or k.startswith("spconv.") or k == "cumm" or k.startswith("cumm.")]:
        del _sys.modules[_m]
    try:
        import spconv.pytorch  # noqa: F401
        print("✓ spconv installed (matched pair, JIT disabled)")
    except Exception as _e:
        raise RuntimeError(
            f"spconv still failing after pinning spconv-cu121==2.3.8 + cumm-cu121==0.7.11 "
            f"({type(_e).__name__}: {_e}).\n"
            "This kernel already imported a broken/JIT cumm, so it can't recover in place.\n"
            "FIX: Runtime > Restart session, then run cell 1.1 FIRST, then re-run this cell.\n"
            "(The disk now has the correct matched pair + JIT disabled, so the next run imports cleanly.)") from _e

# Final numpy repair: spconv/cumm force-reinstalls above can also re-link numpy, so do a
# clean rewrite (not a no-op version pin) to guarantee consistent 1.26.4 binaries on disk.
!pip install -q --force-reinstall --no-deps "numpy==1.26.4"
print("\n✓ Compiled extensions done (numpy force-reinstalled to a clean 1.26.4)")

In [ ]:
import os

# Colab has no CONDA_PREFIX — point CUDA_HOME to the system CUDA toolkit.
# inference.py does: os.environ["CUDA_HOME"] = os.environ["CONDA_PREFIX"]
# We pre-set both so that line doesn't crash.
cuda_home = "/usr/local/cuda"
os.environ["CUDA_HOME"]    = cuda_home
os.environ["CONDA_PREFIX"] = cuda_home
os.environ["LIDRA_SKIP_INIT"] = "true"

# Run the Hydra patch shipped with sam-3d-objects
!python /content/sam-3d-objects/patching/hydra
print("✓ Hydra patched")

### 1.7 — Hugging Face auth + checkpoints

Logs in with the `HF_TOKEN` Colab secret, then downloads the **SAM 3D Objects**
checkpoint (gated) and the **Depth-Anything-3** model. The SAM 3D download lands
in `sam-3d-objects/checkpoints/hf/` and is symlinked into
`MV-SAM3D/checkpoints/hf` in Section 9 so `pipeline.yaml` resolves.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Login programmatically, bypassing the broken widget
login(token=hf_token)

In [ ]:
from huggingface_hub import snapshot_download
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

dl_path = snapshot_download(
    repo_id="facebook/sam-3d-objects",
    token=hf_token,
    local_dir=str(SAM3D_ROOT / "checkpoints" / "hf"),
)
print(f"✓ SAM 3D checkpoint saved to {dl_path}")
!ls -lh {SAM3D_ROOT / "checkpoints" / "hf"}

In [ ]:
from huggingface_hub import snapshot_download

DA3_MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE"
da3_cache = snapshot_download(repo_id=DA3_MODEL_ID)
print(f"✓ DA3 model cached at {da3_cache}")

In [ ]:
import sys, os

os.environ.setdefault("CUDA_HOME",       "/usr/local/cuda")
os.environ.setdefault("CONDA_PREFIX",    "/usr/local/cuda")
os.environ["LIDRA_SKIP_INIT"] = "true"

# MV-SAM3D root (so `from sam3d_objects.…` resolves to MV-SAM3D's copy)
sys.path.insert(0, str(MVSAM3D_ROOT))
# MV-SAM3D notebook helpers (inference.py, load_images_and_masks.py, mesh_alignment.py)
sys.path.insert(0, str(MVSAM3D_ROOT / "notebook"))
# DA3 source directory
sys.path.insert(0, str(DA3_ROOT / "src"))

# Smoke tests
from depth_anything_3.api import DepthAnything3;    print("✓ depth_anything_3")
from inference import Inference;                     print("✓ SAM 3D Inference")
from load_images_and_masks import load_images_and_masks_from_path; print("✓ load_images_and_masks")
from sam3d_objects.utils.cross_attention_logger import CrossAttentionLogger; print("✓ CrossAttentionLogger")
from sam3d_objects.utils.latent_weighting import WeightingConfig; print("✓ LatentWeightManager")
from pytorch3d.transforms import Transform3d;       print("✓ pytorch3d")

print("\n✓ All imports OK")

In [ ]:
# 1.9 — Extra deps (cv2 already ships with the SAM 3D stack). No OSM/shapely here:
# this notebook has no footprint step, so those heavy geo deps are not installed.
!pip install -q numpy==1.26.4   # re-pin numpy AFTER any installs above

import numpy as np, cv2
print("numpy", np.__version__, "| cv2", cv2.__version__)


## 2 — Configuration

Edit the `<-- EDIT` lines. Defaults target the **MV-SAM3D + DA3 + Stage-1 prior** path on **ULTRRA `va_split` aerial views** (`satellite/images/` + `satellite/masks/`).


In [ ]:
# 2.0 — Paths and hyper-parameters  (EDIT the <-- marked lines)
from pathlib import Path
import glob, os

# ---- preprocessed bundle (optional; not the default va_split path) ----
USE_PREPROCESSED = False
PREPROCESSED_DIR = Path("/content/drive/MyDrive/ultrra_pp")  # only used when USE_PREPROCESSED=True

# ---- va_split on Drive (same layout as flowmatch_injection.ipynb / genrecon_ultrra.ipynb) ----
DATA_ROOT   = Path("/content/drive/MyDrive/SatelliteFuse/datasets/va_split")  # <-- EDIT scene root
AERIAL_DIR  = DATA_ROOT / "satellite"   # near-nadir aerial views (we do NOT use ground/ here)
JSONS_DIR   = DATA_ROOT / "jsons"       # va_split/jsons/ — lat/lon/alt per view (position-only)
IMG_SUBDIR  = "images"                  # under satellite/
MASK_SUBDIR = "masks"                   # under satellite/ (pre-made masks)

# local dev fallback
if not DATA_ROOT.exists():
    _local = Path("/Users/taiganishida/Documents/GSV-MSV/datasets/va_split")
    if _local.exists():
        DATA_ROOT = _local
        AERIAL_DIR = DATA_ROOT / "satellite"
        JSONS_DIR = DATA_ROOT / "jsons"

SCENE_DIR    = DATA_ROOT                # compatibility alias for export / CASE_NAME
CASE_NAME    = DATA_ROOT.name           # "va_split"
SELECT_VIEWS = None                     # optional subset: ["00001", "00050.jpg", 3]; None = all aerial views

VIEW_DIR     = Path("/content/baneus_ultrra_out") / "views"   # MV-SAM3D layout: images/ + masks/
OUTPUT_DIR   = Path("/content/baneus_ultrra_out"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
from datetime import datetime, timezone
RUN_TS       = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
DRIVE_ROOT   = Path("/content/drive/MyDrive/SatelliteFuse/BAneus")
DRIVE_OUT    = DRIVE_ROOT / RUN_TS / CASE_NAME

def _latest_baneus_drive(case=CASE_NAME):
    if not DRIVE_ROOT.exists():
        return None
    cands = sorted([p for p in DRIVE_ROOT.iterdir() if p.is_dir() and (p / case).exists()], reverse=True)
    return (cands[0] / case) if cands else None
DRIVE_OUT_LATEST = _latest_baneus_drive()
MASK_NAME    = MASK_SUBDIR
TEXT_PROMPT  = "building"               # <-- SAM3 text prompt (only if masks missing)
SKIP_SAM3_IF_MASKS_EXIST = True         # True = use satellite/masks/ as-is when complete

# ---- MV-SAM3D Stage-1 (coarse topology prior) ----
SEED              = 0
STAGE1_STEPS      = 25
MODE              = "multidiffusion"
SS_WEIGHTING      = True
SS_ENTROPY_ALPHA  = 60.0
SS_WARMUP_STEPS   = 1
DA3_PROCESS_RES   = 504

# ---- DA3 camera noise (applied in §8 on load; 0 = off) ----
# Independent Gaussian jitter per view. Units below — see §8 printout for scene-relative %.
DA3_NOISE_SEED          = None   # None = nondeterministic; int (e.g. SEED) for reproducibility
DA3_NOISE_TRANS_STD     = 0.0    # world units (DA3 metric-ish frame): σ on c2w translation (each xyz axis)
DA3_NOISE_ROT_STD_DEG   = 0.0    # degrees: σ of random axis-angle rotation (left-multiply R)
DA3_NOISE_FOCAL_STD_FRAC = 0.0   # unitless relative σ: fx,fy *= (1 + N(0,σ)); 0.01 ≈ 1% focal length
DA3_NOISE_PP_STD_PX     = 0.0    # pixels at DA3 process_res: σ on principal point cx, cy

# ---- BA-NeuS training (aligned to Stage-1 frame) ----
BANEUS_ROOT       = Path("/content/bundle-adjusting-neus")
# CASE_NAME set above from DATA_ROOT.name
BANEUS_END_ITER   = 80_000
BANEUS_SAVE_FREQ  = 5_000           # checkpoint every N iters (resume after Colab kill)
BANEUS_VAL_FREQ   = 40_000          # image/pose validation (was 5k — OOM risk ~10 min in)
BANEUS_VAL_MESH   = 80_000          # mesh extract once at end (was 10k — VRAM spike)
BANEUS_INIT_POSE  = True          # seed from DA3 extrinsics (exported in §12)
BANEUS_LEARN_POSE = True          # refine DA3 cameras during BA-NeuS (bundle adjustment)
# ---- Geo-grounding: JSON positions + DA3 orientations/depth ----
ALIGN_DA3_TO_JSON_GEO = True        # Sim(3)-align DA3 camera centers -> jsons/ ENU positions
MIN_GEO_VIEWS_FOR_ALIGN = 3

# ---- Pose audit (compare vs jsons/ geodetic positions) ----
POSE_AUDIT_DIR   = OUTPUT_DIR / "pose_audit"
POSE_LOG_FREQ    = 5_000
BANEUS_POSE_NOISE = 0.0

# ---- Raw BA-NeuS: no MV-SAM3D Stage-1 occupancy prior ----
SKIP_STAGE1          = True         # skip §9 (no coarse mesh prior)
BANEUS_USE_PRIOR     = False
BANEUS_PRIOR_W       = 0.0
GT_NATIVE            = False        # geo JSON path not applicable to ULTRRA
# ---- BA-NeuS multi-view losses (align with stock ba_noisy_poses.conf + mvs when possible) ----
BANEUS_TYPE_SOURCES  = "sequential"   # va_split has no BlendedMVS pair.txt / cam4feat
BANEUS_PC_WEIGHT     = [0.0, 0.1, 0.01]
BANEUS_FEAT_WEIGHT   = [0.0, 0.01, 0.005] if BANEUS_TYPE_SOURCES == "mvs" else [0.0, 0.0, 0.0]
BANEUS_PHASE_DELIM   = [0.2, 0.6]
BANEUS_DEPTH_WEIGHT  = 0.1   # stock BA-NeuS depth_weight (DA3 undistortion handles scale)

print("POSE_AUDIT:", POSE_AUDIT_DIR, "| geo_align:", ALIGN_DA3_TO_JSON_GEO)

print("DATA_ROOT:", DATA_ROOT, "| aerial:", AERIAL_DIR)
print("VIEW_DIR:", VIEW_DIR, "| masks:", AERIAL_DIR / MASK_SUBDIR)
print("Drive out (this run):", DRIVE_OUT)
print("Drive latest (restore):", DRIVE_OUT_LATEST)
print(f"Stage1 steps={STAGE1_STEPS} | BA-NeuS iters={BANEUS_END_ITER} | prior_w={BANEUS_PRIOR_W} | learn_pose={BANEUS_LEARN_POSE}")
print(f"USE_PREPROCESSED={USE_PREPROCESSED} | PREPROCESSED_DIR={PREPROCESSED_DIR}")
print(f"GT_NATIVE={GT_NATIVE} | use_prior={BANEUS_USE_PRIOR} | depth_weight={BANEUS_DEPTH_WEIGHT}")
print(f"type_sources={BANEUS_TYPE_SOURCES} | pc_weight={BANEUS_PC_WEIGHT} | feat_weight={BANEUS_FEAT_WEIGHT}")
print(f"DA3 noise: trans={DA3_NOISE_TRANS_STD} rot={DA3_NOISE_ROT_STD_DEG}° focal_frac={DA3_NOISE_FOCAL_STD_FRAC} pp={DA3_NOISE_PP_STD_PX}px")

# ---- Restore critical artifacts from Drive after runtime reset ----
if Path("/content/drive/MyDrive").exists():
    _restore_from = DRIVE_OUT if DRIVE_OUT.is_dir() else DRIVE_OUT_LATEST
else:
    _restore_from = None
if _restore_from is not None and Path(_restore_from).is_dir():
    import shutil as _shutil
    DRIVE_OUT_RESTORE = Path(_restore_from)
    print("Restore source:", DRIVE_OUT_RESTORE)
    _restore_map = {
        "baneus_scene_meta.npz":     OUTPUT_DIR,
        "per_view_points.json":      OUTPUT_DIR,
    }
    _restored = []
    for _fn, _dest in _restore_map.items():
        _src = DRIVE_OUT_RESTORE / _fn
        _dst = _dest / _fn
        if _src.exists() and not _dst.exists():
            _dest.mkdir(parents=True, exist_ok=True)
            _shutil.copy2(_src, _dst)
            _restored.append(_fn)
    # Restore stage1 dir
    _drive_s1 = DRIVE_OUT_RESTORE / "stage1"
    if _drive_s1.is_dir():
        _local_s1 = OUTPUT_DIR / "stage1"
        _local_s1.mkdir(parents=True, exist_ok=True)
        for _f in _drive_s1.iterdir():
            if _f.is_file() and not (_local_s1 / _f.name).exists():
                _shutil.copy2(_f, _local_s1 / _f.name)
                _restored.append(f"stage1/{_f.name}")
    # Restore pose audit
    _drive_pa = DRIVE_OUT_RESTORE / "pose_audit"
    if _drive_pa.is_dir():
        POSE_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        (POSE_AUDIT_DIR / "snapshots").mkdir(parents=True, exist_ok=True)
        for _f in _drive_pa.glob("*"):
            if _f.is_file() and not (POSE_AUDIT_DIR / _f.name).exists():
                _shutil.copy2(_f, POSE_AUDIT_DIR / _f.name)
                _restored.append(f"pose_audit/{_f.name}")
        for _f in (_drive_pa / "snapshots").glob("*"):
            if _f.is_file() and not (POSE_AUDIT_DIR / "snapshots" / _f.name).exists():
                _shutil.copy2(_f, POSE_AUDIT_DIR / "snapshots" / _f.name)
                _restored.append(f"pose_audit/snapshots/{_f.name}")
    if _restored:
        print(f"Restored {len(_restored)} file(s) from Drive: {_restored[:10]}{'...' if len(_restored)>10 else ''}")
    else:
        print("All local artifacts present (nothing to restore from Drive).")


## 2.1 — Deploy preprocessed bundle (optional)

Run this **instead of** §2.5, §8, §9, and §12 when `USE_PREPROCESSED=True`.

Expects `ultrra_pp/` on Drive (or local path) with:
`public_data/va_split/`, `views/`, `meta/`, `view_names.txt`.


In [ ]:
# 2.1 — Deploy ultrra_pp (GT-native preprocessed bundle)
import json, shutil
from pathlib import Path
import numpy as np

if not USE_PREPROCESSED:
    print("Skipping §2.1 — USE_PREPROCESSED=False (run §2.5 → §8 → §9 → §12 for MV-SAM3D path)")
else:
    assert PREPROCESSED_DIR.is_dir(), f"Missing {PREPROCESSED_DIR} — upload ultrra_pp to Drive"

    pp_pub = PREPROCESSED_DIR / "public_data" / CASE_NAME
    pp_meta = PREPROCESSED_DIR / "meta"
    pp_views = PREPROCESSED_DIR / "views"
    assert pp_pub.is_dir() and (pp_pub / "cameras_sphere.npz").exists(), f"Bad bundle: {pp_pub}"

    # staged views for pose audit / optional preview
    VIEW_DIR.mkdir(parents=True, exist_ok=True)
    if pp_views.is_dir():
        for sub in ("images", "masks"):
            src, dst = pp_views / sub, VIEW_DIR / sub
            if src.is_dir():
                dst.mkdir(parents=True, exist_ok=True)
                for f in src.glob("*"):
                    if f.is_file():
                        t = dst / f.name
                        if not t.exists() or f.stat().st_mtime > t.stat().st_mtime:
                            shutil.copy2(f, t)
    IMAGES_DIR = VIEW_DIR / "images"
    ( VIEW_DIR / MASK_NAME ).mkdir(parents=True, exist_ok=True)

    # view list
    vn_path = PREPROCESSED_DIR / "view_names.txt"
    if vn_path.exists():
        _stems = [ln.strip() for ln in vn_path.read_text().splitlines() if ln.strip()]
    else:
        _meta = np.load(pp_meta / "baneus_scene_meta.npz", allow_pickle=True)
        _stems = [str(s) for s in _meta["view_names"]]
    VIEW_MANIFEST = [(s, PREPROCESSED_DIR / "views" / "images" / f"{s}.png", None) for s in _stems]
    view_names = list(_stems)

    # meta -> OUTPUT_DIR (normalization frame for §14 mesh export)
    for name in ("baneus_scene_meta.npz", "neus_export_baseline.npz", "preprocess_report.json"):
        src = pp_meta / name
        if src.exists():
            shutil.copy2(src, OUTPUT_DIR / name)
    shutil.copytree(pp_meta, OUTPUT_DIR / "preprocessed_meta", dirs_exist_ok=True)

    # pose audit dir + baseline snapshot
    POSE_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
    bl = OUTPUT_DIR / "neus_export_baseline.npz"
    if not bl.exists() and (pp_meta / "neus_export_baseline.npz").exists():
        shutil.copy2(pp_meta / "neus_export_baseline.npz", bl)

    # deploy NeuS dataset after §11 clones BA-NeuS (or now if repo exists)
    def deploy_preprocessed_to_baneus():
        dst = BANEUS_ROOT / "public_data" / CASE_NAME
        dst.mkdir(parents=True, exist_ok=True)
        for sub in ("image", "mask", "depth"):
            (dst / sub).mkdir(parents=True, exist_ok=True)
        for item in pp_pub.iterdir():
            if item.is_file():
                shutil.copy2(item, dst / item.name)
            elif item.is_dir():
                for f in item.iterdir():
                    if f.is_file():
                        t = dst / item.name / f.name
                        if not t.exists() or f.stat().st_mtime > t.stat().st_mtime:
                            shutil.copy2(f, t)
        return dst

    if BANEUS_ROOT.exists():
        _dst = deploy_preprocessed_to_baneus()
        print("Deployed public_data ->", _dst)
    else:
        print("BA-NeuS not cloned yet — will deploy in §11.2 after §11.1")

    _rep = pp_meta / "preprocess_report.json"
    if _rep.exists():
        r = json.loads(_rep.read_text())
        print(f"Bundle: {r.get('n_views')} views | export_ok={r.get('export_ok')} | radius={r.get('radius', 0):.2f}")
        if not r.get("export_ok"):
            print("⚠ preprocess_report says export_ok=False — fix bundle before training")
    print(f"Views: {len(view_names)} | IMAGES_DIR={IMAGES_DIR}")

## 2.5 — Stage `va_split` aerial images + masks

> **Skip when `USE_PREPROCESSED=True`** — masks/images already in a preprocessed bundle.

Stage RGB from `satellite/images/` and copy existing `satellite/masks/` (SAM3 only fills gaps). Only needed on the full (non-preprocessed) path.


In [ ]:
# 2.5.1 — Install SAM3 (segmentation) and fix its tokenizer asset path
!pip install -q --no-deps sam3
!pip install -q decord ftfy iopath
!pip install -q numpy==1.26.4          # keep numpy pinned (PIP_CONSTRAINT enforces this too)

# SAM3's text encoder loads a BPE vocab from dist-packages/assets/. Locate & link it.
import glob, os, urllib.request
_target = "/usr/local/lib/python3.12/dist-packages/assets"
os.makedirs(_target, exist_ok=True)
_dst = f"{_target}/bpe_simple_vocab_16e6.txt.gz"
if os.path.exists(_dst):
    print("BPE vocab already present.")
else:
    _hits = (glob.glob("/usr/local/lib/python3.12/dist-packages/sam3/**/bpe_simple*", recursive=True)
             or glob.glob("/usr/local/lib/python3.12/dist-packages/**/bpe_simple_vocab_16e6*", recursive=True))
    if _hits:
        os.symlink(_hits[0], _dst); print("Linked BPE vocab from", _hits[0])
    else:
        urllib.request.urlretrieve(
            "https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", _dst)
        print("Downloaded BPE vocab ->", _dst)

In [ ]:
# 2.5.2 — SAM3 segmenter (one shared, cached processor for picker + batch run)
import torch, numpy as np
from pathlib import Path
from PIL import Image
from loguru import logger

_sam3_model_cache = {}   # picker (_preview_mask) and the segmenter both read this

def _get_sam3_processor(confidence_threshold: float = 0.1):
    """Build (once) and cache a SAM3 image processor; weights stream from HF."""
    if "processor" not in _sam3_model_cache:
        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor
        print("Loading SAM3 (one-time; first run downloads ~3.4 GB)...")
        model = build_sam3_image_model(load_from_HF=True, checkpoint_path=None)
        _sam3_model_cache["processor"] = Sam3Processor(model, confidence_threshold=confidence_threshold)
        print("SAM3 ready.")
    return _sam3_model_cache["processor"]


class SAM3MultiObjectSegmenter:
    """Segment one object across many views; optional per-view point disambiguation."""

    def __init__(self, processor=None, confidence_threshold: float = 0.1):
        self.processor = processor or _get_sam3_processor(confidence_threshold)

    @staticmethod
    def _select_mask_by_points(masks, points, labels, image_size):
        w, h = image_size
        best_idx, best_score = 0, -float("inf")
        for mi in range(len(masks)):
            m = masks[mi]
            m_np = m.squeeze(0).cpu().numpy() if torch.is_tensor(m) else (m.squeeze(0) if m.ndim > 2 else m)
            if m_np.shape != (h, w):
                m_pil = Image.fromarray((m_np * 255).astype(np.uint8)).resize((w, h), Image.NEAREST)
                m_np = np.array(m_pil) / 255.0
            m_bool = m_np > 0.5
            score = 0
            for (px, py), lbl in zip(points, labels):
                inside = m_bool[max(0, min(int(py), h - 1)), max(0, min(int(px), w - 1))]
                score += 1 if (lbl == 1) == bool(inside) else -1
            if score > best_score:
                best_score, best_idx = score, mi
        return best_idx

    def segment_object_multiview(self, images_dir, object_name, text_prompt, output_dir,
                                 per_view_points=None, image_names=None):
        images_dir, output_dir = Path(images_dir), Path(output_dir)
        logger.info(f"[Segmenting: {object_name}]  prompt='{text_prompt}'")
        if per_view_points:
            logger.info(f"  Per-view point prompts for {len(per_view_points)} view(s)")

        def _key(p):
            try:    return (0, int(p.stem), p.stem)
            except ValueError: return (1, 0, p.stem)

        all_files = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")), key=_key)
        # Map stem -> index in the FULL list so picker point keys stay valid even when we
        # only re-segment a subset of views below.
        full_index = {p.stem: idx for idx, p in enumerate(all_files)}
        image_files = all_files
        if image_names is not None:   # restrict to requested stems (e.g. only missing masks)
            _want = set(image_names)
            image_files = [p for p in all_files if p.stem in _want]
        if not image_files:
            return {"success": False, "error": "No images found", "success_views": 0, "total_views": 0}
        logger.info(f"  Processing {len(image_files)} views...")

        mask_dir = output_dir / object_name
        mask_dir.mkdir(parents=True, exist_ok=True)
        success, failed, disagree = 0, [], []

        for img_path in image_files:
            i = full_index[img_path.stem]   # index into the picker's full-list keys
            try:
                image = Image.open(img_path).convert("RGB")
                state = self.processor.set_image(image)
                out = self.processor.set_text_prompt(state=state, prompt=text_prompt)
                masks, scores = out["masks"], out["scores"]
                if len(masks) == 0:
                    logger.warning(f"  View {i}: no mask (confidence too low)"); failed.append(i); continue

                vp = per_view_points.get(i) if per_view_points else None
                if vp:
                    # Prefer an actual point-conditioned prediction; the text candidates may not
                    # even contain the object we want (e.g. 'building' locks onto a neighbour).
                    try:
                        out_p = self.processor.set_point_prompt(
                            state=state, points=vp["points"], point_labels=vp["labels"],
                            add_to_existing=True,
                        )
                        if len(out_p["masks"]):
                            masks, scores = out_p["masks"], out_p["scores"]
                    except Exception as e:
                        logger.warning(f"  View {i}: point prompt unsupported ({e}) — re-ranking text masks")
                    best_idx = self._select_mask_by_points(masks, vp["points"], vp["labels"], image.size)
                else:
                    best_idx = scores.argmax().item()

                best_mask, best_score = masks[best_idx], scores[best_idx].item()
                m_np = best_mask.squeeze(0).cpu().numpy() if torch.is_tensor(best_mask) else best_mask.squeeze(0)
                if m_np.shape != (image.size[1], image.size[0]):
                    m_np = np.array(Image.fromarray((m_np * 255).astype(np.uint8)).resize(image.size, Image.NEAREST)) / 255.0

                image_np, mask_bool = np.array(image), m_np > 0.5
                rgba = np.zeros((image_np.shape[0], image_np.shape[1], 4), dtype=np.uint8)
                rgba[mask_bool, :3] = image_np[mask_bool]
                rgba[mask_bool, 3] = 255
                Image.fromarray(rgba, "RGBA").save(mask_dir / f"{img_path.stem}.png")

                area = float((rgba[:, :, 3] > 0).mean())
                _note = ""
                if vp:
                    _h, _w = mask_bool.shape
                    _bad = sum(1 for (px, py), l in zip(vp["points"], vp["labels"])
                               if bool(mask_bool[min(max(int(py), 0), _h - 1),
                                                 min(max(int(px), 0), _w - 1)]) != (l == 1))
                    if _bad:
                        disagree.append(i)
                        _note = f"  [WARN {_bad}/{len(vp['labels'])} points disagree]"
                logger.info(f"  View {i}: {img_path.stem}.png  area={area*100:.1f}%  score={best_score:.3f}{_note}")
                success += 1
            except Exception as e:
                logger.error(f"  View {i}: failed - {e}"); failed.append(i)

        logger.info(f"  Result: {success}/{len(image_files)} views segmented")
        if failed:
            logger.warning(f"  Failed views: {failed}")
        if disagree:
            logger.warning(f"  Mask ignores the clicks in {len(disagree)} view(s): {disagree[:20]}"
                           f"{' ...' if len(disagree) > 20 else ''} — add clicks there in §2.5.4")
        return {"success": success > 0, "object_name": object_name, "total_views": len(image_files),
                "success_views": success, "failed_views": failed, "disagree_views": disagree,
                "mask_dir": mask_dir}

print("SAM3MultiObjectSegmenter ready.")

In [ ]:
# 2.5.3 — Stage va_split aerial views into views/images/ + views/masks/
# Layout (same as flowmatch_injection.ipynb):
#   va_split/satellite/images/<stem>.jpg
#   va_split/satellite/masks/<stem>.png
import shutil, os
from pathlib import Path

_exts = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG", ".webp")

def _img_stem(p):
    return Path(p).stem.replace("_masked", "")

def _natural_sort_key(p):
    try:
        return (0, int(Path(p).stem), Path(p).stem)
    except ValueError:
        return (1, 0, Path(p).stem)

def _select_views(paths, select):
    if not select:
        return sorted(paths, key=_natural_sort_key)
    want_str, want_int = set(), set()
    for s in select:
        if isinstance(s, int):
            want_int.add(int(s))
        else:
            k = _img_stem(str(s)); want_str.add(k); want_str.add(os.path.basename(str(s)))
            if k.isdigit():
                want_int.add(int(k))
    out = []
    for p in sorted(paths, key=_natural_sort_key):
        k = _img_stem(p)
        if k in want_str or Path(p).name in want_str or (k.isdigit() and int(k) in want_int):
            out.append(p)
    return out

def _resolve_img_dir(mod_root):
    for cand in (IMG_SUBDIR, "images", ""):
        d = mod_root / cand if cand else mod_root
        if d.is_dir() and (list(d.glob("*.jpg")) or list(d.glob("*.png"))):
            return d
    raise FileNotFoundError(f"No images under {mod_root} (expected {mod_root}/{IMG_SUBDIR}/)")

def stage_va_split_aerial_images(aerial_dir, view_dir, mask_name="masks", select=None):
    aerial_dir, view_dir = Path(aerial_dir), Path(view_dir)
    img_dir = _resolve_img_dir(aerial_dir)
    src_mask_dir = aerial_dir / mask_name
    src_paths = _select_views([p for p in img_dir.iterdir() if p.suffix.lower() in _exts and p.is_file()], select)
    assert src_paths, f"No aerial images matched under {img_dir} — check DATA_ROOT / SELECT_VIEWS"

    images_dir = view_dir / "images"
    mask_dir = view_dir / mask_name
    images_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    manifest = []
    img_copied = mask_copied = 0
    missing_masks = []
    for src in src_paths:
        stem = _img_stem(src.name)
        dst_img = images_dir / f"{stem}{src.suffix.lower()}"
        if not dst_img.exists() or src.stat().st_mtime > dst_img.stat().st_mtime:
            shutil.copy2(src, dst_img); img_copied += 1

        src_mask = None
        if src_mask_dir.is_dir():
            for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG"):
                cand = src_mask_dir / f"{stem}{ext}"
                if cand.exists():
                    src_mask = cand; break
        if src_mask is not None:
            dst_mask = mask_dir / f"{stem}.png"
            if not dst_mask.exists() or src_mask.stat().st_mtime > dst_mask.stat().st_mtime:
                shutil.copy2(src_mask, dst_mask); mask_copied += 1
        else:
            missing_masks.append(stem)

        manifest.append((stem, src, dst_img, src_mask))

    print(f"Aerial images from : {img_dir}")
    print(f"Source masks dir   : {src_mask_dir} ({'found' if src_mask_dir.is_dir() else 'missing'})")
    print(f"Staged {len(manifest)} view(s) -> {images_dir} (copied {img_copied} new image(s))")
    print(f"Copied {mask_copied} mask(s) -> {mask_dir}")
    if missing_masks:
        print(f"  {len(missing_masks)} view(s) missing masks (SAM3 in §2.5.5 will fill): {missing_masks[:8]}{'...' if len(missing_masks)>8 else ''}")
    if manifest:
        print(f"  sample: {manifest[0][0]} <- {manifest[0][1].name}")
    return images_dir, len(manifest), manifest

VIEW_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR, _nv, VIEW_MANIFEST = stage_va_split_aerial_images(AERIAL_DIR, VIEW_DIR, MASK_NAME, SELECT_VIEWS)
assert _nv > 0, f"No aerial images staged from {AERIAL_DIR}"


### 2.5.4 — Pick the right building (point prompts)

SAM3's text prompt alone returns *a* building, not necessarily **your target building**. Click a few views
here to disambiguate:

- **Left click** = positive (this is the object)
- **Right click** = negative (not this object)
- **Preview Mask** = run SAM3 on the current view with your clicks (red overlay)
- Hit **Save All to Python** when done -> fills `PER_VIEW_POINTS`

You only need a handful of views. §8.5 back-projects these clicks through DA3 depth and propagates
them to **all** views automatically, so 3-5 well-spread views is usually enough.
Views with no points fall back to the highest-scoring text-prompt mask.


In [ ]:
# 2.5.4 — Interactive per-view point picker (writes PER_VIEW_POINTS)
import base64, io, json
from pathlib import Path
from IPython.display import display, HTML
from google.colab import output
from PIL import Image as PILImage


def _natural_sort_key(p):
    try:
        return (0, int(p.stem), p.stem)
    except ValueError:
        return (1, 0, p.stem)


# Same ordering the segmenter uses, so point keys line up with view indices.
image_files = sorted(list(IMAGES_DIR.glob("*.png")) + list(IMAGES_DIR.glob("*.jpg")),
                     key=_natural_sort_key)
N_IMAGES = len(image_files)

MAX_W = 800
img_data, img_dims = [], []
for f in image_files:
    im = PILImage.open(f)
    ow, oh = im.size
    img_dims.append((ow, oh))
    if ow > MAX_W:
        im = im.resize((MAX_W, int(oh * MAX_W / ow)), PILImage.LANCZOS)
    buf = io.BytesIO()
    im.convert("RGB").save(buf, format="JPEG", quality=70)
    img_data.append(base64.b64encode(buf.getvalue()).decode())

print(f"Loaded {N_IMAGES} images into viewer")

PER_VIEW_POINTS = {}
_names_json = json.dumps([f.name for f in image_files])
_dims_json = json.dumps(img_dims)
_imgs_json = json.dumps(img_data)


def _receive_all_points(data_json):
    global PER_VIEW_POINTS
    raw = json.loads(data_json)
    PER_VIEW_POINTS = {}
    for k, v in raw.items():
        if v:
            PER_VIEW_POINTS[int(k)] = {"points": [[p["x"], p["y"]] for p in v],
                                       "labels": [p["l"] for p in v]}
    print(f"Saved {len(PER_VIEW_POINTS)}/{N_IMAGES} views with point prompts")
    for k in sorted(PER_VIEW_POINTS):
        v = PER_VIEW_POINTS[k]
        s = ", ".join(f"{'+' if l == 1 else '-'}({p[0]},{p[1]})"
                      for p, l in zip(v["points"], v["labels"]))
        print(f"  [{k}] {image_files[k].name}: {s}")
    _p = OUTPUT_DIR / "per_view_points.json"
    _p.write_text(json.dumps({str(k): v for k, v in PER_VIEW_POINTS.items()}))
    print("Saved ->", _p)
    if Path("/content/drive/MyDrive").exists():
        import shutil
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(_p, DRIVE_OUT / _p.name)
        print("Backed up to Drive ->", DRIVE_OUT / _p.name)


def _preview_mask(img_idx, points_json):
    """Run SAM3 on one view with current clicks; return red overlay as base64 JPEG."""
    import io
    import numpy as np
    import torch
    from PIL import Image as PILImg

    img_idx = int(img_idx)
    pts_raw = json.loads(points_json)
    if not pts_raw:
        return ""

    pts = [[p["x"], p["y"]] for p in pts_raw]
    lbls = [p["l"] for p in pts_raw]

    proc = _get_sam3_processor()
    img = PILImg.open(image_files[img_idx]).convert("RGB")
    w, h = img.size

    state = proc.set_image(img)
    out = proc.set_text_prompt(state=state, prompt=TEXT_PROMPT)
    masks, scores = out["masks"], out["scores"]
    if len(masks) == 0:
        return ""

    try:
        out_p = proc.set_point_prompt(
            state=state, points=pts, point_labels=lbls, add_to_existing=True,
        )
        if len(out_p["masks"]):
            masks, scores = out_p["masks"], out_p["scores"]
    except Exception:
        pass

    seg = SAM3MultiObjectSegmenter(processor=proc)
    best_idx = seg._select_mask_by_points(masks, pts, lbls, (w, h))

    best_m = masks[best_idx]
    m_np = best_m.squeeze(0).cpu().numpy() if torch.is_tensor(best_m) else best_m.squeeze(0)
    if m_np.shape != (h, w):
        m_np = np.array(PILImg.fromarray((m_np * 255).astype(np.uint8)).resize((w, h), PILImg.NEAREST)) / 255.0

    img_np = np.array(img)
    overlay = img_np.copy()
    mask_bool = m_np > 0.5
    overlay[mask_bool] = (overlay[mask_bool] * 0.5 + np.array([255, 80, 80]) * 0.5).astype(np.uint8)

    thumb = PILImg.fromarray(overlay)
    if w > MAX_W:
        thumb = thumb.resize((MAX_W, int(h * MAX_W / w)), PILImg.LANCZOS)
    buf = io.BytesIO()
    thumb.save(buf, format="JPEG", quality=75)
    return base64.b64encode(buf.getvalue()).decode()


output.register_callback("_receive_all_points", _receive_all_points)
output.register_callback("_preview_mask", _preview_mask)

html = f"""
<style>
#vw{{position:relative;display:inline-block;cursor:crosshair}}
#vw img{{display:block;max-width:900px}}
.mk{{position:absolute;width:18px;height:18px;border-radius:50%;border:2px solid #fff;
     transform:translate(-50%,-50%);pointer-events:none;font-size:11px;text-align:center;
     line-height:18px;color:#fff;font-weight:bold}}
.mp{{background:rgba(0,200,0,.85)}}.mn{{background:rgba(220,0,0,.85)}}
#hd{{font-size:15px;margin-bottom:6px}}
#pl{{margin-top:6px;font-family:monospace;font-size:13px;min-height:20px}}
#ct{{margin-top:8px}}
#ct button{{padding:6px 18px;margin-right:8px;font-size:14px;cursor:pointer}}
.bg{{background:#4CAF50;color:#fff;border:none;border-radius:4px}}
.bo{{background:#2196F3;color:#fff;border:none;border-radius:4px}}
</style>
<div id="hd"></div>
<div id="vw"><img id="im" /></div>
<div id="pl"></div>
<div id="ct">
  <button onclick="prev()">&larr; Prev</button>
  <button onclick="jump(-10)">-10</button>
  <button onclick="jump(10)">+10</button>
  <button onclick="undo()">Undo</button>
  <button onclick="clr()">Clear</button>
  <button onclick="next()">Next &rarr;</button>
  <button class="bo" onclick="previewMask()">Preview Mask</button>
  <button class="bg" onclick="saveAll()">Save All to Python</button>
</div>
<div id="pv" style="margin-top:6px;font-style:italic;color:#666"></div>

<script>
var imgs={_imgs_json};
var names={_names_json};
var dims={_dims_json};
var N=imgs.length;
var allPts={{}};
var ci=0;

function show(i){{
  ci=i;
  document.getElementById('im').src='data:image/jpeg;base64,'+imgs[i];
  document.getElementById('hd').innerHTML=
    '<b>View '+i+'/'+(N-1)+'</b> &mdash; '+names[i]+' ('+dims[i][0]+'x'+dims[i][1]+
    ') | Left=<span style="color:green">+pos</span> Right=<span style="color:red">-neg</span>';
  renderPts();
}}

function getC(e){{
  var r=document.getElementById('im').getBoundingClientRect();
  var ow=dims[ci][0],oh=dims[ci][1];
  return[Math.round((e.clientX-r.left)/r.width*ow),
         Math.round((e.clientY-r.top)/r.height*oh)];
}}

document.getElementById('vw').addEventListener('click',function(e){{
  e.preventDefault();var c=getC(e);
  if(!allPts[ci])allPts[ci]=[];
  allPts[ci].push({{x:c[0],y:c[1],l:1}});renderPts();
}});
document.getElementById('vw').addEventListener('contextmenu',function(e){{
  e.preventDefault();var c=getC(e);
  if(!allPts[ci])allPts[ci]=[];
  allPts[ci].push({{x:c[0],y:c[1],l:0}});renderPts();
}});

function renderPts(){{
  var co=document.getElementById('vw');
  co.querySelectorAll('.mk').forEach(m=>m.remove());
  var pts=allPts[ci]||[];
  var r=document.getElementById('im').getBoundingClientRect();
  var ow=dims[ci][0],oh=dims[ci][1];
  pts.forEach(function(p){{
    var m=document.createElement('div');
    m.className='mk '+(p.l===1?'mp':'mn');
    m.style.left=(p.x/ow*r.width)+'px';
    m.style.top=(p.y/oh*r.height)+'px';
    m.textContent=p.l===1?'+':'-';
    co.appendChild(m);
  }});
  var el=document.getElementById('pl');
  if(pts.length===0){{el.innerHTML='No points (text-only for this view)';}}
  else{{el.innerHTML=pts.map(p=>(p.l===1?'<span style="color:green">+</span>':'<span style="color:red">-</span>')+'('+p.x+','+p.y+')').join(' ');}}
  var n=Object.keys(allPts).filter(k=>allPts[k].length>0).length;
  document.getElementById('hd').innerHTML+=' | <b>'+n+' views annotated</b>';
}}

function undo(){{if(allPts[ci]&&allPts[ci].length)allPts[ci].pop();renderPts();}}
function clr(){{allPts[ci]=[];renderPts();}}
function prev(){{if(ci>0)show(ci-1);}}
function next(){{if(ci<N-1)show(ci+1);}}
function jump(d){{var t=Math.max(0,Math.min(N-1,ci+d));show(t);}}
function saveAll(){{
  google.colab.kernel.invokeFunction('_receive_all_points',[JSON.stringify(allPts)],{{}});
}}
var _previewing=false;
async function previewMask(){{
  var pts=allPts[ci]||[];
  if(pts.length===0){{document.getElementById('pv').innerHTML='Add points first';return;}}
  if(_previewing)return;
  _previewing=true;
  document.getElementById('pv').innerHTML='Running SAM3 preview...';
  try{{
    var res=await google.colab.kernel.invokeFunction('_preview_mask',[ci,JSON.stringify(pts)],{{}});
    var b64=res.data['text/plain'].replace(/^'/,'').replace(/'$/,'');
    if(b64){{
      document.getElementById('im').src='data:image/jpeg;base64,'+b64;
      document.getElementById('pv').innerHTML='Mask overlay (red). Prev/Next restores the image.';
    }} else {{
      document.getElementById('pv').innerHTML='No masks found for this view.';
    }}
  }}catch(e){{
    document.getElementById('pv').innerHTML='Preview error: '+e;
  }}
  _previewing=false;
}}

show(0);
</script>
"""
display(HTML(html))


In [ ]:
# 2.5.5 — Run SAM3 over aerial views (only if masks missing) -> views/masks/<stem>.png
# Only segments when masks are missing. Set FORCE_RESEGMENT=True to overwrite.
FORCE_RESEGMENT = False

# Restore masks from Drive if local copies were lost (Colab runtime reset)
if Path("/content/drive/MyDrive").exists():
    _drive_masks = DRIVE_OUT / "masks"
    _local_masks = VIEW_DIR / MASK_NAME
    if _drive_masks.is_dir() and (not _local_masks.is_dir() or len(list(_local_masks.glob("*.png"))) == 0):
        import shutil
        _local_masks.mkdir(parents=True, exist_ok=True)
        _restored = 0
        for _mp in _drive_masks.glob("*.png"):
            shutil.copy2(_mp, _local_masks / _mp.name)
            _restored += 1
        if _restored:
            print(f"Restored {_restored} masks from Drive -> {_local_masks}")

def _stems(d):
    d = Path(d)
    if not d.exists(): return set()
    return {p.stem for p in (list(d.glob("*.png")) + list(d.glob("*.jpg")))}

def segment_folder(base_dir, images_dir, per_view_points=None, label=""):
    base_dir, images_dir = Path(base_dir), Path(images_dir)
    mask_dir = base_dir / MASK_NAME
    image_stems = _stems(images_dir)
    mask_stems  = _stems(mask_dir)
    missing     = sorted(image_stems - mask_stems)

    if not FORCE_RESEGMENT and missing == [] and mask_stems:
        print(f"[{label}] all {len(mask_stems)} mask(s) exist in {mask_dir} — skipping SAM3.")
        return {"success_views": len(mask_stems), "total_views": len(image_stems)}
    if mask_stems and not FORCE_RESEGMENT:
        print(f"[{label}] {len(mask_stems)} mask(s) exist; segmenting {len(missing)} missing: {missing}")
        names = missing
    else:
        if FORCE_RESEGMENT:
            print(f"[{label}] FORCE_RESEGMENT=True — (re)segmenting all views.")
        names = None

    _mode = "text + point prompts" if per_view_points else "text prompt only"
    print(f"[{label}] prompt {TEXT_PROMPT!r} -> {MASK_NAME}/ ({_mode})")
    segmenter = SAM3MultiObjectSegmenter(confidence_threshold=0.1)
    res = segmenter.segment_object_multiview(
        images_dir=images_dir, object_name=MASK_NAME, text_prompt=TEXT_PROMPT,
        output_dir=base_dir, per_view_points=per_view_points, image_names=names,
    )
    print(f"[{label}] {res['success_views']}/{res['total_views']} masks -> {mask_dir}")
    return res

_pvp = globals().get("PER_VIEW_POINTS") or None
if not _pvp:
    # Fallback: reload from Drive (survives runtime resets)
    for _try_path in [OUTPUT_DIR / "per_view_points.json", DRIVE_OUT / "per_view_points.json"]:
        if _try_path.exists():
            import json as _json
            _pvp = {int(k): v for k, v in _json.loads(_try_path.read_text()).items()}
            if _pvp:
                PER_VIEW_POINTS = _pvp
                print(f"Restored {len(_pvp)} point prompts from {_try_path}")
                break
if _pvp:
    print(f"Using point prompts for {len(_pvp)} view(s)")
    FORCE_RESEGMENT = True   # clicks changed the target object — old masks are stale
if SKIP_SAM3_IF_MASKS_EXIST:
    segment_folder(VIEW_DIR, IMAGES_DIR, per_view_points=_pvp, label="ultrra_aerial")
else:
    FORCE_RESEGMENT = True
    segment_folder(VIEW_DIR, IMAGES_DIR, per_view_points=_pvp, label="ultrra_aerial")
assert len(_stems(VIEW_DIR / MASK_NAME)) > 0, "No masks — check SAM3 prompt / staged images."

# ---- Back up masks to Drive immediately ----
if Path("/content/drive/MyDrive").exists():
    import shutil
    _drive_masks = DRIVE_OUT / "masks"
    _drive_masks.mkdir(parents=True, exist_ok=True)
    _mask_dir = VIEW_DIR / MASK_NAME
    _copied = 0
    for _mp in sorted(_mask_dir.glob("*.png")):
        _dst = _drive_masks / _mp.name
        if not _dst.exists() or _mp.stat().st_mtime > _dst.stat().st_mtime:
            shutil.copy2(_mp, _dst)
            _copied += 1
    print(f"Masks backed up to Drive: {_drive_masks} ({_copied} new/updated)")


### 2.5.6 — Check the masks before anything else

Everything downstream (Stage-1 shape, the depth cloud, the NeuS frame) is built from these masks.
If SAM3 locked onto the wrong building, the whole reconstruction is wrong no matter how good the
cameras are — so confirm here, then go back to §2.5.4 and add clicks for the flagged views.


In [ ]:
# 2.5.6 — Preview the masks (are they on the building?)
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

_mask_dir = VIEW_DIR / MASK_NAME


def _key(p):
    try:
        return (0, int(p.stem), p.stem)
    except ValueError:
        return (1, 0, p.stem)


_files = sorted(list(IMAGES_DIR.glob("*.png")) + list(IMAGES_DIR.glob("*.jpg")), key=_key)
areas, cents = [], []
for f in _files:
    mp = _mask_dir / f"{f.stem}.png"
    a = np.array(Image.open(mp).convert("RGBA"))[:, :, 3] > 127 if mp.exists() else None
    if a is None or not a.any():
        areas.append(0.0); cents.append((np.nan, np.nan)); continue
    ys, xs = np.nonzero(a)
    areas.append(float(a.mean()))
    cents.append((xs.mean() / a.shape[1], ys.mean() / a.shape[0]))
areas = np.asarray(areas)

_med = float(np.median(areas[areas > 0])) if (areas > 0).any() else 0.0
_ratio = np.where(_med > 0, areas / max(_med, 1e-9), 0.0)
_odd = [i for i, r in enumerate(_ratio) if r < 0.4 or r > 2.5]
print(f"Mask area: median={_med * 100:.1f}% of frame | min={areas.min() * 100:.1f}% "
      f"max={areas.max() * 100:.1f}% | empty={int((areas == 0).sum())}")
if _odd:
    print(f"{len(_odd)} view(s) way off the median area (likely a different object): "
          f"{[ _files[i].stem for i in _odd ][:20]}")
    print("  -> add a positive click on the cathedral (and a negative on the wrong building)")
    print("     for those views in §2.5.4, then re-run §2.5.5.")
else:
    print("Mask areas are consistent across views.")

_show = list(range(0, len(_files), max(1, len(_files) // 8)))[:8]
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, i in zip(axes.ravel(), _show):
    f = _files[i]
    img = np.array(Image.open(f).convert("RGB"))
    mp = _mask_dir / f"{f.stem}.png"
    ax.imshow(img)
    if mp.exists():
        m = np.array(Image.open(mp).convert("RGBA"))[:, :, 3] > 127
        ov = np.zeros((*m.shape, 4)); ov[m] = (1, 0, 0, 0.45)
        ax.imshow(ov)
    ax.set_title(f"[{i}] {f.stem} — {areas[i] * 100:.1f}%", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(_show):]:
    ax.axis("off")
fig.suptitle("SAM3 masks (red) — every red region must be the target building", fontsize=13)
fig.tight_layout()
_png = OUTPUT_DIR / "mask_preview.png"
fig.savefig(_png, dpi=110, bbox_inches="tight")
plt.show()
print("Saved:", _png)
if Path("/content/drive/MyDrive").exists():
    import shutil
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_png, DRIVE_OUT / _png.name)
    print("Backed up to Drive ->", DRIVE_OUT / _png.name)


## 2.6 — Pose audit helpers + ULTRRA geo reference (`jsons/`)

Loads **geodetic camera positions** from `va_split/jsons/<stem>.json` for pose audit.
JSONs are position-only (no rotation/intrinsics) — DA3 supplies those in §8.
During BA-NeuS training we log how far poses drift from the geo-aligned init.


In [ ]:
# 2.6 — Pose audit helpers + ULTRRA geo reference (jsons/)
import json, os, subprocess, sys
import numpy as np
from pathlib import Path

try:
    import pymap3d  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymap3d"])

# Self-contained if §2 config was not run in this kernel session
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/baneus_ultrra_out")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if "POSE_AUDIT_DIR" not in globals():
    POSE_AUDIT_DIR = OUTPUT_DIR / "pose_audit"
POSE_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
(POSE_AUDIT_DIR / "snapshots").mkdir(parents=True, exist_ok=True)

def _img_stem_pose(p):
    return Path(p).stem.replace("_masked", "")

def read_mvsnet_cam(path, num_depth=192):
    """Parse MVSNet cam.txt -> (E 4x4 world->cam, K 3x3, dmin, dmax)."""
    w = Path(path).read_text().split()
    ie, ii = w.index("extrinsic"), w.index("intrinsic")
    E = np.array(w[ie + 1:ie + 17], np.float64).reshape(4, 4)
    K = np.array(w[ii + 1:ii + 10], np.float64).reshape(3, 3)
    tail = w[ii + 10:]
    dmin = float(tail[0]) if tail else None
    dmax = None
    if len(tail) >= 4:
        dmax = float(tail[3])
    elif len(tail) == 3:
        dmax = dmin + float(tail[1]) * (int(float(tail[2])) - 1)
    elif len(tail) == 2:
        dmax = dmin + float(tail[1]) * (num_depth - 1)
    return E, K, dmin, dmax

def _to_c2w(ext):
    """DA3 `prediction.extrinsics` are WORLD->CAMERA — invert them.

    DA3's own exporters (utils/export/glb.py, colmap.py) annotate the field `# w2c`
    and do `c2w = inv(ext)`. Using the raw matrix as c2w puts every camera at the
    mirrored pose (R^T, -R^T t), which is ~180 deg off and breaks all ray geometry.
    """
    ext = np.asarray(ext, np.float64)
    w2c = np.eye(4, dtype=np.float64)
    w2c[:3, :] = ext[:3, :4]
    return np.linalg.inv(w2c)

def w2c_to_c2w(E):
    E = np.asarray(E, np.float64)
    c2w = np.linalg.inv(E)
    return c2w

def _cam_center_c2w(c2w):
    return np.asarray(c2w, np.float64)[:3, 3]

def _rot_angle_deg(Ra, Rb):
    R = Ra.T @ Rb
    c = (np.trace(R) - 1.0) * 0.5
    return float(np.degrees(np.arccos(np.clip(c, -1.0, 1.0))))

def pose_metrics_one(c2w_a, c2w_b):
    Ra, Rb = c2w_a[:3, :3], c2w_b[:3, :3]
    ta, tb = c2w_a[:3, 3], c2w_b[:3, 3]
    return {
        "rot_deg": _rot_angle_deg(Ra, Rb),
        "trans_l2": float(np.linalg.norm(ta - tb)),
        "center_l2": float(np.linalg.norm(_cam_center_c2w(c2w_a) - _cam_center_c2w(c2w_b))),
    }

def sim3_align_centers(ref_c2ws, est_c2ws):
    """Umeyama-like sim3 aligning est camera centers to ref (for cross-frame compare)."""
    ref_c = np.stack([_cam_center_c2w(c) for c in ref_c2ws])
    est_c = np.stack([_cam_center_c2w(c) for c in est_c2ws])
    rc, ec = ref_c.mean(0), est_c.mean(0)
    ref_z = ref_c - rc
    est_z = est_c - ec
    s = np.sqrt((ref_z ** 2).sum()) / (np.sqrt((est_z ** 2).sum()) + 1e-9)
    H = est_z.T @ ref_z
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt = Vt.copy(); Vt[-1] *= -1; R = Vt.T @ U.T
    t = rc - s * (R @ ec)
    return float(s), R.astype(np.float64), t.astype(np.float64)

def sim3_align_points(ref_pts, est_pts, max_pts=4000):
    """Sim3 aligning est -> ref (unpaired clouds OK; NN correspondences)."""
    ref = np.asarray(ref_pts, np.float64)
    est = np.asarray(est_pts, np.float64)
    if len(ref) < 3 or len(est) < 3:
        return 1.0, np.eye(3, dtype=np.float64), np.zeros(3, dtype=np.float64)
    if len(ref) > max_pts:
        ref = ref[np.linspace(0, len(ref) - 1, max_pts, dtype=int)]
    if len(est) > max_pts:
        est = est[np.linspace(0, len(est) - 1, max_pts, dtype=int)]
    from scipy.spatial import cKDTree
    _, nn = cKDTree(ref).query(est, k=1)
    ref = ref[nn]                     # pair each mesh sample to nearest DA3 point
    rc, ec = ref.mean(0), est.mean(0)
    ref_z, est_z = ref - rc, est - ec
    s = np.sqrt((ref_z ** 2).sum()) / (np.sqrt((est_z ** 2).sum()) + 1e-9)
    H = est_z.T @ ref_z
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt = Vt.copy(); Vt[-1] *= -1; R = Vt.T @ U.T
    t = rc - s * (ec @ R)
    return float(s), R.astype(np.float64), t.astype(np.float64)

def apply_sim3_points(pts, s, R, t):
    pts = np.asarray(pts, np.float64)
    return float(s) * pts @ np.asarray(R, np.float64) + np.asarray(t, np.float64)

def invert_sim3_points(pts, s, R, t):
    pts = np.asarray(pts, np.float64)
    R = np.asarray(R, np.float64); t = np.asarray(t, np.float64)
    return (pts - t) @ R.T / max(float(s), 1e-9)

# MV-SAM3D / SAM3D coordinate chain (see MV-SAM3D/sam3d_objects/utils/coordinate_transforms.py)
Z_UP_TO_Y_UP = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=np.float64)
PYTORCH3D_TO_OPENCV = np.diag([-1.0, -1.0, 1.0]).astype(np.float64)

def _homog_row_mul(pts, M):
    pts = np.asarray(pts, np.float64)
    M = np.asarray(M, np.float64)
    return (np.c_[pts, np.ones(len(pts))] @ M)[:, :3]

def _canonical_to_opencv_world_points(verts, scale, rotation, translation,
                                      c2w_opencv=None, device="cpu"):
    """Transform canonical mesh vertices to DA3 OpenCV world space.

    The occupancy decoder outputs a grid whose axes already match the
    format expected by the pose decoder (no Z_UP_TO_Y_UP needed — that
    rotation is only required when loading from GLB, which stores
    canonical Z-up coords; marching cubes on the decoder grid skips it).

    After the pose, points are in PyTorch3D camera space of the reference
    view.  PYTORCH3D_TO_OPENCV flips X and Y for OpenCV convention, then
    c2w of the reference view brings them to OpenCV world.

    Verified locally (test_alignment.py + check_with_sim3.py, 2026-07-27):
      identity | wxyz | P3D+c2w   → 3D offset 0.74  (mesh center to NeuS center)
      identity | wxyz | P3D only  → 3D offset 1.56  (2× worse, camera frame ≠ world)

    Chain:
      1. v = vertices  (no pre-rotation)
      2. pts_p3d = pose_transform.transform_points(v)   (→ P3D camera space)
      3. pts_cv  = pts_p3d @ PYTORCH3D_TO_OPENCV        (→ OpenCV camera space)
      4. pts_world = c2w @ pts_cv                        (→ OpenCV world)
    """
    from pytorch3d.transforms import Transform3d, quaternion_to_matrix
    dev = torch.device(device)

    v = np.asarray(verts, np.float64)

    # Step 1: apply decoded SAM3D pose → PyTorch3D camera space (ref view)
    sc = torch.as_tensor(scale, dtype=torch.float32, device=dev).reshape(-1)
    if sc.numel() == 1:
        sc = sc.expand(3)
    R = quaternion_to_matrix(torch.as_tensor(rotation, dtype=torch.float32, device=dev))
    if R.dim() == 2:
        R = R.unsqueeze(0)
    t = torch.as_tensor(translation, dtype=torch.float32, device=dev).reshape(1, 3)
    pose_transform = (
        Transform3d(device=dev, dtype=torch.float32)
        .scale(sc.unsqueeze(0)).rotate(R).translate(t)
    )
    pts_p3d = pose_transform.transform_points(
        torch.from_numpy(v).float().unsqueeze(0).to(dev)
    ).squeeze(0).detach().cpu().numpy().astype(np.float64)

    # Step 2: PyTorch3D camera → OpenCV camera (flip X and Y)
    pts_cv = pts_p3d @ PYTORCH3D_TO_OPENCV

    # Step 3: OpenCV camera → OpenCV world (reference view c2w)
    if c2w_opencv is not None:
        c2w = np.asarray(c2w_opencv, np.float64)
        pts_world = (c2w @ np.c_[pts_cv, np.ones(len(pts_cv))].T).T[:, :3]
    else:
        pts_world = pts_cv

    return pts_world

def stage1_canonical_to_da3_world_matrix(scale, rotation, translation, c2w_opencv=None, device="cpu"):
    """Compute the combined 4x4 row-vector matrix (canonical -> OpenCV world).

    c2w_opencv: 4x4 camera-to-world matrix for the reference view.
    """
    basis = np.array([[1,0,0],[0,1,0],[0,0,1],[0,0,0]], dtype=np.float64)
    world = _canonical_to_opencv_world_points(basis, scale, rotation, translation,
                                              c2w_opencv=c2w_opencv, device=device)
    origin = world[3]
    M_col = np.eye(4, dtype=np.float64)
    M_col[:3, :3] = (world[:3] - origin).T   # columns = transformed basis vectors
    M_col[:3, 3] = origin
    return M_col.T                            # row-vector form for _homog_row_mul

def stage1_canonical_to_da3_world(verts_can, M_can2w):
    return _homog_row_mul(verts_can, M_can2w)

def stage1_da3_world_to_canonical(verts_world, M_can2w):
    return _homog_row_mul(verts_world, np.linalg.inv(M_can2w))

def apply_sim3_c2ws(c2ws, s, R, t):
    out = []
    for c2w in c2ws:
        c = np.asarray(c2w, np.float64).copy()
        c[:3, :3] = R @ c[:3, :3]
        c[:3, 3] = s * (R @ c[:3, 3]) + t
        out.append(c)
    return np.stack(out)



def _to_jsonable(obj):
    """Recursively convert numpy/scalars to JSON-safe Python types."""
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer, np.bool_)):
        return obj.item()
    if isinstance(obj, Path):
        return str(obj)
    return obj

def summarize_pose_errors(est_c2ws, ref_c2ws, label="vs_gt", align_sim3=False):
    est = np.asarray(est_c2ws, np.float64)
    ref = np.asarray(ref_c2ws, np.float64)
    sim3 = None
    if align_sim3:
        s, R, t = sim3_align_centers(ref, est)
        est = apply_sim3_c2ws(est, s, R, t)
        sim3 = {"scale": float(s), "R": R.tolist(), "t": t.tolist()}
    per_view = []
    for i, (e, r) in enumerate(zip(est, ref)):
        m = pose_metrics_one(e, r)
        m["view"] = i
        per_view.append(m)
    rot = np.array([m["rot_deg"] for m in per_view])
    trans = np.array([m["trans_l2"] for m in per_view])
    cent = np.array([m["center_l2"] for m in per_view])
    summary = {
        "label": label,
        "align_sim3": align_sim3,
        "n": len(per_view),
        "rot_deg_mean": float(rot.mean()), "rot_deg_max": float(rot.max()),
        "trans_mean": float(trans.mean()), "trans_max": float(trans.max()),
        "center_mean": float(cent.mean()), "center_max": float(cent.max()),
        "sim3": sim3,
    }
    return summary, per_view

def save_pose_snapshot(stage, c2ws, view_names, Ks=None, extra=None):
    snap = POSE_AUDIT_DIR / "snapshots" / f"{stage}.npz"
    payload = {
        "c2ws": np.asarray(c2ws, np.float64),
        "view_names": np.array(view_names),
    }
    if Ks is not None:
        payload["Ks"] = np.asarray(Ks, np.float64)
    if extra:
        for k, v in extra.items():
            payload[k] = v
    np.savez(snap, **payload)
    return snap

def pose_audit_log(stage, c2ws, view_names, ref_c2ws=None, ref_label="gt", Ks=None,
                   align_sim3=False, extra=None):
    c2ws = np.asarray(c2ws, np.float64)
    entry = {
        "stage": stage,
        "n_views": len(view_names),
        "view_names": list(view_names),
        "extra": extra or {},
    }
    if ref_c2ws is not None:
        summary, per_view = summarize_pose_errors(c2ws, ref_c2ws, label=ref_label, align_sim3=align_sim3)
        entry["vs_ref"] = summary
        entry["per_view_vs_ref"] = per_view
    snap = save_pose_snapshot(stage, c2ws, view_names, Ks=Ks, extra=extra)
    entry["snapshot"] = str(snap)
    _log = POSE_AUDIT_DIR / "pose_log.jsonl"
    with open(_log, "a", encoding="utf-8") as f:
        f.write(json.dumps(_to_jsonable(entry)) + "\n")
    # Back up audit to Drive
    if Path("/content/drive/MyDrive").exists():
        _drpa = DRIVE_OUT / "pose_audit"
        _drpa.mkdir(parents=True, exist_ok=True)
        (_drpa / "snapshots").mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy2(snap, _drpa / "snapshots" / snap.name)
        shutil.copy2(_log, _drpa / _log.name)
    print(f"[pose_audit] {stage}: saved {snap.name}")
    if ref_c2ws is not None:
        s = entry["vs_ref"]
        print(
            f"  {ref_label} | rot mean/max={s['rot_deg_mean']:.3f}/{s['rot_deg_max']:.3f} deg | "
            f"center mean/max={s['center_mean']:.4f}/{s['center_max']:.4f}"
            + (" (sim3 aligned)" if align_sim3 else "")
        )
    return entry

def read_pfm(path):
    """Read a .pfm depth map (BlendedMVS rendered_depth_maps) -> float32 HxW."""
    with open(path, "rb") as f:
        header = f.readline().decode("latin-1").rstrip()
        color = header == "PF"
        dims = f.readline().decode("latin-1").rstrip()
        w, h = (int(x) for x in dims.split())
        scale = float(f.readline().decode("latin-1").rstrip())
        endian = "<" if scale < 0 else ">"
        data = np.fromfile(f, endian + "f")
    ch = 3 if color else 1
    data = data.reshape(h, w, ch) if color else data.reshape(h, w)
    data = np.flipud(data).astype(np.float32)  # PFM is bottom-to-top
    return data

def _c2w_to_ext(c2w):
    """Inverse of _to_c2w: store back in DA3's world->camera convention."""
    return np.linalg.inv(np.asarray(c2w, np.float64))[:3, :4]


def _rotmat_axis_angle(axis, angle_rad):
    axis = np.asarray(axis, np.float64)
    axis = axis / (np.linalg.norm(axis) + 1e-9)
    x, y, z = axis
    K = np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]], dtype=np.float64)
    return np.eye(3) + np.sin(angle_rad) * K + (1.0 - np.cos(angle_rad)) * (K @ K)


def da3_depth_to_pointmap_sam3d(depth, K):
    """Camera-space pointmap in SAM3D layout (3, H, W); matches MV-SAM3D run_da3.py."""
    depth = np.asarray(depth, np.float64)
    K = np.asarray(K, np.float64)
    H, W = depth.shape
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    v, u = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    x = (u - cx) * depth / fx
    y = (v - cy) * depth / fy
    pm = np.stack([x, y, depth], axis=-1)
    return pm.transpose(2, 0, 1).astype(np.float32)


def da3_noise_enabled():
    return any(float(x) > 0 for x in (
        DA3_NOISE_TRANS_STD, DA3_NOISE_ROT_STD_DEG,
        DA3_NOISE_FOCAL_STD_FRAC, DA3_NOISE_PP_STD_PX,
    ))


def print_da3_noise_scale_reference(c2ws, Ks, img_hw):
    """Explain noise knob magnitudes relative to this scene's DA3 cameras."""
    centers = np.stack([_cam_center_c2w(c) for c in c2ws])
    spread = float(np.linalg.norm(centers - centers.mean(0), axis=1).mean())
    baselines = []
    for i in range(len(centers)):
        d = np.linalg.norm(centers[i] - centers[(i + 1) % len(centers)])
        baselines.append(d)
    baseline_med = float(np.median(baselines)) if baselines else spread
    fx = float(np.mean([K[0, 0] for K in Ks]))
    fy = float(np.mean([K[1, 1] for K in Ks]))
    H, W = int(img_hw[0]), int(img_hw[1])
    hfov = float(np.degrees(2 * np.arctan(W / (2 * fx))))
    vfov = float(np.degrees(2 * np.arctan(H / (2 * fy))))
    print("--- DA3 noise scale reference (this scene) ---")
    print(f"  camera centers: mean spread={spread:.4f} | median neighbor baseline={baseline_med:.4f} world units")
    print(f"  intrinsics @ {W}x{H}: mean fx={fx:.1f} fy={fy:.1f} | hfov≈{hfov:.1f}° vfov≈{vfov:.1f}°")
    if DA3_NOISE_TRANS_STD > 0:
        print(f"  trans_std={DA3_NOISE_TRANS_STD}: 1σ ≈ {100*DA3_NOISE_TRANS_STD/spread:.1f}% of center spread, "
              f"{100*DA3_NOISE_TRANS_STD/baseline_med:.1f}% of neighbor baseline")
    if DA3_NOISE_ROT_STD_DEG > 0:
        print(f"  rot_std={DA3_NOISE_ROT_STD_DEG}°: 1σ random tilt; compare to ~5–30° typical view-to-view rotation")
    if DA3_NOISE_FOCAL_STD_FRAC > 0:
        print(f"  focal_frac={DA3_NOISE_FOCAL_STD_FRAC}: 1σ ≈ ±{DA3_NOISE_FOCAL_STD_FRAC*fx:.1f}px on fx "
              f"(small-angle ≈ ±{DA3_NOISE_FOCAL_STD_FRAC*hfov:.2f}° hfov)")
    if DA3_NOISE_PP_STD_PX > 0:
        print(f"  pp_std={DA3_NOISE_PP_STD_PX}px: 1σ shift on cx,cy ({100*DA3_NOISE_PP_STD_PX/W:.2f}% of image width)")
    print("  (All knobs are per-view i.i.d. Gaussian; 0 = disabled)")


def apply_da3_camera_noise(extrinsics, intrinsics, depth, trans_std, rot_std_deg,
                           focal_std_frac, pp_std_px, seed=None):
    """Perturb DA3 cameras; recompute camera-space pointmaps from depth + noisy K."""
    rng = np.random.default_rng(seed)
    N = len(extrinsics)
    ext_out, K_out, pm_out = [], [], []
    rot_applied, trans_applied, focal_applied, pp_applied = [], [], [], []
    for i in range(N):
        c2w = _to_c2w(extrinsics[i])
        K = np.asarray(intrinsics[i], np.float64).copy()
        if rot_std_deg > 0:
            axis = rng.standard_normal(3)
            angle = np.radians(rng.normal(0.0, rot_std_deg))
            c2w = c2w.copy()
            c2w[:3, :3] = _rotmat_axis_angle(axis, angle) @ c2w[:3, :3]
            rot_applied.append(float(np.degrees(abs(angle))))
        if trans_std > 0:
            delta = rng.normal(0.0, trans_std, 3)
            c2w = c2w.copy()
            c2w[:3, 3] += delta
            trans_applied.append(float(np.linalg.norm(delta)))
        if focal_std_frac > 0:
            dfx = rng.normal(0.0, focal_std_frac)
            dfy = rng.normal(0.0, focal_std_frac)
            K[0, 0] *= (1.0 + dfx)
            K[1, 1] *= (1.0 + dfy)
            focal_applied.append(float(max(abs(dfx), abs(dfy))))
        if pp_std_px > 0:
            dcx = rng.normal(0.0, pp_std_px)
            dcy = rng.normal(0.0, pp_std_px)
            K[0, 2] += dcx
            K[1, 2] += dcy
            pp_applied.append(float(np.hypot(dcx, dcy)))
        ext_out.append(_c2w_to_ext(c2w))
        K_out.append(K)
        pm_out.append(da3_depth_to_pointmap_sam3d(depth[i], K))
    stats = {
        "n_views": N,
        "rot_deg_mean": float(np.mean(rot_applied)) if rot_applied else 0.0,
        "trans_l2_mean": float(np.mean(trans_applied)) if trans_applied else 0.0,
        "focal_frac_mean": float(np.mean(focal_applied)) if focal_applied else 0.0,
        "pp_px_mean": float(np.mean(pp_applied)) if pp_applied else 0.0,
    }
    return np.stack(ext_out), np.stack(K_out), np.stack(pm_out), stats


def _json_has_geo(meta):
    ext = meta.get("extrinsics", {})
    return all(k in ext for k in ("lat", "lon", "alt"))

def _geodetic_origin(metas):
    lats, lons, alts = [], [], []
    for m in metas:
        e = m.get("extrinsics", {})
        if all(k in e for k in ("lat", "lon", "alt")):
            lats.append(e["lat"]); lons.append(e["lon"]); alts.append(e["alt"])
    return (float(np.mean(lats)), float(np.mean(lons)), float(np.mean(alts))) if lats else None

def _geo_ref_c2w(meta, origin):
    """Position-only reference c2w in ENU (identity rotation — JSON has no omega/phi/kappa)."""
    from pymap3d import geodetic2enu
    d = meta["extrinsics"]
    t = np.array(geodetic2enu(d["lat"], d["lon"], d["alt"], *origin), dtype=np.float64)
    c2w = np.eye(4, dtype=np.float64); c2w[:3, 3] = t
    return c2w

def load_geo_json_refs(jsons_dir, stems):
  refs = {}
  for stem in stems:
    jp = Path(jsons_dir) / f"{stem}.json"
    assert jp.exists(), f"Missing {jp} — need jsons/<stem>.json for every staged view"
    meta = json.loads(jp.read_text())
    assert _json_has_geo(meta), f"{jp} lacks lat/lon/alt in extrinsics"
    refs[stem] = {"meta": meta, "json_path": str(jp)}
  return refs

if USE_PREPROCESSED:
    assert "view_names" in dir() and view_names, "Run §2.1 first"
    _gt_stems = list(view_names)
    if "VIEW_MANIFEST" not in dir():
        VIEW_MANIFEST = [(s, None, None) for s in _gt_stems]
else:
    assert "VIEW_MANIFEST" in dir(), "Run §2.5.3 first"
    _gt_stems = [m[0] for m in VIEW_MANIFEST]

GEO_REF = load_geo_json_refs(JSONS_DIR, _gt_stems)
GEO_ORIGIN = _geodetic_origin([GEO_REF[s]["meta"] for s in _gt_stems])
for s in _gt_stems:
    GEO_REF[s]["c2w"] = _geo_ref_c2w(GEO_REF[s]["meta"], GEO_ORIGIN)
    GEO_REF[s]["enu"] = GEO_REF[s]["c2w"][:3, 3].copy()

HAS_GEO_JSON = True
GEO_C2WS = np.stack([GEO_REF[s]["c2w"] for s in _gt_stems])
GEO_VIEW_NAMES = list(_gt_stems)
save_pose_snapshot("00_geo_json_reference", GEO_C2WS, GEO_VIEW_NAMES,
                   extra={"source": str(JSONS_DIR), "origin_lla": GEO_ORIGIN, "note": "position-only"})
pose_audit_log("00_geo_json_reference", GEO_C2WS, GEO_VIEW_NAMES, ref_c2ws=None)
print(f"Geo JSON refs: {len(GEO_VIEW_NAMES)} views from {JSONS_DIR}")
print(f"ENU origin (lat,lon,alt): {tuple(round(x, 6) for x in GEO_ORIGIN)}")


## 3 — Load views (optional preview)

> **Skip when `USE_PREPROCESSED=True`** if you only need training — §2.1 already set `view_names`.


In [ ]:
# 3.0 — Load views (preprocessed or MV-SAM3D loader)
import numpy as np
from pathlib import Path
from PIL import Image

if USE_PREPROCESSED:
    if "view_names" not in dir() or not view_names:
        raise RuntimeError("Run §2.1 first")
    view_images, view_masks = [], []
    for name in view_names:
        ip = IMAGES_DIR / f"{name}.png"
        if not ip.exists():
            ip = next(IMAGES_DIR.glob(f"{name}.*"))
        mp = VIEW_DIR / MASK_NAME / f"{name}.png"
        view_images.append(np.array(Image.open(ip).convert("RGB")))
        if mp.exists():
            m = np.array(Image.open(mp).convert("L"))
            view_masks.append((m > 127).astype(np.uint8) * 255)
        else:
            view_masks.append(np.ones(view_images[-1].shape[:2], np.uint8) * 255)
    print(f"Preprocessed views: {len(view_images)}  shape={view_images[0].shape}")
else:
    from load_images_and_masks import load_images_and_masks_from_path
    view_images, view_masks, view_names = load_images_and_masks_from_path(
        VIEW_DIR, mask_prompt=MASK_NAME, image_names=None,
    )
    print(f"Aerial views: {len(view_images)}  names={view_names}")
print("view img0", np.array(view_images[0]).shape)



> **Skip when `USE_PREPROCESSED=True`**

## 8 — DA3 on the aerial views

MV-SAM3D's Stage 1 consumes a DA3 pointmap per view (camera pose + metric-ish points).
We run DA3 on the staged `views/images/` set; pointmaps are matched back by filename stem.

**Optional camera noise** (`DA3_NOISE_*` in §2): perturbs DA3 extrinsics/intrinsics before Stage-1 and BA-NeuS export. §8 prints a scene-relative scale guide when any knob is nonzero. Raw DA3 is kept in `da3_output_raw.npz` so you can change noise without re-inference.

In [ ]:
# 8.0 — Run DA3 on aerial views + geo-align to jsons/ ENU
import subprocess, shutil, numpy as np, glob, os
from pathlib import Path

DA3_OUT = OUTPUT_DIR / "da3_views"; DA3_OUT.mkdir(parents=True, exist_ok=True)
DA3_NPZ = DA3_OUT / "da3_output.npz"
DA3_NPZ_RAW = DA3_OUT / "da3_output_raw.npz"  # unperturbed DA3 (kept for re-noising without re-inference)

# Restore DA3 from Drive if local copies were lost
if not DA3_NPZ_RAW.exists() and Path("/content/drive/MyDrive").exists():
    _drive_da3 = DRIVE_OUT / "da3"
    for _fn in ("da3_output_raw.npz", "da3_output.npz"):
        _src = _drive_da3 / _fn
        _dst = DA3_OUT / _fn
        if _src.exists() and not _dst.exists():
            shutil.copy2(_src, _dst)
            print(f"Restored {_fn} from Drive")

# Resolve DA3 weights to the actual snapshot dir (from_pretrained can't load the cache root).
DA3_MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE"
_snap_glob = os.path.expanduser(
    "~/.cache/huggingface/hub/models--depth-anything--DA3NESTED-GIANT-LARGE/snapshots/*/")
_snaps = sorted(glob.glob(_snap_glob), key=os.path.getmtime, reverse=True)
DA3_MODEL_PATH = _snaps[0] if _snaps else DA3_MODEL_ID
print("DA3 model path:", DA3_MODEL_PATH)

if not DA3_NPZ_RAW.exists():
    if not DA3_NPZ.exists():
        cmd = [
            "python", str(MVSAM3D_ROOT / "scripts" / "run_da3.py"),
            "--image_dir", str(IMAGES_DIR),
            "--output_dir", str(DA3_OUT),
            "--model_path", str(DA3_MODEL_PATH),
            "--process_res", str(DA3_PROCESS_RES), "--no_vis",
        ]
        print("Running:", " ".join(cmd))
        proc = subprocess.run(cmd, cwd=str(MVSAM3D_ROOT), capture_output=True, text=True)
        if proc.stdout: print(proc.stdout)
        if proc.returncode != 0:
            print("------ DA3 STDERR ------"); print(proc.stderr)
            raise RuntimeError(f"run_da3.py failed (exit {proc.returncode}). See STDERR above.")
    shutil.copy2(DA3_NPZ, DA3_NPZ_RAW)
    print("Saved raw DA3 ->", DA3_NPZ_RAW)

da3_raw = np.load(DA3_NPZ_RAW)
_ext = da3_raw["extrinsics"].copy()
_intr = da3_raw["intrinsics"].copy()
_depth = da3_raw["depth"]
_pm_sam3d = da3_raw["pointmaps_sam3d"].copy()
_image_files = da3_raw["image_files"]
_process_res = int(da3_raw["process_res"]) if "process_res" in da3_raw else DA3_PROCESS_RES

if da3_noise_enabled():
    _noise_seed = DA3_NOISE_SEED if DA3_NOISE_SEED is not None else SEED
    _c2ws_pre = np.stack([_to_c2w(_ext[i]) for i in range(len(_ext))])
    print_da3_noise_scale_reference(_c2ws_pre, _intr, _depth.shape[1:3])
    _ext, _intr, _pm_sam3d, _nstats = apply_da3_camera_noise(
        _ext, _intr, _depth,
        DA3_NOISE_TRANS_STD, DA3_NOISE_ROT_STD_DEG,
        DA3_NOISE_FOCAL_STD_FRAC, DA3_NOISE_PP_STD_PX,
        seed=_noise_seed,
    )
    print("Applied DA3 camera noise:", _nstats, "| seed=", _noise_seed)
else:
    print("DA3 camera noise: off (all DA3_NOISE_* = 0)")

np.savez(
    DA3_NPZ,
    depth=_depth,
    pointmaps_sam3d=_pm_sam3d,
    extrinsics=_ext,
    intrinsics=_intr,
    image_files=_image_files,
    process_res=_process_res,
    da3_noise_trans_std=np.float32(DA3_NOISE_TRANS_STD),
    da3_noise_rot_std_deg=np.float32(DA3_NOISE_ROT_STD_DEG),
    da3_noise_focal_std_frac=np.float32(DA3_NOISE_FOCAL_STD_FRAC),
    da3_noise_pp_std_px=np.float32(DA3_NOISE_PP_STD_PX),
    da3_noise_seed=np.int64(-1 if DA3_NOISE_SEED is None else int(DA3_NOISE_SEED)),
)
print("Working DA3 bundle ->", DA3_NPZ)

# ---- Back up DA3 outputs to Drive immediately ----
if Path("/content/drive/MyDrive").exists():
    _drive_da3 = DRIVE_OUT / "da3"
    _drive_da3.mkdir(parents=True, exist_ok=True)
    for _f in (DA3_NPZ, DA3_NPZ_RAW):
        if _f.exists():
            shutil.copy2(_f, _drive_da3 / _f.name)
    print("DA3 backed up to Drive ->", _drive_da3)

da3_pointmaps = _pm_sam3d
da3_stems = [Path(str(p)).stem for p in _image_files]
print("DA3 pointmaps", da3_pointmaps.shape, "| views", len(da3_stems))
# ---- Sim(3) align DA3 camera centers to jsons/ geodetic ENU ----
def _ext_to44(ext):
    ext = np.asarray(ext, np.float64)
    if ext.shape[-2:] == (3, 4):
        pad = np.eye(4, dtype=np.float64); pad[:3, :4] = ext; return pad
    return ext

def _cam_center_from_ext(ext):
    E = _ext_to44(ext)
    R, t = E[:3, :3], E[:3, 3]
    return -R.T @ t

def _umeyama_sim3(src, dst):
    src = np.asarray(src, np.float64); dst = np.asarray(dst, np.float64)
    n = src.shape[0]
    mu_s, mu_d = src.mean(0), dst.mean(0)
    src_c, dst_c = src - mu_s, dst - mu_d
    cov = dst_c.T @ src_c / n
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    var_s = (src_c ** 2).sum() / n
    s = float(np.trace(np.diag(D) @ S) / var_s) if var_s > 1e-12 else 1.0
    t = mu_d - s * (R @ mu_s)
    return R, t, s

def _apply_sim3_to_ext(R, t, s, ext):
    E = _ext_to44(ext)
    c2w = np.linalg.inv(E)
    c2w[:3, :3] = R @ c2w[:3, :3]
    c2w[:3, 3] = s * (R @ c2w[:3, 3]) + t
    w2c = np.linalg.inv(c2w)
    out = np.eye(4, dtype=np.float64)
    out[:3, :] = w2c[:3, :]
    return out[:3, :4] if ext.shape == (3, 4) else out

geo_align_info = {"applied": False}
if ALIGN_DA3_TO_JSON_GEO and "GEO_REF" in dir() and len(GEO_REF) >= MIN_GEO_VIEWS_FOR_ALIGN:
    _geo_stems = [n for n in view_names if n in GEO_REF]
    if len(_geo_stems) >= MIN_GEO_VIEWS_FOR_ALIGN:
        _est = np.stack([_cam_center_from_ext(_ext[da3_stems.index(n)]) for n in _geo_stems])
        _ref = np.stack([GEO_REF[n]["enu"] for n in _geo_stems])
        Rg, tg, sg = _umeyama_sim3(_est, _ref)
        _ext_new = []
        for j in range(len(_ext)):
            e44 = _ext[j]
            if e44.shape == (3, 4):
                e44 = _apply_sim3_to_ext(Rg, tg, sg, e44)
            else:
                e44 = _apply_sim3_to_ext(Rg, tg, sg, e44)[:3, :4]
            _ext_new.append(e44)
        _ext = np.stack(_ext_new)
        _depth = _depth * sg
        _pm_sam3d = _pm_sam3d * sg
        _aln = np.stack([_cam_center_from_ext(_ext[da3_stems.index(n)]) for n in _geo_stems])
        geo_align_info = {
            "applied": True, "scale": sg,
            "center_rmse_m": float(np.sqrt((( _aln - _ref) ** 2).sum(1).mean())),
            "center_max_m": float(np.linalg.norm(_aln - _ref, axis=1).max()),
            "n_views": len(_geo_stems),
        }
        np.savez(DA3_NPZ, depth=_depth, pointmaps_sam3d=_pm_sam3d, extrinsics=_ext,
                 intrinsics=_intr, image_files=_image_files, process_res=_process_res,
                 geo_align_scale=np.float32(sg), geo_align_applied=np.bool_(True))
        print(f"[geo align] DA3 -> ENU | scale={sg:.4f} | center RMSE={geo_align_info['center_rmse_m']:.2f} m "
              f"(max {geo_align_info['center_max_m']:.2f} m) over {len(_geo_stems)} views")
    else:
        print(f"[geo align] skipped — only {len(_geo_stems)} geo JSONs (need {MIN_GEO_VIEWS_FOR_ALIGN})")
elif ALIGN_DA3_TO_JSON_GEO:
    print("[geo align] skipped — run §2.6 first (GEO_REF)")



# ---- pose audit: DA3 extrinsics vs GT ----
_da3_c2ws, _da3_Ks = [], []
for name in view_names:
    j = da3_stems.index(name)
    _da3_c2ws.append(_to_c2w(_ext[j]))
    _da3_Ks.append(_intr[j].astype(np.float64))
_da3_c2ws = np.stack(_da3_c2ws)
_extra = {"camera_src": "da3"}
if da3_noise_enabled():
    _extra["da3_noise"] = {
        "trans_std": DA3_NOISE_TRANS_STD,
        "rot_std_deg": DA3_NOISE_ROT_STD_DEG,
        "focal_std_frac": DA3_NOISE_FOCAL_STD_FRAC,
        "pp_std_px": DA3_NOISE_PP_STD_PX,
        "seed": DA3_NOISE_SEED,
    }
save_pose_snapshot("01_da3_raw", _da3_c2ws, view_names, Ks=_da3_Ks, extra=_extra)
if HAS_GEO_JSON:
    _ref_geo = np.stack([GEO_REF[n]["c2w"] for n in view_names])
    _extra["geo_align"] = geo_align_info
    _extra["note"] = "GEO_REF is position-only; compare camera centers (rotations from DA3)"
    pose_audit_log("01_da3_vs_geo", _da3_c2ws, view_names, ref_c2ws=_ref_geo, ref_label="geo_json",
                   Ks=_da3_Ks, align_sim3=False, extra=_extra)
else:
    print("No geo JSON refs — run §2.6 first.")


### 8.5 — Propagate clicks to every view (needs §8 DA3 + §2.5.4 points)

Back-projects the points you clicked in §2.5.4 through DA3 depth into 3D, then re-projects them
into **all** views (with an occlusion test), so every view gets a positive/negative prompt from a
few clicks. Then re-runs SAM3 with those prompts.

Run this **after §8**, then continue with §9. Skip it if `PER_VIEW_POINTS` is empty.


In [ ]:
# 8.5 — Propagate §2.5.4 clicks to all views through DA3 geometry, then re-segment
import json
import numpy as np
from pathlib import Path

_pvp = globals().get("PER_VIEW_POINTS") or {}
if not _pvp:
    _p = OUTPUT_DIR / "per_view_points.json"
    if _p.exists():
        _pvp = {int(k): v for k, v in json.loads(_p.read_text()).items()}
        print("Loaded clicks from", _p)

if not _pvp:
    print("No point prompts — run §2.5.4 and click a few views (skip if text prompt is correct).")
else:
    OCCL_TOL = 0.06        # |z_proj - z_depth| / z_proj tolerance for 'visible in this view'
    _da3 = np.load(DA3_NPZ)
    _dep_all, _K_all, _ext_all = _da3["depth"], _da3["intrinsics"], _da3["extrinsics"]
    _ppg_stems = [Path(str(p)).stem for p in _da3["image_files"]]   # not `_stems`: that name is the helper fn from §2.5.5

    def _key(p):
        try:
            return (0, int(p.stem), p.stem)
        except ValueError:
            return (1, 0, p.stem)

    _files = sorted(list(IMAGES_DIR.glob("*.png")) + list(IMAGES_DIR.glob("*.jpg")), key=_key)
    _img_wh = {}
    from PIL import Image as _PILImage
    for f in _files:
        _img_wh[f.stem] = _PILImage.open(f).size          # (W, H) of the clicked image

    # ---- 1. clicked pixels -> 3D points in DA3 world ----
    pts3d, lbl3d = [], []
    for vi, vp in _pvp.items():
        stem = _files[vi].stem
        j = _ppg_stems.index(stem)
        Z = _dep_all[j]
        dh, dw = Z.shape
        K = _K_all[j].astype(np.float64)
        c2w = _to_c2w(_ext_all[j])
        iw, ih = _img_wh[stem]
        for (px, py), lbl in zip(vp["points"], vp["labels"]):
            u = int(round(px * dw / iw)); v = int(round(py * dh / ih))
            u = min(max(u, 0), dw - 1); v = min(max(v, 0), dh - 1)
            z = float(Z[v, u])
            if z <= 0:
                print(f"  skip click ({px},{py}) in view {vi}: no depth")
                continue
            Xc = np.array([(u - K[0, 2]) / K[0, 0] * z, (v - K[1, 2]) / K[1, 1] * z, z])
            pts3d.append(c2w[:3, :3] @ Xc + c2w[:3, 3])
            lbl3d.append(int(lbl))
    pts3d = np.asarray(pts3d, np.float64)
    lbl3d = np.asarray(lbl3d, int)
    print(f"Lifted {len(pts3d)} clicks to 3D ({int((lbl3d == 1).sum())} positive, "
          f"{int((lbl3d == 0).sum())} negative)")

    # ---- 2. project into every view, keep the ones that are actually visible ----
    PER_VIEW_POINTS_ALL = {}
    for vi, f in enumerate(_files):
        j = _ppg_stems.index(f.stem)
        Z = _dep_all[j]
        dh, dw = Z.shape
        K = _K_all[j].astype(np.float64)
        w2c = np.linalg.inv(_to_c2w(_ext_all[j]))
        iw, ih = _img_wh[f.stem]
        pts, lbls = [], []
        for X, lbl in zip(pts3d, lbl3d):
            Xc = w2c[:3, :3] @ X + w2c[:3, 3]
            if Xc[2] <= 1e-6:
                continue
            u = K[0, 0] * Xc[0] / Xc[2] + K[0, 2]
            v = K[1, 1] * Xc[1] / Xc[2] + K[1, 2]
            ui, vi_ = int(round(u)), int(round(v))
            if not (0 <= ui < dw and 0 <= vi_ < dh):
                continue
            zd = float(Z[vi_, ui])
            if zd <= 0 or abs(zd - Xc[2]) / Xc[2] > OCCL_TOL:
                continue                                   # occluded in this view
            pts.append([int(round(u * iw / dw)), int(round(v * ih / dh))])
            lbls.append(int(lbl))
        if pts:
            PER_VIEW_POINTS_ALL[vi] = {"points": pts, "labels": lbls}

    _cov = len(PER_VIEW_POINTS_ALL)
    _pos = sum(1 for v in PER_VIEW_POINTS_ALL.values() if 1 in v["labels"])
    print(f"Propagated prompts to {_cov}/{len(_files)} views ({_pos} with a positive point)")
    if _cov < 0.6 * len(_files):
        print("  Coverage is low — click a few more views in §2.5.4 (spread around the orbit),")
        print("  or raise OCCL_TOL if the object is thin / depth is noisy.")

    _p = OUTPUT_DIR / "per_view_points_propagated.json"
    _p.write_text(json.dumps({str(k): v for k, v in PER_VIEW_POINTS_ALL.items()}))
    print("Saved ->", _p)

    # ---- 3. re-segment every view with the propagated prompts ----
    FORCE_RESEGMENT = True
    res = segment_folder(VIEW_DIR, IMAGES_DIR, per_view_points=PER_VIEW_POINTS_ALL,
                         label="ultrra_aerial+points")

    # ---- Back up propagated points + masks to Drive ----
    if Path("/content/drive/MyDrive").exists():
        import shutil
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(_p, DRIVE_OUT / _p.name)
        _drive_masks = DRIVE_OUT / "masks"
        _drive_masks.mkdir(parents=True, exist_ok=True)
        _mask_dir = VIEW_DIR / MASK_NAME
        _mc = 0
        for _mp in sorted(_mask_dir.glob("*.png")):
            _dst = _drive_masks / _mp.name
            if not _dst.exists() or _mp.stat().st_mtime > _dst.stat().st_mtime:
                shutil.copy2(_mp, _dst); _mc += 1
        print(f"Backed up to Drive: propagated points + {_mc} masks")

    print("Re-run §9 (Stage-1) and §12 after this — masks changed.")


> **Skip §9 when `SKIP_STAGE1=True`** (default for this notebook — raw BA-NeuS, no Stage-1 prior).

## 9 — Stage-1 reconstruction (optional MV-SAM3D coarse mesh)

Only needed if `BANEUS_USE_PRIOR=True`. Default raw path uses DA3 depth only.


In [ ]:
# 9.0 — Build the pipeline (same checkpoint layout as mvsam3d_example_current.ipynb)
import os
from inference import Inference   # MV-SAM3D wrapper around the pipeline

# The working notebook exposes pipeline.yaml by symlinking the SAM 3D checkpoint into
# MV-SAM3D/checkpoints/hf, i.e. it runs (after snapshot_download of facebook/sam-3d-objects):
#   !ln -s /content/sam-3d-objects/checkpoints/hf/checkpoints /content/MV-SAM3D/checkpoints/hf
# So run the heavy install + checkpoint download from mvsam3d_example_current.ipynb first.
MODEL_TAG = "hf"
PIPELINE_YAML = MVSAM3D_ROOT / "checkpoints" / MODEL_TAG / "pipeline.yaml"

if not PIPELINE_YAML.exists():
    # Recreate the same symlink if the SAM 3D checkpoint is already downloaded
    src = SAM3D_ROOT / "checkpoints" / "hf" / "checkpoints"
    dst = MVSAM3D_ROOT / "checkpoints" / "hf"
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(src, dst)
        print("Created checkpoint symlink:", dst, "->", src)

assert PIPELINE_YAML.exists(), (
    f"{PIPELINE_YAML} missing. Download the SAM 3D checkpoint and create the "
    f"checkpoints/hf symlink via mvsam3d_example_current.ipynb first."
)

inference = Inference(str(PIPELINE_YAML))
pipeline = inference._pipeline
ss_decoder = pipeline.models["ss_decoder"]      # <- handle used by the guidance

from sam3d_objects.pipeline.inference_utils import get_pose_decoder
# pipeline.yaml often sets pose_decoder_name="default" (no-op). Multi-view staging needs SSI.
pipeline.pose_decoder = get_pose_decoder("ScaleShiftInvariant")
print("pose_decoder: ScaleShiftInvariant")
print("Pipeline loaded. ss_decoder:", type(ss_decoder).__name__)


In [ ]:
# 9.1 — Match DA3 pointmaps to the views (Stage-1 input), in view_names order
_da3_idx = {st: i for i, st in enumerate(da3_stems)}
view_pointmaps = []
for name in view_names:
    if name in _da3_idx:
        view_pointmaps.append(da3_pointmaps[_da3_idx[name]])
    else:
        print(f"  WARN: no DA3 pointmap for view {name}; pipeline will compute one")
        view_pointmaps.append(None)
print(f"view pointmaps: {sum(p is not None for p in view_pointmaps)}/{len(view_names)}")

In [ ]:
# 9.2 — preprocess_views: faithful copy of run_multi_view's per-view loop.
# Produces (ss_input_dicts, slat_input_dicts, raw_pointmaps) for ONE view set, so
# we can feed Stage 1 and Stage 2 different sets. Mirrors inference_pipeline.py.
import numpy as np, torch
from PIL import Image

def preprocess_views(pipeline, view_images, view_masks=None, view_pointmaps=None):
    n = len(view_images)
    if view_masks is None:
        view_masks = [None] * n
    if view_pointmaps is None:
        view_pointmaps = [None] * n
    ss_dicts, slat_dicts, raw_pms = [], [], []

    for i, (image, mask, ext_pm) in enumerate(zip(view_images, view_masks, view_pointmaps)):
        # ---- merge mask into the alpha channel (RGBA), exactly like run_multi_view ----
        if mask is not None:
            image = np.array(image)
            mask = np.array(mask)
            if mask.dtype == bool:
                mask = mask.astype(np.uint8) * 255
            elif mask.dtype != np.uint8:
                mask = (mask * 255).astype(np.uint8) if mask.max() <= 1.0 else mask.astype(np.uint8)
            if mask.ndim == 2:
                mask = mask[..., None]
            if image.shape[-1] == 3:
                rgba = np.concatenate([image, mask], axis=-1).astype(np.uint8)
            elif image.shape[-1] == 4:
                rgba = np.concatenate([image[..., :3], mask], axis=-1).astype(np.uint8)
            else:
                raise ValueError(f"Unexpected image shape: {image.shape}")
        else:
            rgba = np.array(image)   # assume already RGBA
        rgba_pil = Image.fromarray(rgba)

        # ---- pointmap + per-stage preprocessing ----
        if hasattr(pipeline, "compute_pointmap"):
            if ext_pm is not None:
                pmd = pipeline.compute_pointmap(rgba_pil, pointmap=torch.from_numpy(ext_pm).float())
            else:
                pmd = pipeline.compute_pointmap(rgba_pil, pointmap=None)
            pointmap = pmd["pointmap"]
            if pointmap is not None:
                pmm = pointmap.detach()
                if hasattr(type(pipeline), "_down_sample_img"):
                    pmm = type(pipeline)._down_sample_img(pmm)
                raw_pms.append(pmm.cpu().permute(1, 2, 0).numpy())
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor, pointmap=pointmap))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))
        else:
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))

    return ss_dicts, slat_dicts, raw_pms


def run_split_staging(pipeline, *, stage1_images, stage1_masks, stage1_pointmaps,
                      stage2_images, stage2_masks,
                      seed=0, stage1_steps=25, stage2_steps=25, mode="multidiffusion",
                      decode_formats=("mesh", "gaussian"),
                      ss_weighting=True, ss_entropy_layer=9, ss_entropy_alpha=60.0, ss_warmup_steps=1,
                      with_mesh_postprocess=False, with_texture_baking=False, use_vertex_color=True,
                      stage1_only=False, weighting_config=None):
    """Stage 1 on stage1_* (aerial); Stage 2 (SLAT) on stage2_* (ground) at the
    Stage-1 coords. Faithful to run_multi_view, just split across two view sets."""
    if seed is not None:
        torch.manual_seed(seed)

    # ---- Stage 1: AERIAL -> coarse sparse structure (coords) ----
    print(f"[Stage 1] preprocessing {len(stage1_images)} AERIAL view(s)...")
    ss_dicts, _, raw_pms = preprocess_views(pipeline, stage1_images, stage1_masks, stage1_pointmaps)
    print("[Stage 1] sampling sparse structure...")
    ss_return = pipeline.sample_sparse_structure_multi_view(
        ss_dicts, inference_steps=stage1_steps, mode=mode,
        ss_weighting=ss_weighting, ss_entropy_layer=ss_entropy_layer,
        ss_entropy_alpha=ss_entropy_alpha, ss_warmup_steps=ss_warmup_steps,
    )
    pm_scale = ss_dicts[0].get("pointmap_scale", None)
    pm_shift = ss_dicts[0].get("pointmap_shift", None)
    from sam3d_objects.pipeline.inference_utils import get_pose_decoder
    _pose_dec = get_pose_decoder("ScaleShiftInvariant")
    _pose_raw = dict(ss_return)   # keep the raw MM-DiT keys; decode reads from these
    ss_return["pose_raw"] = _pose_raw
    ss_return.update(_pose_dec(_pose_raw, scene_scale=pm_scale, scene_shift=pm_shift))
    ss_return["pointmap_scale"] = pm_scale
    ss_return["pointmap_shift"] = pm_shift
    # translation/rotation/scale are metric now; decoding again would re-exp() the scale
    ss_return["pose_decoded"] = True
    if "all_view_poses_raw" in ss_return and hasattr(pipeline, "_decode_all_view_poses"):
        ss_return["all_view_poses_decoded"] = pipeline._decode_all_view_poses(
            ss_return["all_view_poses_raw"], ss_dicts)
    if "scale" in ss_return:
        ss_return["scale"] = ss_return["scale"] * ss_return["downsample_factor"]
    coords = ss_return["coords"]
    print(f"[Stage 1] coords: {tuple(coords.shape)}")

    if stage1_only:
        ss_return["voxel"] = ss_return["coords"][:, 1:] / 64 - 0.5
        print("[Stage 1] stage1_only=True -> returning sparse structure (no Stage 2).")
        return ss_return

    # ---- Stage 2: GROUND -> SLAT on the Stage-1 coords ----
    print(f"[Stage 2] preprocessing {len(stage2_images)} GROUND view(s)...")
    _, slat_dicts, _ = preprocess_views(pipeline, stage2_images, stage2_masks, None)
    print("[Stage 2] sampling structured latent...")
    if weighting_config is not None:
        # visibility/mixed needs the Stage-1 object pose to place ground cameras in canonical space
        if getattr(weighting_config, "weight_source", "entropy") in ("visibility", "mixed"):
            weighting_config.object_pose = {
                "scale": ss_return.get("scale"),
                "rotation": ss_return.get("rotation"),
                "translation": ss_return.get("translation"),
            }
        slat, _ = pipeline.sample_slat_multi_view_weighted(
            slat_dicts, coords, inference_steps=stage2_steps, weighting_config=weighting_config)
    else:
        slat = pipeline.sample_slat_multi_view(slat_dicts, coords, inference_steps=stage2_steps, mode=mode)

    outputs = pipeline.decode_slat(slat, list(decode_formats))
    outputs = pipeline.postprocess_slat_output(
        outputs, with_mesh_postprocess, with_texture_baking, use_vertex_color)
    result = {**ss_return, **outputs, "view_ss_input_dicts": ss_dicts}
    if raw_pms:
        result["raw_view_pointmaps"] = raw_pms
    print("Finished split-staging run.")
    return result

print("preprocess_views + run_split_staging ready.")


## 9.3 — Decode, preview & save the Stage-1 geometry

Runs Stage 1 on the aerial views, decodes the sparse-structure latent to a 64³ occupancy, turns
it into a coarse mesh, shows a 3-angle preview, and copies everything to Drive. The raw
`stage1_occupancy.npz` (probabilities + voxel coords in the canonical `[-0.5, 0.5]` cube) is the
artifact the MVS notebook will later read in place of OSM.

In [ ]:
# 9.3 — Decode + preview the Stage-1 coarse geometry (va_split aerial); save to Drive
import numpy as np, torch
try:
    from skimage import measure
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
    from skimage import measure
import trimesh, matplotlib.pyplot as plt
from pathlib import Path

ss_decoder = pipeline.models["ss_decoder"]
_dev = "cuda" if torch.cuda.is_available() else "cpu"

print("Running Stage 1 (va_split aerial) to produce coarse geometry...")
ss = run_split_staging(
    pipeline,
    stage1_images=view_images, stage1_masks=view_masks, stage1_pointmaps=view_pointmaps,
    stage2_images=view_images, stage2_masks=view_masks,   # unused under stage1_only=True
    seed=SEED, stage1_steps=STAGE1_STEPS, mode=MODE,
    ss_weighting=SS_WEIGHTING, ss_entropy_alpha=SS_ENTROPY_ALPHA, ss_warmup_steps=SS_WARMUP_STEPS,
    stage1_only=True,
)

# Decode the dense shape latent -> continuous 64^3 occupancy (same call the pipeline uses).
shape_latent = ss.get("shape")
assert torch.is_tensor(shape_latent) and shape_latent.dim() == 3, "No Stage-1 'shape' latent to decode."
with torch.no_grad(), torch.autocast(device_type=_dev, enabled=(_dev == "cuda")):
    B = shape_latent.shape[0]
    vol = shape_latent.permute(0, 2, 1).contiguous().view(B, 8, 16, 16, 16)
    logits = ss_decoder(vol.to(_dev))                 # (B,1,64,64,64)
prob = torch.sigmoid(logits[0, 0].float()).cpu().numpy()
print("Decoded occupancy:", prob.shape, "occupied frac", round(float((prob > 0.5).mean()), 4))

# Marching cubes -> coarse surface; fall back to binary voxel coords if degenerate.
try:
    verts, faces, normals, _ = measure.marching_cubes(prob, level=0.5)
    srcname = "decoder occupancy @0.5"
except Exception as e:
    print("(soft marching cubes failed:", e, "-> using voxel coords)")
    c = ss["coords"].detach().cpu().numpy()
    occ = np.zeros((64, 64, 64), np.float32); occ[c[:, 1], c[:, 2], c[:, 3]] = 1.0
    verts, faces, normals, _ = measure.marching_cubes(occ, level=0.5)
    srcname = "binary voxels @0.5"
verts = verts / 64.0 - 0.5                            # center into the canonical [-0.5, 0.5] cube
mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals, process=True)
print(f"Stage-1 coarse mesh ({srcname}): {len(mesh.vertices)} verts, {len(mesh.faces)} faces")

out_dir = OUTPUT_DIR / "stage1"; out_dir.mkdir(parents=True, exist_ok=True)
obj_path = out_dir / "stage1_coarse_mesh.obj"; glb_path = out_dir / "stage1_coarse_mesh.glb"
mesh.export(obj_path); mesh.export(glb_path)
# Raw occupancy (prob + voxel coords) so mvs_osm.ipynb can consume this geometry later.
occ_path = out_dir / "stage1_occupancy.npz"
np.savez(occ_path, prob=prob, coords=ss["coords"].detach().cpu().numpy())

# Object pose + world AABB — shared coordinate frame for BA-NeuS (§12)
from pytorch3d.transforms import quaternion_to_matrix, Transform3d

def compose_transform(scale, rotation, translation, device="cpu"):
    """PyTorch3D Translate needs (N,3); keep pose + points on the same device."""
    if translation is None:
        raise ValueError("compose_transform: translation is None")
    dev = torch.device(device)
    scale = torch.as_tensor(scale, dtype=torch.float32, device=dev)
    rotation = torch.as_tensor(rotation, dtype=torch.float32, device=dev)
    translation = torch.as_tensor(translation, dtype=torch.float32, device=dev)
    if translation.dim() == 1:
        translation = translation.unsqueeze(0)
    if scale.dim() == 1:
        scale = scale.unsqueeze(0)
    if rotation.dim() == 2:
        rotation = rotation.unsqueeze(0)
    tfm = Transform3d(device=dev, dtype=torch.float32)
    return tfm.scale(scale).rotate(rotation).translate(translation)

def _squeeze_pose_tensor(x):
    if x is None:
        return None
    t = x.detach().float() if torch.is_tensor(x) else torch.as_tensor(x, dtype=torch.float32)
    while t.dim() > 1:
        t = t.squeeze(0)
    return t

def _decode_stage1_pose(ss_out):
    from sam3d_objects.pipeline.inference_utils import get_pose_decoder
    # Decode ONCE. §9.2 already ran the ScaleShiftInvariant decoder (and the
    # downsample_factor rescale), so translation/rotation/scale are metric here.
    # Re-decoding feeds decoded values back in as raw ones: scale gets exp()'d a
    # second time and the pointmap scale/shift are applied twice, which inflates the
    # Stage-1 world frame by orders of magnitude and desynchronises it from DA3.
    if not ss_out.get("pose_decoded"):
        dec = get_pose_decoder("ScaleShiftInvariant")
        ss_out.update(dec(
            ss_out,
            scene_scale=ss_out.get("pointmap_scale"),
            scene_shift=ss_out.get("pointmap_shift"),
        ))
        ss_out["pose_decoded"] = True
    else:
        print("Stage-1 pose already decoded in §9.2 — reusing metric scale/rotation/translation.")
    missing = [k for k in ("translation", "rotation", "scale") if ss_out.get(k) is None]
    if missing:
        raise RuntimeError(
            "Stage-1 pose decode failed; missing " + ", ".join(missing)
            + ". ss keys: " + ", ".join(sorted(ss_out.keys()))
        )
    return _squeeze_pose_tensor(ss_out["scale"]), _squeeze_pose_tensor(ss_out["rotation"]), _squeeze_pose_tensor(ss_out["translation"])

sc, r_pose, t_pose = _decode_stage1_pose(ss)
sc, r_pose, t_pose = sc.detach().float().cpu(), r_pose.detach().float().cpu(), t_pose.detach().float().cpu()

# Decoded pose maps canonical -> ref-view camera space.
# Chain: canonical -> pose -> P3D_TO_OPENCV -> c2w(ref view) -> OpenCV world.
_da3 = np.load(DA3_NPZ)
_ref_stem = view_names[0]
_da3_stems = [Path(str(p)).stem for p in _da3["image_files"]]
_ref_idx = _da3_stems.index(_ref_stem)
_ref_c2w = _to_c2w(_da3["extrinsics"][_ref_idx])
print(f"Stage-1 decode reference view: {_ref_stem} (DA3 idx {_ref_idx})")

_M_can2w = stage1_canonical_to_da3_world_matrix(sc, r_pose, t_pose, c2w_opencv=_ref_c2w, device="cpu")
verts_w = stage1_canonical_to_da3_world(mesh.vertices, _M_can2w)
_v_rt = stage1_da3_world_to_canonical(verts_w, _M_can2w)
_recomp_err = float(np.abs(_v_rt - mesh.vertices).max())
print(f"Canonical round-trip error: {_recomp_err:.6f} (want ~0)")
_center_can = 0.5 * (mesh.vertices.min(0) + mesh.vertices.max(0))
_center_direct = _canonical_to_opencv_world_points(
    _center_can[None], sc, r_pose, t_pose, c2w_opencv=_ref_c2w, device="cpu")[0]
_center_matrix = stage1_canonical_to_da3_world(np.array([_center_can]), _M_can2w)[0]
print(f"Matrix vs direct center diff: {float(np.abs(_center_direct - _center_matrix).max()):.6f} (want ~0)")
center_w = 0.5 * (verts_w.min(0) + verts_w.max(0))
radius_w = float(np.linalg.norm(verts_w - center_w, axis=1).max() * 1.15 + 1e-6)
frame_path = out_dir / "stage1_frame.npz"
np.savez(frame_path,
         scale=sc.numpy(), rotation=r_pose.numpy(), translation=t_pose.numpy(),
         center=center_w, radius=np.float32(radius_w),
         M_canonical_to_da3_world=_M_can2w.astype(np.float32),
         ref_c2w=_ref_c2w.astype(np.float32))
print("Saved stage1_frame:", frame_path, "| center", center_w.round(3), "radius", round(radius_w, 4))

print("Saved locally:", obj_path.name, glb_path.name, occ_path.name, "->", out_dir)

# 3-angle inline preview (rightmost is near top-down); also saved as PNG.
vx, vy, vz = mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2]
fig = plt.figure(figsize=(15, 5))
for i, (az, el) in enumerate([(-60, 25), (30, 25), (90, 80)]):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    ax.plot_trisurf(vx, vy, vz, triangles=mesh.faces, cmap="viridis", linewidth=0, antialiased=False)
    ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z"); ax.set_title(f"Stage-1 mesh ({az},{el})")
plt.tight_layout()
png_path = out_dir / "stage1.png"
fig.savefig(png_path, dpi=120, bbox_inches="tight")
plt.show()
print("Saved preview image:", png_path.name)

# Persist to Drive (same convention as split_staging, namespaced by scene folder).
try:
    import shutil
    if Path("/content/drive/MyDrive").exists():
        drive_s1 = DRIVE_OUT / "stage1"
        drive_s1.mkdir(parents=True, exist_ok=True)
        for p in (obj_path, glb_path, png_path, occ_path, frame_path):
            shutil.copy2(p, drive_s1 / p.name)
        print("Copied Stage-1 geometry to Drive:", drive_s1)
    else:
        print("Drive not mounted at /content/drive/MyDrive — run the drive.mount cell first.")
except Exception as e:
    print("Stage-1 Drive copy failed:", e)

> **Skip when `USE_PREPROCESSED=True`**

## 10 — Release MV-SAM3D GPU memory

BA-NeuS trains a separate SDF network. Free the SAM3D pipeline before the next section.


In [ ]:
import gc, torch
try:
    del pipeline, inference, ss_decoder
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory cleared for BA-NeuS.")


## 11 — Install Bundle-Adjusting NeuS

Clones [DarkGeekMS/bundle-adjusting-neus](https://github.com/DarkGeekMS/bundle-adjusting-neus) and applies small Colab compatibility patches (torch 2.5, disable wandb login).


In [ ]:
# 11.1 — Clone BA-NeuS + Colab compatibility patches
import os, sys, re
from pathlib import Path

BANEUS_ROOT = Path("/content/bundle-adjusting-neus")
if not BANEUS_ROOT.exists():
    !git clone --depth 1 https://github.com/DarkGeekMS/bundle-adjusting-neus.git /content/bundle-adjusting-neus

os.environ["WANDB_DISABLED"] = "true"
!pip install -q pyhocon icecream PyMCubes wandb tqdm plotly trimesh

def _patch_file(path, subs):
    p = Path(path)
    if not p.exists():
        return
    txt = p.read_text()
    orig = txt
    for old, new in subs:
        if old in txt:
            txt = txt.replace(old, new)
    if txt != orig:
        p.write_text(txt)
        print("patched", p.name)

# --- prior stub (real callback registered in §13.0) ---
(BANEUS_ROOT / "optimizers" / "prior_patch.py").write_text(
    "import torch\n_PRIOR = None\n"
    "def register_prior(fn):\n    global _PRIOR; _PRIOR = fn\n"
    "def prior_loss(renderer, n=4096):\n"
    "    if _PRIOR is None:\n        return torch.zeros((), device='cuda')\n"
    "    pts = torch.randn(n, 3, device='cuda')\n"
    "    pts = pts / (pts.norm(dim=-1, keepdim=True) + 1e-6) * torch.rand(n, 1, device='cuda') ** 0.33\n"
    "    tgt = _PRIOR(pts).detach()\n"
    "    pred = renderer.sdf_network.sdf(pts)\n"
    "    return ((pred - tgt) ** 2).mean()\n"
)

# --- main.py: deprecated default tensor type ---
_patch_file(BANEUS_ROOT / "main.py", [
    ('torch.set_default_tensor_type("torch.cuda.FloatTensor")', 'torch.set_default_dtype(torch.float32)'),
])

# --- dataset.py: torch 2.x meshgrid + safe sequential src indices ---
_patch_file(BANEUS_ROOT / "dataset" / "dataset.py", [
    ("pixels_x, pixels_y = torch.meshgrid(tx, ty)", "pixels_x, pixels_y = torch.meshgrid(tx, ty, indexing='ij')"),
    (
        "src_idxs = [img_idx.item() - 1, (img_idx.item() + 1) % self.n_images]",
        "ii = int(img_idx.item() if hasattr(img_idx, 'item') else img_idx)\n            src_idxs = [(ii - 1) % self.n_images, (ii + 1) % self.n_images]",
    ),
])

# --- renderer.py: full CUDA literal patches (z_vals, zeros, ones, arange) ---
def _patch_renderer_cuda_literals(rp=BANEUS_ROOT / "models" / "renderer.py"):
    """Patch all CPU-default tensor literals in renderer.py for CUDA training."""
    import re as _re
    p = Path(rp)
    if not p.exists():
        return False
    txt = p.read_text()
    orig = txt

    # Repair earlier bad patch that commented out cumprod(..., -1)
    txt = txt.replace("], -1)  # AERIAL_ONES_CUDA, -1", "], -1), -1")

    txt = _re.sub(
        r"torch\.Tensor\(\[sample_dist\]\)\.expand\(dists\[\.\.\., :1\]\.shape\)",
        "torch.tensor([sample_dist], device=dists.device, dtype=dists.dtype).expand(dists[..., :1].shape)",
        txt,
    )

    subs = [
        (
            "z_vals = torch.linspace(0.0, 1.0, self.n_samples)\n        z_vals = near + (far - near) * z_vals[None, :]",
            "_dev, _dt = rays_o.device, rays_o.dtype\n        near = near.to(device=_dev, dtype=_dt)\n        far = far.to(device=_dev, dtype=_dt)\n        z_vals = torch.linspace(0.0, 1.0, self.n_samples, device=_dev, dtype=_dt)  # AERIAL_ZVALS_CUDA\n        z_vals = near + (far - near) * z_vals[None, :]",
        ),
        (
            "z_vals = torch.linspace(0.0, 1.0, self.n_samples, device=near.device, dtype=near.dtype)  # AERIAL_ZVALS_CUDA\n        z_vals = near + (far - near) * z_vals[None, :]",
            "_dev, _dt = rays_o.device, rays_o.dtype\n        near = near.to(device=_dev, dtype=_dt)\n        far = far.to(device=_dev, dtype=_dt)\n        z_vals = torch.linspace(0.0, 1.0, self.n_samples, device=_dev, dtype=_dt)  # AERIAL_ZVALS_CUDA\n        z_vals = near + (far - near) * z_vals[None, :]",
        ),
        (
            "t_rand = torch.rand([batch_size, 1]) - 0.5",
            "t_rand = torch.rand([batch_size, 1], device=_dev, dtype=_dt) - 0.5  # AERIAL_ZVALS_CUDA",
        ),
        (
            "t_rand = torch.rand([batch_size, z_vals_outside.shape[-1]])",
            "t_rand = torch.rand([batch_size, z_vals_outside.shape[-1]], device=_dev, dtype=_dt)  # AERIAL_ZVALS_CUDA",
        ),
        (
            "[torch.zeros([batch_size, 1]), cos_val[:, :-1]], dim=-1",
            "[torch.zeros([batch_size, 1], device=cos_val.device, dtype=cos_val.dtype), cos_val[:, :-1]], dim=-1",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1]), 1.0 - alpha + 1e-7], -1), -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha.device, dtype=alpha.dtype), 1.0 - alpha + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1]), 1.0 - alpha_in + 1e-7], -1), -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha_in.device, dtype=alpha_in.dtype), 1.0 - alpha_in + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha.device, dtype=alpha.dtype), 1.0 - alpha + 1e-7], -1)  # AERIAL_ONES_CUDA, -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha.device, dtype=alpha.dtype), 1.0 - alpha + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha_in.device, dtype=alpha_in.dtype), 1.0 - alpha_in + 1e-7], -1)  # AERIAL_ONES_CUDA, -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha_in.device, dtype=alpha_in.dtype), 1.0 - alpha_in + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "inv_s = deviation_network(torch.zeros([1, 3]))[:, :1].clip(",
            "inv_s = deviation_network(torch.zeros([1, 3], device=pts.device, dtype=pts.dtype))[:, :1].clip(",
        ),
        (
            "torch.arange(batch_size)[:, None]",
            "torch.arange(batch_size, device=z_vals.device)[:, None]",
        ),
        (
            "pc_loss = torch.tensor(0)",
            "pc_loss = torch.tensor(0.0, device=pts.device, dtype=pts.dtype)",
        ),
        (
            "feat_loss = torch.tensor(0)",
            "feat_loss = torch.tensor(0.0, device=pts.device, dtype=pts.dtype)",
        ),
    ]
    for old, new in subs:
        if old in txt:
            txt = txt.replace(old, new)

    for old in [
        "t_rand = torch.rand([batch_size, 1], device=near.device, dtype=near.dtype) - 0.5  # AERIAL_ZVALS_CUDA",
        "t_rand = torch.rand([batch_size, z_vals_outside.shape[-1]], device=near.device, dtype=near.dtype) - 0.5  # AERIAL_ZVALS_CUDA",
    ]:
        if old in txt:
            txt = txt.replace(old, old.replace("near.device", "_dev").replace("near.dtype", "_dt"))

    if txt != orig:
        p.write_text(txt)
        print("patched renderer.py (full CUDA literals)")
        return True
    if "device=alpha.device" in txt and "], -1)  # AERIAL_ONES_CUDA, -1" not in txt:
        print("renderer.py already has full CUDA literal patches")
        return False
    print("WARN: renderer.py CUDA literal patch incomplete — check file manually")
    return False


_patch_renderer_cuda_literals()


# --- geometry_utils.py: sample_pdf u on cdf.device (searchsorted CUDA) ---
_geom = BANEUS_ROOT / "utils" / "geometry_utils.py"
if _geom.exists():
    _gt = _geom.read_text()
    _g_old = """        u = torch.linspace(
            0.0 + 0.5 / n_samples, 1.0 - 0.5 / n_samples, steps=n_samples
        )
        u = u.expand(list(cdf.shape[:-1]) + [n_samples])
    else:
        u = torch.rand(list(cdf.shape[:-1]) + [n_samples])"""
    _g_new = """        u = torch.linspace(
            0.0 + 0.5 / n_samples, 1.0 - 0.5 / n_samples, steps=n_samples,
            device=cdf.device, dtype=cdf.dtype,
        )  # AERIAL_SAMPLEPDF_CUDA
        u = u.expand(list(cdf.shape[:-1]) + [n_samples])
    else:
        u = torch.rand(list(cdf.shape[:-1]) + [n_samples], device=cdf.device, dtype=cdf.dtype)  # AERIAL_SAMPLEPDF_CUDA"""
    if "AERIAL_SAMPLEPDF_CUDA" not in _gt and _g_old in _gt:
        _geom.write_text(_gt.replace(_g_old, _g_new))
        print("patched geometry_utils.py (sample_pdf on CUDA)")
    elif "AERIAL_SAMPLEPDF_CUDA" in _gt:
        print("geometry_utils.py already has sample_pdf CUDA patch")
    else:
        print("WARN: sample_pdf block not found in geometry_utils.py")


# --- ba_neus_optimizer.py: wandb off + prior loss + SDF prefit load ---
_opt = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
otxt = _opt.read_text()
if "mode='disabled'" not in otxt:
    otxt = otxt.replace("wandb.init(project=", "wandb.init(mode='disabled', project=")
if "self.prior_weight" not in otxt:
    otxt = otxt.replace(
        'self.object_mask_type = self.conf.get_string("train.object_mask_type")',
        'self.object_mask_type = self.conf.get_string("train.object_mask_type")\n'
        '        self.prior_weight = self.conf.get_float("train.prior_weight", default=0.0)',
    )
if "prior_loss(self.renderer)" not in otxt:
    otxt = otxt.replace(
        "from utils.camera_utils import scale_camera",
        "from utils.camera_utils import scale_camera\nfrom optimizers.prior_patch import prior_loss",
    )
    otxt = otxt.replace(
        "+ pc_loss * pc_weight",
        "+ pc_loss * pc_weight\n            + prior_loss(self.renderer) * self.prior_weight",
    )
if "BANEUS_SDF_PREFIT" not in otxt:
    otxt = otxt.replace(
        "self.renderer = NeuSRenderer(",
        'prefit = os.environ.get("BANEUS_SDF_PREFIT")\n'
        '        if prefit and os.path.isfile(prefit):\n'
        '            _ckpt = torch.load(prefit, map_location=self.device, weights_only=False)\n'
        '            self.sdf_network.load_state_dict(_ckpt["sdf_network"])\n'
        '            print("Loaded Stage-1 SDF prefit from", prefit)\n'
        "        self.renderer = NeuSRenderer(",
    )


def _patch_ba_neus_background_rgb():
    """Fix background_rgb device: train uses rays_o tensor; validate uses rays_o_batch."""
    p = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
    if not p.exists():
        return
    txt = p.read_text()
    orig = txt

    # repair earlier bad global replace
    txt = txt.replace(
        "background_rgb = torch.ones([1, 3], device=rays_o.device, dtype=rays_o.dtype)  # AERIAL_BG_CUDA if self.use_white_bkgd else None",
        "background_rgb = (torch.ones([1, 3], device=rays_o_batch.device, dtype=rays_o_batch.dtype) if self.use_white_bkgd else None)  # AERIAL_BG_CUDA",
    )

    subs = [
        (
            "            background_rgb = None\n            if self.use_white_bkgd:\n                background_rgb = torch.ones([1, 3])",
            "            background_rgb = None\n            if self.use_white_bkgd:\n                background_rgb = torch.ones([1, 3], device=rays_o.device, dtype=rays_o.dtype)  # AERIAL_BG_CUDA",
        ),
        (
            "            background_rgb = torch.ones([1, 3]) if self.use_white_bkgd else None",
            "            background_rgb = (torch.ones([1, 3], device=rays_o_batch.device, dtype=rays_o_batch.dtype) if self.use_white_bkgd else None)  # AERIAL_BG_CUDA",
        ),
    ]
    for old, new in subs:
        if old in txt:
            txt = txt.replace(old, new)

    if txt != orig:
        p.write_text(txt)
        print("patched ba_neus_optimizer.py (background_rgb per-context)")
    elif "AERIAL_BG_CUDA" in txt:
        print("ba_neus_optimizer.py already has background_rgb patch")

_patch_ba_neus_background_rgb()
_opt.write_text(otxt)


# --- point_cloud_loss.py: meshgrid indexing ---
_patch_file(BANEUS_ROOT / "losses" / "point_cloud_loss.py", [
    ("torch.meshgrid(torch.arange(0, h), torch.arange(0, w))",
     "torch.meshgrid(torch.arange(0, h), torch.arange(0, w), indexing='ij')"),
])



# --- AERIAL_SEQUENTIAL_PATCH: skip VisMVSNet unless cam4feat/ exists (aerial has none) ---
import re as _re
_ds = BANEUS_ROOT / "dataset" / "dataset.py"
_dst = _ds.read_text()
if "cam4feat\")):  # AERIAL_SEQUENTIAL_PATCH" not in _dst:
    _dst = _dst.replace(
        'self.type_src = conf.get_string("type_sources", default="mvs")',
        'self.type_src = conf.get_string("type_sources", default="sequential").strip().strip(\'"\')\n'
        '        print(f"  dataset.type_sources = {self.type_src!r}")  # AERIAL_SEQUENTIAL_PATCH',
    )
    _dst = _dst.replace(
        'if self.type_src == "mvs":\n            self.pair = load_pair',
        'if self.type_src == "mvs" and os.path.isdir(os.path.join(self.data_dir, "cam4feat")):  # AERIAL_SEQUENTIAL_PATCH\n            self.pair = load_pair',
    )
    old_feat = """        cam = scale_camera(depth_cam, self.feat_img_scale)
        src_cams = torch.stack(
            [
                scale_camera(
                    torch.stack(
                        [self.pose_network(i), self.intrinsic_network()], dim=0
                    ),
                    self.feat_img_scale,
                )
                for i in src_idxs
            ]
        )
        feat_input = {
            \"depth_cams\": depth_cam,
            \"size\": self.size,
            \"center\": self.center,
            \"feat\": self.feats[img_idx],
            \"feat_src\": self.feats[src_idxs],
            \"cam\": cam,
            \"src_cams\": src_cams,
            \"rays_v_norm\": rays_v_norm,
            \"H\": self.H,
            \"W\": self.W,
        }"""
    new_feat = """        if self.type_src == \"mvs\" and hasattr(self, \"feats\"):  # AERIAL_SEQUENTIAL_PATCH
            cam = scale_camera(depth_cam, self.feat_img_scale)
            src_cams = torch.stack(
                [
                    scale_camera(
                        torch.stack(
                            [self.pose_network(i), self.intrinsic_network()], dim=0
                        ),
                        self.feat_img_scale,
                    )
                    for i in src_idxs
                ]
            )
            feat_input = {
                \"depth_cams\": depth_cam,
                \"size\": self.size,
                \"center\": self.center,
                \"feat\": self.feats[img_idx],
                \"feat_src\": self.feats[src_idxs],
                \"cam\": cam,
                \"src_cams\": src_cams,
                \"rays_v_norm\": rays_v_norm,
                \"H\": self.H,
                \"W\": self.W,
            }
        else:
            feat_input = None  # AERIAL_SEQUENTIAL_PATCH"""
    if old_feat in _dst:
        _dst = _dst.replace(old_feat, new_feat)
    else:
        print("WARN: feat_input block not found — check dataset.py")
    _dst = _dst.replace(
        "src_idxs = [img_idx.item() - 1, (img_idx.item() + 1) % self.n_images]",
        "ii = int(img_idx.item() if hasattr(img_idx, \"item\") else img_idx)\n            src_idxs = [(ii - 1) % self.n_images, (ii + 1) % self.n_images]",
    )
    _ds.write_text(_dst)
    print("patched dataset.py")

_opt = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
_ot = _opt.read_text()
if "AERIAL_FEAT_NONE_PATCH" not in _ot:
    _ot = _ot.replace(
        """        for attr in [
            "depth_cams",
            "size",
            "center",
            "feat",
            "feat_src",
            "cam",
            "src_cams",
            "rays_v_norm",
        ]:
            feat_input[attr] = feat_input[attr].cuda()
        for attr in ["H", "W"]:
            feat_input[attr] = feat_input[attr]""",
        """        if feat_input is not None:  # AERIAL_FEAT_NONE_PATCH
            for attr in [
                "depth_cams",
                "size",
                "center",
                "feat",
                "feat_src",
                "cam",
                "src_cams",
                "rays_v_norm",
            ]:
                feat_input[attr] = feat_input[attr].cuda()
            for attr in ["H", "W"]:
                feat_input[attr] = feat_input[attr]""",
    )
    _opt.write_text(_ot)
    print("patched ba_neus_optimizer.py (feat_input optional)")




# --- CUDA_DEVICE_PATCH: camera nets live on CUDA; pixel tensors default to CPU in torch 2.x ---
_ds = BANEUS_ROOT / "dataset" / "dataset.py"
_dst = _ds.read_text()
if "CUDA_DEVICE_PATCH" not in _dst:
    _dst = _dst.replace(
        """        p = torch.stack(
            [pixels_x, pixels_y, torch.ones_like(pixels_y)], dim=-1
        ).float()  # batch_size, 3
        p = torch.matmul(
            self.intrinsic_network(inverse=True)[:3, :3], p[:, :, None]
        ).squeeze()  # batch_size, 3""",
        """        Kinv = self.intrinsic_network(inverse=True)[:3, :3]  # CUDA_DEVICE_PATCH
        p = torch.stack(
            [pixels_x.float(), pixels_y.float(), torch.ones_like(pixels_y, dtype=torch.float32)], dim=-1
        ).to(Kinv.device)
        p = torch.matmul(Kinv, p[:, :, None]).squeeze()""",
    )
    _dst = _dst.replace(
        """        rays_v = torch.matmul(
            self.pose_network(img_idx)[:3, :3], rays_v[:, :, None]
        ).squeeze()  # batch_size, 3
        rays_o = self.pose_network(img_idx)[:3, 3].expand(rays_v.shape)  # batch_size, 3""",
        """        R = self.pose_network(img_idx)[:3, :3]  # CUDA_DEVICE_PATCH
        rays_v = torch.matmul(R, rays_v.to(R.device)[:, :, None]).squeeze()
        rays_o = self.pose_network(img_idx)[:3, 3].expand(rays_v.shape)  # batch_size, 3""",
    )
    _dst = _dst.replace(
        """        p = torch.matmul(
            self.intrinsic_network(inverse=True)[:3, :3], p[:, :, :, None]
        ).squeeze()  # W, H, 3
        rays_v = p / torch.linalg.norm(p, ord=2, dim=-1, keepdim=True)  # W, H, 3
        rays_v = torch.matmul(
            self.pose_network(img_idx)[:3, :3], rays_v[:, :, :, None]
        ).squeeze()  # W, H, 3
        rays_o = self.pose_network(img_idx)[:3, 3].expand(rays_v.shape)  # W, H, 3""",
        """        Kinv = self.intrinsic_network(inverse=True)[:3, :3]  # CUDA_DEVICE_PATCH
        p = p.to(Kinv.device)
        p = torch.matmul(Kinv, p[:, :, :, None]).squeeze()
        rays_v = p / torch.linalg.norm(p, ord=2, dim=-1, keepdim=True)
        R = self.pose_network(img_idx)[:3, :3]
        rays_v = torch.matmul(R, rays_v[:, :, :, None]).squeeze()
        rays_o = self.pose_network(img_idx)[:3, 3].expand(rays_v.shape)  # W, H, 3""",
    )
    _dst = _dst.replace(
        """        p = torch.matmul(
            self.intrinsic_network(inverse=True)[:3, :3], p[:, :, :, None]
        ).squeeze()  # W, H, 3
        rays_v = p / torch.linalg.norm(p, ord=2, dim=-1, keepdim=True)  # W, H, 3
        trans = (
            self.pose_network(idx_0)[:3, 3] * (1.0 - ratio)
            + self.pose_network(idx_1)[:3, 3] * ratio
        )""",
        """        Kinv = self.intrinsic_network(inverse=True)[:3, :3]  # CUDA_DEVICE_PATCH
        p = p.to(Kinv.device)
        p = torch.matmul(Kinv, p[:, :, :, None]).squeeze()
        rays_v = p / torch.linalg.norm(p, ord=2, dim=-1, keepdim=True)
        trans = (
            self.pose_network(idx_0)[:3, 3] * (1.0 - ratio)
            + self.pose_network(idx_1)[:3, 3] * ratio
        )""",
    )
    _ds.write_text(_dst)
    print("patched dataset.py (CUDA_DEVICE_PATCH)")


sys.path.insert(0, str(BANEUS_ROOT))
print("BA-NeuS ready at", BANEUS_ROOT)

# --- dataset.py: zero noise when noise_magnitude==0; freeze pose MLP when learn_pose=False ---
_ds = BANEUS_ROOT / "dataset" / "dataset.py"
if _ds.exists():
    _dt = _ds.read_text()
    _old_noise = """        rot_noise = torch.randn(3, device="cpu") * self.noise_magnitude
        trans_noise = torch.randn(3, device="cpu") * self.noise_magnitude
        c2w_noise = make_c2w(rot_noise, trans_noise)
        self.pose_all.append(c2w_noise @ torch.from_numpy(pose).float())"""
    _new_noise = """        if self.noise_magnitude > 0:  # AERIAL_FREEZE_POSE_PATCH
            rot_noise = torch.randn(3, device="cpu") * self.noise_magnitude
            trans_noise = torch.randn(3, device="cpu") * self.noise_magnitude
            c2w_noise = make_c2w(rot_noise, trans_noise)
            pose_init = c2w_noise @ torch.from_numpy(pose).float()
        else:
            pose_init = torch.from_numpy(pose).float()
        self.pose_all.append(pose_init)"""
    if "AERIAL_FREEZE_POSE_PATCH" not in _dt and _old_noise in _dt:
        _dt = _dt.replace(_old_noise, _new_noise)

    # Repair earlier bad patch that orphaned `else:` → overwrote pose_network with init_c2w=None when learn_pose=True
    _broken_pose = """        if self.init_pose:
            self.pose_network = LearnPose(
                num_cams=len(self.pose_all),
                init_c2w=self.pose_all,
                trans_offset=self.trans_offset,
            ).to(self.device)
        if self.init_pose:  # CATHEDRAL_FORCE_POSE_ZEROINIT
            with torch.no_grad():
                torch.nn.init.zeros_(self.pose_network.lin3.weight)
                torch.nn.init.zeros_(self.pose_network.lin3.bias)
        if not self.learn_pose:  # AERIAL_FREEZE_POSE_PATCH
            for _p in self.pose_network.parameters():
                _p.requires_grad = False
        else:
            self.pose_network = LearnPose(
                num_cams=len(self.pose_all),
                init_c2w=None,
                trans_offset=self.trans_offset,
            ).to(self.device)"""
    _broken_pose_force = _broken_pose.replace(
        "# CATHEDRAL_POSE_POSTINIT / CATHEDRAL_FORCE_POSE_ZEROINIT",
        "# CATHEDRAL_FORCE_POSE_ZEROINIT",
    )
    _fixed_pose = """        if self.init_pose:
            self.pose_network = LearnPose(
                num_cams=len(self.pose_all),
                init_c2w=self.pose_all,
                trans_offset=self.trans_offset,
            ).to(self.device)
        else:
            self.pose_network = LearnPose(
                num_cams=len(self.pose_all),
                init_c2w=None,
                trans_offset=self.trans_offset,
            ).to(self.device)
        if self.init_pose:  # CATHEDRAL_POSE_POSTINIT / CATHEDRAL_FORCE_POSE_ZEROINIT
            with torch.no_grad():
                torch.nn.init.zeros_(self.pose_network.lin3.weight)
                torch.nn.init.zeros_(self.pose_network.lin3.bias)
        if not self.learn_pose:  # AERIAL_FREEZE_POSE_PATCH
            for _p in self.pose_network.parameters():
                _p.requires_grad = False"""
    _pose_patch_msg = None
    if _broken_pose in _dt or _broken_pose_force in _dt:
        if _broken_pose in _dt:
            _dt = _dt.replace(_broken_pose, _fixed_pose)
        else:
            _dt = _dt.replace(_broken_pose_force, _fixed_pose)
        _pose_patch_msg = "repaired dataset.py (CATHEDRAL_BROKEN_POSE_ELSE — learn_pose=True was clobbering init_c2w)"

    # Older bad patch variant (freeze inserted between if/else without zero-init)
    _broken_pose2 = """        if self.init_pose:
            self.pose_network = LearnPose(
                num_cams=len(self.pose_all),
                init_c2w=self.pose_all,
                trans_offset=self.trans_offset,
            ).to(self.device)
        if not self.learn_pose:  # AERIAL_FREEZE_POSE_PATCH
            for _p in self.pose_network.parameters():
                _p.requires_grad = False
        else:
            self.pose_network = LearnPose(
                num_cams=len(self.pose_all),
                init_c2w=None,
                trans_offset=self.trans_offset,
            ).to(self.device)"""
    if _broken_pose2 in _dt:
        _dt = _dt.replace(_broken_pose2, _fixed_pose)
        _pose_patch_msg = "repaired dataset.py (CATHEDRAL_BROKEN_POSE_ELSE v2)"

    # Insert zero-init + freeze AFTER the full LearnPose if/else (never between if and else)
    _post_anchor = """                init_c2w=None,
                trans_offset=self.trans_offset,
            ).to(self.device)
        if self.init_intrinsic:"""
    _post_tail = """                init_c2w=None,
                trans_offset=self.trans_offset,
            ).to(self.device)
        if self.init_pose:  # CATHEDRAL_POSE_POSTINIT / CATHEDRAL_FORCE_POSE_ZEROINIT
            with torch.no_grad():
                torch.nn.init.zeros_(self.pose_network.lin3.weight)
                torch.nn.init.zeros_(self.pose_network.lin3.bias)
        if not self.learn_pose:  # AERIAL_FREEZE_POSE_PATCH
            for _p in self.pose_network.parameters():
                _p.requires_grad = False
        if self.init_intrinsic:"""
    if "CATHEDRAL_POSE_POSTINIT" not in _dt and _post_anchor in _dt:
        _dt = _dt.replace(_post_anchor, _post_tail, 1)
        _pose_patch_msg = _pose_patch_msg or "patched dataset.py (CATHEDRAL_POSE_POSTINIT after LearnPose if/else)"

    if "AERIAL_FREEZE_POSE_PATCH" in _dt or "CATHEDRAL_POSE_POSTINIT" in _dt or "CATHEDRAL_FORCE_POSE_ZEROINIT" in _dt:
        _ds.write_text(_dt)
        if _pose_patch_msg:
            print(_pose_patch_msg)

# --- ba_neus_optimizer.py: skip pose_optimizer.step when learn_pose=False ---
_opt = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
if _opt.exists():
    _ot = _opt.read_text()
    _old_step = """        self.optimizer.step()
        self.intrinsic_optimizer.step()
        self.pose_optimizer.step()
        self.distortion_optimizer.step()"""
    _new_step = """        self.optimizer.step()
        self.intrinsic_optimizer.step()
        if self.dataset.learn_pose:  # AERIAL_FREEZE_POSE_PATCH
            self.pose_optimizer.step()
        self.distortion_optimizer.step()"""
    if "AERIAL_FREEZE_POSE_PATCH" not in _ot and _old_step in _ot:
        _opt.write_text(_ot.replace(_old_step, _new_step))
        print("patched ba_neus_optimizer.py (skip pose step when frozen)")

# --- camera.py: when learn_pose=False, use init_c2w exactly (no MLP delta) ---
_cam = BANEUS_ROOT / "models" / "camera.py"
if _cam.exists():
    _ct = _cam.read_text()
    _old_fwd = """    def forward(self, cam_id):
        cam_id = self.all_points[cam_id]
        cam_id = cam_id.unsqueeze(0)"""
    _new_fwd = """    def forward(self, cam_id):
        # AERIAL_IDENTITY_POSE_PATCH: frozen pose = exported init, no random MLP delta
        if self.init_c2w is not None and not self.lin1.weight.requires_grad:
            view_idx = int(self.all_points[cam_id].item())
            return self.init_c2w[view_idx]
        cam_id = self.all_points[cam_id]
        cam_id = cam_id.unsqueeze(0)"""
    if "AERIAL_IDENTITY_POSE_PATCH" not in _ct and _old_fwd in _ct:
        _cam.write_text(_ct.replace(_old_fwd, _new_fwd))
        print("patched camera.py (identity pose when frozen)")
    elif "AERIAL_IDENTITY_POSE_PATCH" in _ct:
        print("camera.py already has identity pose patch")


# --- camera.py: zero-init pose MLP so learn_pose starts exactly at init_c2w (geo JSON reference) ---
_cam_z = BANEUS_ROOT / "models" / "camera.py"
if _cam_z.exists():
    _zt = _cam_z.read_text()
    _z_old = "        self.lin3 = nn.Linear(64, 6)"
    _z_new = """        self.lin3 = nn.Linear(64, 6)
        if init_c2w is not None:  # CATHEDRAL_POSE_ZEROINIT
            nn.init.zeros_(self.lin3.weight)
            nn.init.zeros_(self.lin3.bias)"""
    if "CATHEDRAL_POSE_ZEROINIT" not in _zt and _z_old in _zt:
        _cam_z.write_text(_zt.replace(_z_old, _z_new))
        print("patched camera.py (zero-init pose MLP)")


## 11.2 — Link preprocessed NeuS dataset

Run when `USE_PREPROCESSED=True` (after §11.1). Copies `ultrra_pp/public_data/` into BA-NeuS.


In [ ]:
# 11.2 — Deploy ultrra_pp into BA-NeuS public_data/
if USE_PREPROCESSED:
    _dst = deploy_preprocessed_to_baneus()
    _npz = _dst / "cameras_sphere.npz"
    assert _npz.exists(), f"Deploy failed: {_npz}"
    import numpy as np
    _n = len([k for k in np.load(_npz) if k.startswith("world_mat_")])
    print(f"✓ BA-NeuS dataset ready: {_dst} ({_n} cameras)")
else:
    print("USE_PREPROCESSED=False — run §12.1 to export NeuS dataset instead")



> **Skip when `USE_PREPROCESSED=True`**

## 12 — Export NeuS dataset

Builds `public_data/<CASE_NAME>/` from va_split aerial views + **DA3** depth/cameras (geo-aligned).
No BlendedMVS `cams/` or GT depth — DA3 supplies depth supervision.


In [ ]:
if USE_PREPROCESSED:
    print("Skipping §12 — using preprocessed public_data from", PREPROCESSED_DIR)
    print("  ->", BANEUS_ROOT / "public_data" / CASE_NAME)
    assert (BANEUS_ROOT / "public_data" / CASE_NAME / "cameras_sphere.npz").exists(), \
        "Run §11.2 to deploy preprocessed data into BA-NeuS"
else:
    # 12.1 — NeuS dataset in the Stage-1 / DA3 WORLD frame (uniform resolution)
    import numpy as np, cv2, torch
    from pathlib import Path
    from PIL import Image
    from pytorch3d.transforms import quaternion_to_matrix, Transform3d

    def compose_transform(scale, rotation, translation, device="cpu"):
        """PyTorch3D Translate needs (N,3); keep pose + points on the same device."""
        if translation is None:
            raise ValueError("compose_transform: translation is None")
        dev = torch.device(device)
        scale = torch.as_tensor(scale, dtype=torch.float32, device=dev)
        rotation = torch.as_tensor(rotation, dtype=torch.float32, device=dev)
        translation = torch.as_tensor(translation, dtype=torch.float32, device=dev)
        if translation.dim() == 1:
            translation = translation.unsqueeze(0)
        if scale.dim() == 1:
            scale = scale.unsqueeze(0)
        if rotation.dim() == 2:
            rotation = rotation.unsqueeze(0)
        tfm = Transform3d(device=dev, dtype=torch.float32)
        return tfm.scale(scale).rotate(rotation).translate(translation)


    def _as_trimesh(obj):
        """GLB/OBJ may load as trimesh.Scene — collapse to a single Trimesh."""
        import trimesh
        if isinstance(obj, trimesh.Trimesh):
            return obj
        if isinstance(obj, trimesh.Scene):
            geoms = [g for g in obj.geometry.values() if isinstance(g, trimesh.Trimesh)]
            if not geoms:
                raise ValueError("Scene has no mesh geometry")
            return trimesh.util.concatenate(geoms) if len(geoms) > 1 else geoms[0]
        raise TypeError(f"Expected Trimesh or Scene, got {type(obj)}")


    def export_cam4feat_pair(view_names, scene_dir, case_dir, num_src=10):
        """Remap BlendedMVS cams/pair.txt -> BA-NeuS public_data/.../cam4feat/pair.txt."""
        import os
        cam_dir = Path(scene_dir) / "cams"
        pair_p = cam_dir / "pair.txt"
        if not pair_p.is_file():
            print("WARN: no cams/pair.txt — cam4feat export skipped; will fall back to sequential sources")
            return False
        cams_all = sorted(cam_dir.glob("*_cam.txt")) or sorted(cam_dir.glob("*"))
        orig_stem = {}
        for i, c in enumerate(cams_all):
            b = c.stem
            orig_stem[b[:-4] if b.endswith("_cam") else b] = i
        lines = pair_p.read_text().splitlines()
        n_all = int(lines[0])
        raw = {}
        for v in range(n_all):
            ref = int(lines[1 + 2 * v])
            toks = lines[2 + 2 * v].split()
            k = int(toks[0])
            raw[ref] = [(int(toks[1 + 2 * j]), float(toks[2 + 2 * j])) for j in range(k)]
        stem_to_export = {str(nm): i for i, nm in enumerate(view_names)}
        orig_to_export = {oi: stem_to_export[s] for s, oi in orig_stem.items() if s in stem_to_export}
        n = len(view_names)
        out_lines = [str(n)]
        for i, nm in enumerate(view_names):
            sid = str(i)
            oi = orig_stem.get(str(nm))
            pairs = []
            if oi is not None and oi in raw:
                for s, score in raw[oi]:
                    if s in orig_to_export:
                        pairs.append((str(orig_to_export[s]), score))
            if len(pairs) < 2:
                for nb_i in [(i - 1) % n, (i + 1) % n]:
                    sid_nb = str(nb_i)
                    if sid_nb not in [p[0] for p in pairs]:
                        pairs.append((sid_nb, 1.0))
            pairs = pairs[:num_src]
            out_lines.append(sid)
            out_lines.append(f"{len(pairs)} " + " ".join(f"{a} {b:.6g}" for a, b in pairs))
        out_dir = Path(case_dir) / "cam4feat"
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "pair.txt").write_text("\n".join(out_lines) + "\n")
        print(f"Wrote cam4feat/pair.txt ({n} views) -> {out_dir}")
        return True

    def _to_c2w(ext):
        """DA3 extrinsics are world->camera (see §2.6) — invert to camera-to-world."""
        ext = np.asarray(ext, np.float64)
        w2c = np.eye(4, dtype=np.float64)
        w2c[:3, :] = ext[:3, :4]
        return np.linalg.inv(w2c)

    def build_neus_cameras(c2ws, Ks, center, radius):
        """Standard NeuS/IDR cameras so Dataset decodes NORMALIZED poses.

        NeuS raycasts inside a unit sphere at the ORIGIN. Its Dataset computes
            pose = load_K_Rt_from_P(world_mat @ scale_mat)
        and uses pose[:3,3] as the camera center + near_far_from_sphere(unit sphere).
        So the decode MUST land in normalized coords, i.e. camera center ~ O(1).

        Convention (IDR/NeuS gen_cameras):
            scale_mat : norm -> world   (diag = radius, translation = +center)
            world_mat : world -> image  (K @ w2c),  NO scale baked in
          => world_mat @ scale_mat = K @ (norm -> cam)  -> decode gives normalized c2w,
             camera center = (C_world - center) / radius.

        (The previous version baked inv(scale_mat) into world_mat AND stored the
         inverse scale_mat, so the two cancelled and the decode stayed in WORLD frame
         with camera centers ~100 units out — rays then missed the object.)
        """
        center = np.asarray(center, np.float64)
        radius = float(radius)
        scale_mat = np.eye(4, dtype=np.float32)
        scale_mat[0, 0] = scale_mat[1, 1] = scale_mat[2, 2] = radius
        scale_mat[:3, 3] = center.astype(np.float32)
        mats = {}
        for i, (c2w, K) in enumerate(zip(c2ws, Ks)):
            w2c = np.linalg.inv(np.asarray(c2w, np.float64))
            P = (np.asarray(K, np.float64) @ w2c[:3, :4]).astype(np.float32)
            world = np.eye(4, dtype=np.float32); world[:3, :] = P
            mats[f"world_mat_{i}"] = world
            mats[f"scale_mat_{i}"] = scale_mat.copy()
        return mats, scale_mat, 1.0 / radius

    def _object_mask_native(name, Wn, Hn):
        """Load the SAM building mask at native depth resolution (bool HxW)."""
        mpath = VIEW_DIR / MASK_NAME / f"{name}.png"
        m = np.array(Image.open(mpath).convert("RGBA"))
        alpha = m[:, :, 3] if m.shape[-1] == 4 else (m if m.ndim == 2 else m[:, :, 0])
        alpha = cv2.resize(alpha, (Wn, Hn), interpolation=cv2.INTER_NEAREST)
        return alpha > 127

    c2ws, Ks, depth_maps = [], [], []

    if GT_NATIVE:
        # ================= GT-NATIVE: one consistent BlendedMVS world frame =================
        # Cameras (cams/), depth (rendered_depth_maps/*.pfm) and normalization all in GT world.
        # No DA3 / Stage-1 needed. center/radius come from back-projected GT depth (object mask).
        _export_src = "blendedmvs_gt"
        print("GT-native export: not used on ULTRRA path (GT_NATIVE=False)")

        # export resolution = native BlendedMVS 768x576 (already multiples of 16)
        TRAIN_W, TRAIN_H = 768, 576
        print(f"Export resolution: {TRAIN_W}x{TRAIN_H}")

        obj_pts = []          # object point cloud in GT world (for center/radius)
        euclid_maps = []      # per-view Euclidean ray distance at train res (world units)
        for name in view_names:
            g = GEO_REF[name]
            assert g["depth_path"] is not None, f"No rendered_depth_maps/{name}.pfm — need GT depth for GT-native"
            c2w = np.asarray(g["c2w"], np.float64)
            Kn = np.asarray(g["K_native"], np.float64)          # intrinsics for native 768x576

            Znat = read_pfm(g["depth_path"]).astype(np.float64)  # (Hn,Wn) perspective Z-depth
            Hn, Wn = Znat.shape
            if Kn is None or (Hn, Wn) != (576, 768):
                # rescale intrinsics to the actual pfm size if it differs from 768x576
                Kn = Kn.copy(); Kn[0] *= Wn / 768.0; Kn[1] *= Hn / 576.0

            uu, vv = np.meshgrid(np.arange(Wn), np.arange(Hn))
            ray = np.stack([(uu - Kn[0, 2]) / Kn[0, 0],
                            (vv - Kn[1, 2]) / Kn[1, 1],
                            np.ones_like(uu, np.float64)], axis=-1)   # cam dirs (z=1)
            euclid_scale = np.linalg.norm(ray, axis=-1)               # Z-depth -> Euclidean factor

            obj = _object_mask_native(name, Wn, Hn)
            valid = (Znat > 0) & obj

            # object points -> GT world (subsampled for the AABB estimate)
            Xc = Znat[..., None] * ray                                # camera-space points
            Xw = Xc @ c2w[:3, :3].T + c2w[:3, 3]
            pv = Xw[valid][::4]
            if pv.size:
                obj_pts.append(pv)

            # Euclidean ray-distance depth (NeuS renders Euclidean depth); invalid -> 0
            euclid = (Znat * euclid_scale)
            euclid[~valid] = 0.0
            if (Hn, Wn) != (TRAIN_H, TRAIN_W):
                euclid = cv2.resize(euclid.astype(np.float32), (TRAIN_W, TRAIN_H),
                                    interpolation=cv2.INTER_NEAREST)
            euclid_maps.append(euclid.astype(np.float32))

            K = Kn.copy(); K[0] *= TRAIN_W / Wn; K[1] *= TRAIN_H / Hn
            c2ws.append(c2w); Ks.append(K)

        obj_pts = np.concatenate(obj_pts, axis=0)
        lo = np.percentile(obj_pts, 1.0, axis=0)
        hi = np.percentile(obj_pts, 99.0, axis=0)
        center = 0.5 * (lo + hi)
        _d = np.linalg.norm(obj_pts - center, axis=1)
        radius = float(np.percentile(_d, 99.0) * 1.1 + 1e-6)
        depth_maps = euclid_maps
        print(f"GT-world object AABB from {len(obj_pts)} pts: center={center.round(3)} radius={radius:.4f}")
        frame = {"scale": np.float32(1.0), "rotation": np.array([1, 0, 0, 0], np.float32),
                 "translation": np.zeros(3, np.float32)}   # identity: canonical == world
    else:
        # ================= DA3 export (geo-aligned when §8 ran ALIGN_DA3_TO_JSON_GEO) =======
        _HAS_STAGE1 = (OUTPUT_DIR / "stage1" / "stage1_frame.npz").exists() and BANEUS_USE_PRIOR
        world_align_s = 1.0
        world_align_R = np.eye(3, dtype=np.float32)
        world_align_t = np.zeros(3, dtype=np.float32)
        if _HAS_STAGE1:
            frame_npz = np.load(OUTPUT_DIR / "stage1" / "stage1_frame.npz")
            frame = {k: frame_npz[k] for k in frame_npz.files}
            s1_center = np.asarray(frame["center"], np.float64)
            s1_radius = float(frame["radius"])
            print(f"Stage-1 world AABB: center={s1_center.round(3)} radius={s1_radius:.4f}")
        else:
            frame = {"scale": np.float32(1.0), "rotation": np.array([1, 0, 0, 0], np.float32),
                     "translation": np.zeros(3, np.float32)}
            print("No Stage-1 frame — DA3-only export (raw BA-NeuS path)")

        da3 = np.load(DA3_NPZ)
        depths_all = da3["depth"]
        ext_all    = da3["extrinsics"]
        K_all      = da3["intrinsics"]
        stems      = [Path(str(p)).stem for p in da3["image_files"]]

        # Build the DA3 masked depth cloud to get a camera-consistent center/radius.
        _step_dc = max(1, len(view_names) // 24)
        _dc_pts = []
        for _nm in view_names[::_step_dc]:
            _j = stems.index(_nm)
            _Zdc = depths_all[_j].astype(np.float64)
            _Hdc, _Wdc = _Zdc.shape
            _Kdc = K_all[_j].astype(np.float64)
            _uu, _vv = np.meshgrid(np.arange(_Wdc), np.arange(_Hdc))
            _rdc = np.stack([(_uu - _Kdc[0, 2]) / _Kdc[0, 0],
                             (_vv - _Kdc[1, 2]) / _Kdc[1, 1],
                             np.ones_like(_uu, np.float64)], axis=-1)
            _mdc = (_Zdc > 0) & _object_mask_native(_nm, _Wdc, _Hdc)
            if not _mdc.any():
                continue
            _c2w_dc = _to_c2w(ext_all[_j])
            _Xdc = (_rdc * _Zdc[..., None])[_mdc][::16]
            _dc_pts.append(_Xdc @ _c2w_dc[:3, :3].T + _c2w_dc[:3, 3])
        _dc_pts = np.concatenate(_dc_pts, axis=0)
        da3_center = 0.5 * (np.percentile(_dc_pts, 1, axis=0) + np.percentile(_dc_pts, 99, axis=0))
        da3_radius = float(np.percentile(np.linalg.norm(_dc_pts - da3_center, axis=1), 99))
        print(f"DA3 depth cloud: center={da3_center.round(3)} radius={da3_radius:.4f}")

        if _HAS_STAGE1:
            _ratio = s1_radius / max(da3_radius, 1e-9)
            _offset = float(np.linalg.norm(s1_center - da3_center)) / max(da3_radius, 1e-9)
            print(f"Stage-1 vs DA3: radius ratio={_ratio:.3f} (want ~1) | center offset={_offset:.3f} (want <0.5)")

            # DA3 center comes from masked depth points;
            # Stage-1 AABB includes ground/artifacts and is often biased.
            # Always use DA3 center; use the tighter of the two radii so the
            # unit sphere hugs the actual object, not SDF artefacts.
            if 0.3 < _ratio < 3.0 and _offset < 1.0:
                center = da3_center
                radius = min(s1_radius, da3_radius * 1.1)
                print(f"Stage-1 & DA3 roughly agree. Using DA3 center + tighter radius:")
                print(f"  center={center.round(3)}  radius={radius:.4f}")
            else:
                center, radius = da3_center, da3_radius * 1.1
                print(f"Stage-1 AABB disagrees with DA3 — using DA3 center/radius.")
                print(f"  center={center.round(3)}  radius={radius:.4f}")
        else:
            center, radius = da3_center, da3_radius * 1.1
            print(f"NeuS normalization from DA3 depth cloud: center={center.round(3)} radius={radius:.4f}")

        # Optional Sim3: nudge mesh world verts onto the masked DA3 cloud (Stage-1 path only).
        if _HAS_STAGE1:
            import trimesh as _trimesh_s1
            if "M_canonical_to_da3_world" in frame:
                _M0 = np.asarray(frame["M_canonical_to_da3_world"], np.float64)
            else:
                _fc2w = np.asarray(frame["ref_c2w"], np.float64) if "ref_c2w" in frame else None
                _M0 = stage1_canonical_to_da3_world_matrix(
                    frame["scale"], frame["rotation"], frame["translation"],
                    c2w_opencv=_fc2w, device="cpu")
                frame = dict(frame)
                frame["M_canonical_to_da3_world"] = _M0.astype(np.float32)
            _s1 = _trimesh_s1.load(OUTPUT_DIR / "stage1" / "stage1_coarse_mesh.glb", process=False)
            if isinstance(_s1, _trimesh_s1.Scene):
                _s1 = _trimesh_s1.util.concatenate([g for g in _s1.geometry.values() if isinstance(g, _trimesh_s1.Trimesh)])
            _vw = stage1_canonical_to_da3_world(np.asarray(_s1.vertices, np.float64), _M0)
            _mp = _vw[::max(1, len(_vw) // 4000)]
            _cp = _dc_pts[::max(1, len(_dc_pts) // 4000)]
            world_align_s, world_align_R, world_align_t = sim3_align_points(_cp, _mp)
            _s3_rot_off = float(np.abs(world_align_R - np.eye(3)).max())
            print(f"Sim3 mesh->DA3 cloud: scale={world_align_s:.4f} | trans={np.linalg.norm(world_align_t):.3f} | rot off-identity={_s3_rot_off:.4f}")
            if _s3_rot_off > 0.3:
                print(f"  WARNING: Sim3 found significant rotation:\n{world_align_R.round(3)}")
        else:
            print("Skipping Sim3 mesh->DA3 (raw DA3-only export; no Stage-1 mesh)")

        _ref = depths_all[stems.index(view_names[0])]
        TRAIN_H, TRAIN_W = int(_ref.shape[0]), int(_ref.shape[1])
        TRAIN_H = max(16, (TRAIN_H // 16) * 16); TRAIN_W = max(16, (TRAIN_W // 16) * 16)
        print(f"Export resolution: {TRAIN_W}x{TRAIN_H}")

        _export_src = "da3"
        print(f"NeuS camera source: {_export_src}")
        for name in view_names:
            j = stems.index(name)
            Znat = depths_all[j].astype(np.float64)        # DA3 emits perspective Z-depth
            _dh, _dw = Znat.shape
            Kd = K_all[j].astype(np.float64)
            uu, vv = np.meshgrid(np.arange(_dw), np.arange(_dh))
            ray = np.stack([(uu - Kd[0, 2]) / Kd[0, 0],
                            (vv - Kd[1, 2]) / Kd[1, 1],
                            np.ones_like(uu, np.float64)], axis=-1)
            # BA-NeuS renders depth as sum(w * z) along UNIT rays -> Euclidean ray distance
            euclid = Znat * np.linalg.norm(ray, axis=-1)
            valid = (Znat > 0) & _object_mask_native(name, _dw, _dh)
            euclid[~valid] = 0.0
            if (_dh, _dw) != (TRAIN_H, TRAIN_W):
                euclid = cv2.resize(euclid.astype(np.float32), (TRAIN_W, TRAIN_H),
                                    interpolation=cv2.INTER_NEAREST)
            depth_maps.append(euclid.astype(np.float32))
            c2w = _to_c2w(ext_all[j])
            K = Kd.copy()
            sx, sy = TRAIN_W / _dw, TRAIN_H / _dh
            K[0] *= sx; K[1] *= sy
            c2ws.append(c2w)
            Ks.append(K.astype(np.float64))

        # BA-NeuS raycasts every view through ONE intrinsic (LearnFocal, initialised from
        # view 0), so per-view DA3 focals would silently mis-aim all but the first camera.
        _K_stack = np.stack(Ks)
        _K_med = np.median(_K_stack, axis=0)
        _fx_dev = float(np.abs(_K_stack[:, 0, 0] / _K_med[0, 0] - 1.0).max())
        _pp_dev = float(np.abs(_K_stack[:, :2, 2] - _K_med[:2, 2]).max())
        print(f"DA3 intrinsics spread across views: fx max dev={100 * _fx_dev:.2f}% | "
              f"principal point max dev={_pp_dev:.1f}px -> exporting shared K")
        Ks = [_K_med.copy() for _ in Ks]

    # ---- Enforce cameras OUTSIDE the NeuS unit sphere ----
    # NeuS near_far_from_sphere requires every camera center to be at normalized
    # distance >1 from the origin; otherwise near=0 and training cannot converge.
    _cam_pos_w = np.stack([np.asarray(c, np.float64)[:3, 3] for c in c2ws])
    _cam_dist_w = np.linalg.norm(_cam_pos_w - np.asarray(center, np.float64), axis=1)
    _min_cam_dist = float(_cam_dist_w.min())
    _SPHERE_MARGIN = 1.15          # cameras at >=1.15× the sphere radius
    _max_safe_radius = _min_cam_dist / _SPHERE_MARGIN
    if radius > _max_safe_radius:
        _old_radius = radius
        radius = _max_safe_radius
        print(f"Clamped radius: {_old_radius:.4f} -> {radius:.4f}  "
              f"(min cam dist {_min_cam_dist:.3f}, margin {_SPHERE_MARGIN}x)")
        print(f"  Old norm cam min = {_min_cam_dist/_old_radius:.3f}  "
              f"-> new = {_min_cam_dist/radius:.3f} (want >1)")

    cam_npz, scale_mat, norm_s = build_neus_cameras(c2ws, Ks, center, radius)

    case_dir = BANEUS_ROOT / "public_data" / CASE_NAME
    for sub in ("image", "mask", "depth"):
        (case_dir / sub).mkdir(parents=True, exist_ok=True)

    shapes = []
    for i, name in enumerate(view_names):
        img_p = IMAGES_DIR / f"{name}.png"
        if not img_p.exists():
            img_p = next(IMAGES_DIR.glob(f"{name}.*"))
        im = Image.open(img_p).convert("RGB").resize((TRAIN_W, TRAIN_H), Image.LANCZOS)
        im.save(case_dir / "image" / f"{i:03d}.png")

        mpath = VIEW_DIR / MASK_NAME / f"{name}.png"
        m = Image.open(mpath).convert("RGBA")
        arr = np.array(m)
        alpha = arr[:,:,3] if arr.shape[-1] == 4 else arr if arr.ndim == 2 else arr[:,:,0]
        mask = (alpha > 127).astype(np.uint8) * 255
        mask = cv2.resize(mask, (TRAIN_W, TRAIN_H), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(str(case_dir / "mask" / f"{i:03d}.png"), np.stack([mask]*3, -1))

        np.save(case_dir / "depth" / f"{i:03d}.npy", (depth_maps[i] * norm_s).astype(np.float32))
        shapes.append((TRAIN_H, TRAIN_W))

    np.savez(case_dir / "cameras_sphere.npz", **cam_npz)
    np.savez(
        OUTPUT_DIR / "baneus_scene_meta.npz",
        center=center, radius=radius, scale_mat=scale_mat, norm_s=norm_s,
        scale=frame["scale"], rotation=frame["rotation"], translation=frame["translation"],
        M_canonical_to_da3_world=np.asarray(frame.get("M_canonical_to_da3_world", np.eye(4)), np.float32),
        world_align_s=np.float32(world_align_s if not GT_NATIVE else 1.0),
        world_align_R=np.asarray(world_align_R if not GT_NATIVE else np.eye(3), np.float32),
        world_align_t=np.asarray(world_align_t if not GT_NATIVE else np.zeros(3), np.float32),
        train_h=TRAIN_H, train_w=TRAIN_W, view_names=np.array(view_names),
    )
    print("Wrote NeuS dataset:", case_dir, "| views:", len(view_names), "| shapes:", shapes[0], "| camera_src:", _export_src)
    if not GT_NATIVE and BANEUS_TYPE_SOURCES == "mvs":
        export_cam4feat_pair(view_names, SCENE_DIR, case_dir)

    if "POSE_AUDIT_DIR" not in globals():
        _out = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path("/content/baneus_ultrra_out")
        POSE_AUDIT_DIR = _out / "pose_audit"
        POSE_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
    if "pose_audit_log" not in globals():
        raise RuntimeError("Run §2.6 (pose audit helpers) before §12.1")

    _ref = (np.stack([GEO_REF[n]["c2w"] for n in view_names]) if HAS_GEO_JSON
            else np.stack(c2ws))
    _ref_label = "geo_json" if HAS_GEO_JSON else "init_export"
    pose_audit_log("02_neus_export", np.stack(c2ws), view_names, ref_c2ws=_ref, ref_label=_ref_label, Ks=Ks,
                   align_sim3=False, extra={"camera_src": _export_src, "geo_aligned": ALIGN_DA3_TO_JSON_GEO})
    NEUS_INIT_C2WS = np.stack(c2ws).copy()  # baseline for training drift checks
    np.savez(POSE_AUDIT_DIR / "neus_export_baseline.npz", c2ws=NEUS_INIT_C2WS, view_names=np.array(view_names), camera_src=_export_src)

    # Back up critical NeuS export files to Drive
    if Path("/content/drive/MyDrive").exists():
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
        for _ef in (OUTPUT_DIR / "baneus_scene_meta.npz",
                    POSE_AUDIT_DIR / "neus_export_baseline.npz",
                    case_dir / "cameras_sphere.npz"):
            if _ef.exists():
                shutil.copy2(_ef, DRIVE_OUT / _ef.name)
        print("NeuS export meta backed up to Drive ->", DRIVE_OUT)

    # ---- sanity check ----
    import shutil
    from datetime import datetime, timezone
    _ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

    if GT_NATIVE:
        # Normalized object points must sit inside the unit sphere (NeuS assumption),
        # and decoded camera centers must equal (C_world - center)/radius.
        _pn = (obj_pts - center) / radius
        _dn = np.linalg.norm(_pn, axis=1)
        print(f"GT-native normalized object radius: mean={_dn.mean():.3f} p99={np.percentile(_dn,99):.3f} max={_dn.max():.3f} (want <~1.0)")
        _cam = np.load(case_dir / "cameras_sphere.npz")
        import sys as _sys; _sys.path.insert(0, str(BANEUS_ROOT))
        from utils.camera_utils import load_K_Rt_from_P as _lkr
        _decC, _wldC = [], []
        for i, name in enumerate(view_names):
            P = _cam[f"world_mat_{i}"] @ _cam[f"scale_mat_{i}"]
            _, c2w_dec = _lkr(None, P[:3, :4].astype(np.float64))
            _decC.append(c2w_dec[:3, 3]); _wldC.append((GEO_REF[name]["c2w"][:3, 3] - center) / radius)
        _err = float(np.linalg.norm(np.array(_decC) - np.array(_wldC), axis=1).max())
        print(f"decoded camera center vs expected normalized center: max_err={_err:.6f} (want ~0)")
        print("GT-native export OK — cameras + object share one normalized frame."
              if _err < 1e-3 else "WARN: normalized-decode mismatch — check build_neus_cameras.")
    else:
        import trimesh, matplotlib.pyplot as plt
        _check_dir = OUTPUT_DIR / "baneus_frame_check" / _ts
        _check_dir.mkdir(parents=True, exist_ok=True)
        s1_glb = OUTPUT_DIR / "stage1" / "stage1_coarse_mesh.glb"
        assert s1_glb.exists(), f"Missing {s1_glb} — run §9.3 first"
        s1_mesh = _as_trimesh(trimesh.load(s1_glb, process=False))
        v_can = np.asarray(s1_mesh.vertices, np.float64)
        _M = np.asarray(frame.get("M_canonical_to_da3_world", np.eye(4)), np.float64)
        if not np.allclose(_M, np.eye(4)):
            v_world = stage1_canonical_to_da3_world(v_can, _M)
        else:
            _fc2w = np.asarray(frame["ref_c2w"], np.float64) if "ref_c2w" in frame else None
            _M = stage1_canonical_to_da3_world_matrix(
                frame["scale"], frame["rotation"], frame["translation"],
                c2w_opencv=_fc2w, device="cpu")
            v_world = stage1_canonical_to_da3_world(v_can, _M)
        v_world = apply_sim3_points(v_world, world_align_s, world_align_R, world_align_t)
        v_norm = (v_world - center) / radius
        v_can_rt = stage1_da3_world_to_canonical(
            invert_sim3_points(v_world, world_align_s, world_align_R, world_align_t), _M)
        err_rt = np.linalg.norm(v_can_rt - v_can, axis=1)
        print("Stage-1 canonical AABB:", v_can.min(0).round(4), "->", v_can.max(0).round(4))
        print("In DA3 world AABB:", v_world.min(0).round(4), "->", v_world.max(0).round(4))
        print("In BA-NeuS norm AABB:", v_norm.min(0).round(4), "->", v_norm.max(0).round(4))
        print(f"Round-trip canonical error: max={err_rt.max():.6f} mean={err_rt.mean():.6f}")

        # DA3 depth cloud was already computed above; reuse it for the final check
        _dc, _dr = da3_center, da3_radius
        print("Final NeuS frame center:", np.asarray(center).round(3), "| radius:", round(float(radius), 3))

        _cam_c = np.stack([np.asarray(c, np.float64)[:3, 3] for c in c2ws])
        _cam_norm = np.linalg.norm((_cam_c - center) / radius, axis=1)
        _obj_p99 = float(np.percentile(np.linalg.norm(v_norm, axis=1), 99))
        print(f"Normalized camera distance: min={_cam_norm.min():.3f} mean={_cam_norm.mean():.3f} "
              f"max={_cam_norm.max():.3f} (NeuS wants >1) | object p99 radius={_obj_p99:.3f} (wants <=1)")
        if _cam_norm.min() < 1.0:
            print("FAIL: cameras sit inside the NeuS unit sphere — rays start inside the object")
            print("      (near_far_from_sphere clamps near to 0) and training cannot converge.")
        else:
            print("OK: every camera is outside the unit sphere.")
        norm_mesh = trimesh.Trimesh(vertices=v_norm.astype(np.float32), faces=s1_mesh.faces, process=False)
        _glb = _check_dir / f"stage1_in_baneus_norm_{_ts}.glb"
        _png = _check_dir / f"stage1_baneus_norm_preview_{_ts}.png"
        norm_mesh.export(_glb)
        fig = plt.figure(figsize=(12, 5))
        ax1 = fig.add_subplot(121, projection="3d")
        ax1.plot_trisurf(v_can[:, 0], v_can[:, 1], v_can[:, 2], triangles=s1_mesh.faces, color="orange", linewidth=0, alpha=0.9)
        ax1.set_title("Stage-1 canonical"); ax1.set_box_aspect((1, 1, 1))
        ax2 = fig.add_subplot(122, projection="3d")
        ax2.plot_trisurf(v_norm[:, 0], v_norm[:, 1], v_norm[:, 2], triangles=s1_mesh.faces, color="steelblue", linewidth=0, alpha=0.9)
        ax2.set_title("Same mesh in BA-NeuS norm"); ax2.set_box_aspect((1, 1, 1))
        plt.tight_layout(); fig.savefig(_png, dpi=120, bbox_inches="tight"); plt.show()
        print("Saved frame-check assets:", _check_dir)
        if Path("/content/drive/MyDrive").exists():
            drive_ts = DRIVE_OUT / "frame_checks" / _ts
            drive_ts.mkdir(parents=True, exist_ok=True)
            for p in (_glb, _png):
                shutil.copy2(p, drive_ts / p.name)
            print("Copied to Drive:", drive_ts)


## 12.2 — Prior vs. images (the decisive alignment check)

Projects the Stage-1 coarse mesh through the **exported** BA-NeuS cameras
(`cameras_sphere.npz`) and compares the silhouette with the SAM masks the network trains on.

If the projected mesh lands on the building in every view, then Stage-1 pose, DA3 cameras and
the NeuS normalization all share one frame — which is exactly what the occupancy prior needs.
Depth-cloud statistics (§12.1) can be skewed by background inside the mask; this test cannot.


In [ ]:
# 12.2 — Project the Stage-1 prior into the exported training views
import sys
import numpy as np
import cv2
import trimesh
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from pytorch3d.transforms import quaternion_to_matrix

if GT_NATIVE:
    print("GT-native path: no Stage-1 occupancy prior to check.")
else:
    if str(BANEUS_ROOT) not in sys.path:
        sys.path.insert(0, str(BANEUS_ROOT))
    from utils.camera_utils import load_K_Rt_from_P

    if "compose_transform" not in dir():   # allow running this cell without §12.1 in kernel
        from pytorch3d.transforms import Transform3d

        def compose_transform(scale, rotation, translation, device="cpu"):
            dev = torch.device(device)
            scale = torch.as_tensor(scale, dtype=torch.float32, device=dev)
            rotation = torch.as_tensor(rotation, dtype=torch.float32, device=dev)
            translation = torch.as_tensor(translation, dtype=torch.float32, device=dev)
            if translation.dim() == 1:
                translation = translation.unsqueeze(0)
            if scale.dim() == 1:
                scale = scale.unsqueeze(0)
            if rotation.dim() == 2:
                rotation = rotation.unsqueeze(0)
            return Transform3d(device=dev, dtype=torch.float32).scale(scale).rotate(rotation).translate(translation)

    case_dir = BANEUS_ROOT / "public_data" / CASE_NAME
    cam = np.load(case_dir / "cameras_sphere.npz")
    meta = np.load(OUTPUT_DIR / "baneus_scene_meta.npz")
    center = np.asarray(meta["center"], np.float64)
    radius = float(meta["radius"])

    _m = trimesh.load(OUTPUT_DIR / "stage1" / "stage1_coarse_mesh.glb", process=False)
    if isinstance(_m, trimesh.Scene):
        _m = trimesh.util.concatenate([g for g in _m.geometry.values() if isinstance(g, trimesh.Trimesh)])
    v_can = np.asarray(_m.vertices, np.float64)

    _M = np.asarray(meta.get("M_canonical_to_da3_world", np.eye(4)), np.float64)
    if np.allclose(_M, np.eye(4)):
        _mc2w = np.asarray(meta["ref_c2w"], np.float64) if "ref_c2w" in meta else None
        _M = stage1_canonical_to_da3_world_matrix(
            meta["scale"], meta["rotation"], meta["translation"],
            c2w_opencv=_mc2w, device="cpu")
    v_world = stage1_canonical_to_da3_world(v_can, _M)
    _was = float(meta.get("world_align_s", 1.0))
    _waR = np.asarray(meta.get("world_align_R", np.eye(3)), np.float64)
    _wat = np.asarray(meta.get("world_align_t", np.zeros(3)), np.float64)
    print(f"Sim3 rotation det={np.linalg.det(_waR):.4f}  "
          f"off-identity={float(np.abs(_waR - np.eye(3)).max()):.4f}")
    v_world = apply_sim3_points(v_world, _was, _waR, _wat)
    v_norm = (v_world - center) / radius          # frame the exported cameras live in
    faces = np.asarray(_m.faces, np.int32)

    n_views = len([k for k in cam.files if k.startswith("world_mat_")])
    _sample = list(range(0, n_views, max(1, n_views // 8)))[:8]

    def _rasterize_silhouette(v_nrm, fcs, w2c, K, H, W):
        """Rasterize front-facing mesh triangles into a filled binary silhouette."""
        Xc = v_nrm @ w2c[:3, :3].T + w2c[:3, 3]
        uv_h = Xc @ K.T
        z = uv_h[:, 2].copy()
        valid_z = z > 1e-6
        uv = np.zeros((len(v_nrm), 2), np.float64)
        uv[valid_z, 0] = uv_h[valid_z, 0] / z[valid_z]
        uv[valid_z, 1] = uv_h[valid_z, 1] / z[valid_z]
        sil = np.zeros((H, W), np.uint8)
        for f in fcs:
            if not (valid_z[f[0]] and valid_z[f[1]] and valid_z[f[2]]):
                continue
            e1 = Xc[f[1]] - Xc[f[0]]
            e2 = Xc[f[2]] - Xc[f[0]]
            nz = e1[0] * e2[1] - e1[1] * e2[0]       # z-component of cross product
            if nz >= 0:                                 # back-face -> skip
                continue
            pts = uv[f].astype(np.int32).reshape(1, 3, 2)
            cv2.fillPoly(sil, pts, 255)
        return sil > 127

    ious, precs, recs, panels = [], [], [], []
    for i in _sample:
        P = cam[f"world_mat_{i}"] @ cam[f"scale_mat_{i}"]
        K4, c2w = load_K_Rt_from_P(None, P[:3, :4].astype(np.float64))
        K = np.asarray(K4, np.float64)[:3, :3]
        w2c = np.linalg.inv(np.asarray(c2w, np.float64))

        img = cv2.cvtColor(cv2.imread(str(case_dir / "image" / f"{i:03d}.png")), cv2.COLOR_BGR2RGB)
        msk = cv2.imread(str(case_dir / "mask" / f"{i:03d}.png"), cv2.IMREAD_GRAYSCALE) > 127
        H, W = msk.shape

        proj_sil = _rasterize_silhouette(v_norm, faces, w2c, K, H, W)

        inter = float((proj_sil & msk).sum())
        union = float((proj_sil | msk).sum()) + 1e-9
        ious.append(inter / union)
        precs.append(inter / (proj_sil.sum() + 1e-9))
        recs.append(inter / (msk.sum() + 1e-9))
        if len(panels) < 3:
            panels.append((i, img, msk, proj_sil))

    ious, precs, recs = np.array(ious), np.array(precs), np.array(recs)
    print(f"Views checked: {_sample}")
    print(f"Silhouette IoU  (mesh vs SAM mask): mean={ious.mean():.3f} min={ious.min():.3f} max={ious.max():.3f}")
    print(f"  mesh-on-object (precision): mean={precs.mean():.3f} | object-covered (recall): mean={recs.mean():.3f}")
    if ious.mean() > 0.45:
        print("OK: the Stage-1 prior projects onto the building — prior, cameras and frame agree.")
    elif ious.mean() > 0.2:
        print("PARTIAL: prior overlaps the building but is offset or mis-scaled.")
        print("  Low recall = prior too small; low precision = prior too large or shifted.")
    else:
        print("FAIL: the Stage-1 prior does not project onto the building — frame mismatch.")

    # --- Diagnostic: search over axis-aligned 90° rotations ---
    _rot90 = {
        'identity':      np.eye(3),
        'Rx+90':  np.array([[1,0,0],[0,0,-1],[0,1,0]], dtype=np.float64),
        'Rx-90':  np.array([[1,0,0],[0,0,1],[0,-1,0]], dtype=np.float64),
        'Ry+90':  np.array([[0,0,1],[0,1,0],[-1,0,0]], dtype=np.float64),
        'Ry-90':  np.array([[0,0,-1],[0,1,0],[1,0,0]], dtype=np.float64),
        'Rz+90':  np.array([[0,-1,0],[1,0,0],[0,0,1]], dtype=np.float64),
        'Rz-90':  np.array([[0,1,0],[-1,0,0],[0,0,1]], dtype=np.float64),
        'Rx180':  np.array([[1,0,0],[0,-1,0],[0,0,-1]], dtype=np.float64),
        'Ry180':  np.array([[-1,0,0],[0,1,0],[0,0,-1]], dtype=np.float64),
        'Rz180':  np.array([[-1,0,0],[0,-1,0],[0,0,1]], dtype=np.float64),
    }
    print("\n--- Rotation correction search (post-hoc on v_norm) ---")
    for rname, Rcorr in _rot90.items():
        _vn = v_norm @ Rcorr
        _ious_r = []
        for ii in _sample:
            P = cam[f"world_mat_{ii}"] @ cam[f"scale_mat_{ii}"]
            K4, c2w_ii = load_K_Rt_from_P(None, P[:3, :4].astype(np.float64))
            Ki = np.asarray(K4, np.float64)[:3, :3]
            w2c_ii = np.linalg.inv(np.asarray(c2w_ii, np.float64))
            msk_ii = cv2.imread(str(case_dir / "mask" / f"{ii:03d}.png"), cv2.IMREAD_GRAYSCALE) > 127
            sil_r = _rasterize_silhouette(_vn, faces, w2c_ii, Ki, msk_ii.shape[0], msk_ii.shape[1])
            inter_r = float((sil_r & msk_ii).sum())
            union_r = float((sil_r | msk_ii).sum()) + 1e-9
            _ious_r.append(inter_r / union_r)
        _ious_r = np.array(_ious_r)
        tag = ' <-- BEST' if _ious_r.mean() > ious.mean() + 0.05 else ''
        print(f"  {rname:10s}  IoU mean={_ious_r.mean():.3f} min={_ious_r.min():.3f} max={_ious_r.max():.3f}{tag}")

    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4), squeeze=False)
    for ax, (i, img, msk, proj_sil) in zip(axes[0], panels):
        ax.imshow(img)
        ax.contour(msk.astype(float), levels=[0.5], colors="lime", linewidths=1.2)
        ax.contour(proj_sil.astype(float), levels=[0.5], colors="red", linewidths=1.2)
        ax.set_title(f"view {i:03d} — mask (green) vs Stage-1 (red)")
        ax.axis("off")
    fig.tight_layout()
    _png = OUTPUT_DIR / "stage1" / "prior_reprojection_check.png"
    fig.savefig(_png, dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved:", _png)
    if Path("/content/drive/MyDrive").exists():
        import shutil
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(_png, DRIVE_OUT / _png.name)
        print("Copied to Drive:", DRIVE_OUT / _png.name)


In [ ]:
# 12.3 — Export debug alignment bundle to Drive
# Downloads one folder to debug the alignment locally.
import shutil, numpy as np
_dbg = DRIVE_OUT / "debug_alignment"
_dbg.mkdir(parents=True, exist_ok=True)
case_dir = BANEUS_ROOT / "public_data" / CASE_NAME

for src, dst_name in [
    (OUTPUT_DIR / "stage1" / "stage1_frame.npz", "stage1_frame.npz"),
    (OUTPUT_DIR / "stage1" / "stage1_coarse_mesh.obj", "stage1_coarse_mesh.obj"),
    (OUTPUT_DIR / "baneus_scene_meta.npz", "baneus_scene_meta.npz"),
    (case_dir / "cameras_sphere.npz", "cameras_sphere.npz"),
    (case_dir / "mask" / "000.png", "mask_000.png"),
    (case_dir / "image" / "000.png", "image_000.png"),
]:
    if src.exists():
        shutil.copy2(src, _dbg / dst_name)
        print(f"  ✓ {dst_name}")
    else:
        print(f"  ✗ {dst_name} — source missing: {src}")

# DA3 npz (large — copy only the arrays we need)
da3 = np.load(DA3_NPZ)
np.savez_compressed(
    _dbg / "da3_slim.npz",
    extrinsics=da3["extrinsics"],
    intrinsics=da3["intrinsics"],
    image_files=da3["image_files"],
)
print(f"  ✓ da3_slim.npz (extrinsics + intrinsics + image_files)")

# Also save the raw decoded pose tensors for direct testing
np.savez(
    _dbg / "decoded_pose.npz",
    scale=ss["scale"].detach().cpu().numpy() if hasattr(ss["scale"], 'detach') else np.array(ss["scale"]),
    rotation=ss["rotation"].detach().cpu().numpy() if hasattr(ss["rotation"], 'detach') else np.array(ss["rotation"]),
    translation=ss["translation"].detach().cpu().numpy() if hasattr(ss["translation"], 'detach') else np.array(ss["translation"]),
    pointmap_scale=ss.get("pointmap_scale").detach().cpu().numpy() if hasattr(ss.get("pointmap_scale", 0), 'detach') else np.array(ss.get("pointmap_scale", [1.0])),
    pointmap_shift=ss.get("pointmap_shift").detach().cpu().numpy() if hasattr(ss.get("pointmap_shift", 0), 'detach') else np.array(ss.get("pointmap_shift", [0.0, 0.0, 0.0])),
    view_names=np.array(view_names) if 'view_names' in dir() else np.array([]),
)
print(f"  ✓ decoded_pose.npz")

print(f"\nAll saved to: {_dbg}")
print("Download this folder from Drive and place contents in:")
print("  /Users/taiganishida/Documents/GSV-MSV/debug_alignment/")

## 13 — Stage-1 prior + train BA-NeuS

**Full BA-NeuS (va_split aerial path):** `init_pose=True`, `learn_pose=True`, `type_sources=sequential`, `depth_weight=0.1`, phased `pc_weight=[0,0.1,0.01]`, `feat_weight=[0,0,0]` (no VisMVSNet / pair.txt on va_split), plus `prior_weight=0.35`.


In [ ]:
# 13.0 — Stage-1 occupancy prior (aligned: NeuS norm <-> canonical cube)
import numpy as np, torch, torch.nn.functional as F
from pytorch3d.transforms import quaternion_to_matrix, Transform3d

def compose_transform(scale, rotation, translation, device="cpu"):
    """PyTorch3D Translate needs (N,3); keep pose + points on the same device."""
    if translation is None:
        raise ValueError("compose_transform: translation is None")
    dev = torch.device(device)
    scale = torch.as_tensor(scale, dtype=torch.float32, device=dev)
    rotation = torch.as_tensor(rotation, dtype=torch.float32, device=dev)
    translation = torch.as_tensor(translation, dtype=torch.float32, device=dev)
    if translation.dim() == 1:
        translation = translation.unsqueeze(0)
    if scale.dim() == 1:
        scale = scale.unsqueeze(0)
    if rotation.dim() == 2:
        rotation = rotation.unsqueeze(0)
    tfm = Transform3d(device=dev, dtype=torch.float32)
    return tfm.scale(scale).rotate(rotation).translate(translation)

if not BANEUS_USE_PRIOR:
    # GT-native: no DA3 Stage-1 prior. Define harmless stubs so later cells don't crash.
    def sample_prob(x_norm):
        return torch.zeros(x_norm.shape[0], device=x_norm.device)
    def pseudo_sdf_from_prob(p):
        return p * 0.0
    print("GT-native: Stage-1 occupancy prior DISABLED (prior_weight=0). Using GT depth + mask + RGB only.")
else:
    occ = np.load(OUTPUT_DIR / "stage1" / "stage1_occupancy.npz")
    prob = torch.from_numpy(occ["prob"]).float().cuda()
    meta = np.load(OUTPUT_DIR / "baneus_scene_meta.npz")
    center = torch.tensor(meta["center"], dtype=torch.float32)
    radius = float(meta["radius"])
    _M = torch.tensor(meta.get("M_canonical_to_da3_world", np.eye(4)), dtype=torch.float32, device="cuda")
    _M_inv = torch.tensor(np.linalg.inv(_M.cpu().numpy()), dtype=torch.float32, device="cuda")
    _was = float(meta.get("world_align_s", 1.0))
    _waR = torch.tensor(meta.get("world_align_R", np.eye(3)), dtype=torch.float32, device="cuda")
    _wat = torch.tensor(meta.get("world_align_t", np.zeros(3)), dtype=torch.float32, device="cuda")

    def neus_norm_to_canonical(x_norm):
        """BA-NeuS normalized coords -> Stage-1 canonical [-0.5,0.5]^3."""
        dev = x_norm.device
        x_world = x_norm * radius + center.to(dev)
        x_world = (x_world - _wat) @ _waR.T / _was
        ones = torch.ones(x_world.shape[0], 1, device=dev, dtype=torch.float32)
        return (torch.cat([x_world, ones], dim=-1) @ _M_inv)[:, :3]

    @torch.no_grad()
    def sample_prob(x_norm):
        g = (neus_norm_to_canonical(x_norm) / 0.5).clamp(-0.99, 0.99)
        g = g[:, [2, 1, 0]].view(1, -1, 1, 1, 3)
        v = F.grid_sample(prob[None, None], g, align_corners=True, padding_mode="border")
        return v.view(-1)

    def pseudo_sdf_from_prob(p):
        return (0.5 - p) * 0.2

    prior_patch = BANEUS_ROOT / "optimizers" / "prior_patch.py"
    prior_patch.write_text(
        "import torch\n"
        "_PRIOR = None\n"
        "def register_prior(fn):\n    global _PRIOR; _PRIOR = fn\n"
        "def prior_loss(renderer, n=4096):\n"
        "    if _PRIOR is None:\n        return torch.zeros((), device='cuda')\n"
        "    pts = torch.randn(n, 3, device='cuda')\n"
        "    pts = pts / (pts.norm(dim=-1, keepdim=True) + 1e-6) * torch.rand(n, 1, device='cuda') ** 0.33\n"
        "    tgt = _PRIOR(pts).detach()\n"
        "    pred = renderer.sdf_network.sdf(pts)\n"
        "    return ((pred - tgt) ** 2).mean()\n"
    )
    print("Prior mapping: NeuS norm -> Stage-1 canonical (pose-aware)")


In [ ]:
# 13.1 — Colab-friendly BA-NeuS config
conf_path = BANEUS_ROOT / "configs" / "ba_aerial_s1prior.conf"
_init = str(BANEUS_INIT_POSE).lower()
_learn = str(BANEUS_LEARN_POSE).lower()
_pcw = "[" + ", ".join(str(x) for x in BANEUS_PC_WEIGHT) + "]"
_featw = "[" + ", ".join(str(x) for x in BANEUS_FEAT_WEIGHT) + "]"
_phd = "[" + ", ".join(str(x) for x in BANEUS_PHASE_DELIM) + "]"
conf_text = f"""
general {{
    base_exp_dir = ./exp/{CASE_NAME}/ba_aerial
    exp_name = {CASE_NAME}_ba_aerial
    recording = [./, ./models]
}}

dataset {{
    data_dir = ./public_data/{CASE_NAME}/
    render_cameras_name = cameras_sphere.npz
    object_cameras_name = cameras_sphere.npz
    init_pose = {_init}
    init_intrinsic = true
    learn_pose = {_learn}
    learn_intrinsic = false
    noise_magnitude = {BANEUS_POSE_NOISE}
    trans_offset = 1.5
    learn_scale = true
    learn_shift = true
    fix_scaleN = true
    num_sources = 2
    type_sources = "{BANEUS_TYPE_SOURCES}"
}}

train {{
    learning_rate = 5e-4
    learning_rate_alpha = 0.05
    end_iter = {BANEUS_END_ITER}
    batch_size = 512
    validate_resolution_level = 4
    warm_up_end = 5000
    anneal_end = 0
    use_white_bkgd = false
    save_freq = {BANEUS_SAVE_FREQ}
    val_freq = {BANEUS_VAL_FREQ}
    val_mesh_freq = {BANEUS_VAL_MESH}
    report_freq = 100
    igr_weight = 0.1
    mask_weight = 0.1
    depth_weight = {BANEUS_DEPTH_WEIGHT}
    prior_weight = {BANEUS_PRIOR_W}
    use_masked_loss = true
    use_ssi_depth_loss = false
    phase_delim = {_phd}
    pc_weight = {_pcw}
    feat_weight = {_featw}
    depth_from_inside_only = [false, true, true]
    object_mask_type = "zero_sdf_mask"
}}

model {{
    nerf {{ D=8, d_in=4, d_in_view=3, W=256, multires=10, multires_view=4, barf_c2f=[0.3,0.6], output_ch=4, skips=[4], use_viewdirs=true }}
    sdf_network {{ d_out=257, d_in=3, d_hidden=256, n_layers=8, skip_in=[4], multires=6, barf_c2f=[0.3,0.6], bias=0.5, scale=1.0, geometric_init=true, weight_norm=true, inside_outside=false }}
    variance_network {{ init_val = 0.3 }}
    rendering_network {{ d_feature=256, mode=idr, d_in=9, d_out=3, d_hidden=256, n_layers=4, weight_norm=true, multires_view=4, barf_c2f=[0.3,0.6], squeeze_out=true }}
    neus_renderer {{ n_samples=64, n_importance=64, n_outside=0, up_sample_steps=4, perturb=1.0 }}
}}
""".strip()
conf_path.write_text(conf_text)
from pyhocon import ConfigFactory
_c = ConfigFactory.parse_file(str(conf_path))
print("Config ->", conf_path)
print("  init_pose=", _c.get_bool("dataset.init_pose"), "learn_pose=", _c.get_bool("dataset.learn_pose"))
print("  type_sources=", _c.get_string("dataset.type_sources"))
print("  depth_weight=", _c.get_float("train.depth_weight"), "prior_weight=", _c.get_float("train.prior_weight"))
print("  pc_weight=", list(_c.get_list("train.pc_weight")), "feat_weight=", list(_c.get_list("train.feat_weight")))

In [ ]:
# 13.2 — Brief SDF pre-fit to Stage-1 occupancy
import sys
import shutil
import torch
import torch.nn.functional as F
sys.path.insert(0, str(BANEUS_ROOT))
from models.fields import SDFNetwork
from pyhocon import ConfigFactory

SDF_PREFIT_ITERS = 25000
_prefit_tag = f"{SDF_PREFIT_ITERS}it"
_prefit_path = OUTPUT_DIR / f"stage1_sdf_prefit_{_prefit_tag}.pth"
_prefit_latest = OUTPUT_DIR / "stage1_sdf_prefit.pth"  # fixed name for §13.3 loader
if not BANEUS_USE_PRIOR:
    # GT-native: no prior -> no prefit. Remove any stale (wrong-frame) prefit so training
    # starts from the SDF geometric_init sphere.
    for p in (_prefit_path, _prefit_latest):
        if p.exists():
            p.unlink()
    print("GT-native: skipping SDF pre-fit (SDF starts from geometric_init sphere).")
else:
    conf = ConfigFactory.parse_file(str(BANEUS_ROOT / "configs" / "ba_aerial_s1prior.conf"))
    sdf = SDFNetwork(**conf["model.sdf_network"]).cuda()
    opt = torch.optim.Adam(sdf.parameters(), lr=1e-3)
    for it in range(SDF_PREFIT_ITERS):
        pts = torch.randn(8192, 3, device="cuda")
        pts = pts / (pts.norm(dim=-1, keepdim=True) + 1e-6) * torch.rand(8192, 1, device="cuda") ** 0.33
        with torch.no_grad():
            tgt = pseudo_sdf_from_prob(sample_prob(pts)).unsqueeze(-1)  # (N,1) matches sdf output
        pred = sdf.sdf(pts)
        loss = F.mse_loss(pred, tgt)
        opt.zero_grad(); loss.backward(); opt.step()
        if it % 100 == 0:
            print(f"prefit {it:4d}  mse={loss.item():.5f}")
    _ckpt = {"sdf_network": sdf.state_dict(), "n_iters": SDF_PREFIT_ITERS}
    torch.save(_ckpt, _prefit_path)
    shutil.copy2(_prefit_path, _prefit_latest)
    print("Saved", _prefit_path, "| latest ->", _prefit_latest.name)


In [ ]:
# 13.2b — Export prefitted SDF MLP isosurface (before 80k BA-NeuS training)
import sys
import shutil
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt
from pathlib import Path

_prefit_latest = OUTPUT_DIR / "stage1_sdf_prefit.pth"
_out_dir = OUTPUT_DIR / "stage1"
_out_dir.mkdir(parents=True, exist_ok=True)

if not BANEUS_USE_PRIOR:
    print("GT-native: no SDF prefit mesh to export (§13.2 skipped).")
elif not _prefit_latest.exists():
    print("Missing", _prefit_latest, "— run §13.2 first.")
else:
    sys.path.insert(0, str(BANEUS_ROOT))
    from pyhocon import ConfigFactory
    from models.fields import SDFNetwork
    from utils.geometry_utils import extract_geometry

    conf = ConfigFactory.parse_file(str(BANEUS_ROOT / "configs" / "ba_aerial_s1prior.conf"))

    # Reuse in-memory network from §13.2 when available; otherwise reload checkpoint.
    if "sdf" in globals() and isinstance(sdf, SDFNetwork):
        sdf_net = sdf.cuda().eval()
        _n_iters = int(globals().get("SDF_PREFIT_ITERS", 500))
        print("Using in-memory SDF from §13.2")
    else:
        _ckpt = torch.load(_prefit_latest, map_location="cuda", weights_only=False)
        _n_iters = int(_ckpt.get("n_iters", globals().get("SDF_PREFIT_ITERS", 500)))
        _prefit_path = OUTPUT_DIR / f"stage1_sdf_prefit_{_n_iters}it.pth"
        if not _prefit_path.exists():
            torch.save(_ckpt, _prefit_path)
        sdf_net = SDFNetwork(**conf["model.sdf_network"]).cuda().eval()
        sdf_net.load_state_dict(_ckpt["sdf_network"])
        print("Loaded SDF from", _prefit_path if _prefit_path.exists() else _prefit_latest)

    _prefit_tag = f"{_n_iters}it"

    _dev = next(sdf_net.parameters()).device
    _res = 128  # 64 matches Stage-1 occupancy grid; 128 is a sharper preview
    bound_min = torch.tensor([-1.0, -1.0, -1.0], device=_dev, dtype=torch.float32)
    bound_max = torch.tensor([1.0, 1.0, 1.0], device=_dev, dtype=torch.float32)

    def _query_sdf(pts):
        # extract_geometry builds grid points on CPU; SDF + barf_c2f progress live on CUDA.
        return -sdf_net.sdf(pts.to(device=_dev, dtype=torch.float32))

    with torch.no_grad():
        verts, faces = extract_geometry(
            bound_min,
            bound_max,
            resolution=_res,
            threshold=0.0,
            query_func=_query_sdf,
        )

    mesh_norm = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
    ply_norm = _out_dir / f"sdf_prefit_{_prefit_tag}_mesh_norm.ply"
    glb_norm = _out_dir / f"sdf_prefit_{_prefit_tag}_mesh_norm.glb"
    mesh_norm.export(ply_norm)
    mesh_norm.export(glb_norm)
    print(f"SDF prefit mesh (NeuS norm): {len(mesh_norm.vertices)} verts, {len(mesh_norm.faces)} faces @ res={_res} ({_prefit_tag})")
    print("Saved:", ply_norm.name, "|", glb_norm.name)

    # Same frame as stage1_coarse_mesh.glb for direct visual comparison.
    v_can = neus_norm_to_canonical(torch.tensor(verts, dtype=torch.float32, device="cuda")).cpu().numpy()
    mesh_can = trimesh.Trimesh(vertices=v_can, faces=faces, process=True)
    ply_can = _out_dir / f"sdf_prefit_{_prefit_tag}_mesh_canonical.ply"
    glb_can = _out_dir / f"sdf_prefit_{_prefit_tag}_mesh_canonical.glb"
    mesh_can.export(ply_can)
    mesh_can.export(glb_can)
    print("Saved canonical frame:", glb_can.name)

    # 3-angle preview (canonical frame, same views as §9.3)
    vx, vy, vz = mesh_can.vertices[:, 0], mesh_can.vertices[:, 1], mesh_can.vertices[:, 2]
    fig = plt.figure(figsize=(15, 5))
    for i, (az, el) in enumerate([(-60, 25), (30, 25), (90, 80)]):
        ax = fig.add_subplot(1, 3, i + 1, projection="3d")
        ax.plot_trisurf(vx, vy, vz, triangles=mesh_can.faces, cmap="coolwarm", linewidth=0, antialiased=False)
        ax.view_init(elev=el, azim=az)
        ax.set_box_aspect((1, 1, 1))
        ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
        ax.set_title(f"SDF prefit ({az},{el})")
    plt.suptitle(f"SDF MLP after Stage-1 occupancy pre-fit ({_prefit_tag}, before 80k train)")
    plt.tight_layout()
    png_path = _out_dir / f"sdf_prefit_{_prefit_tag}_mesh.png"
    fig.savefig(png_path, dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved preview:", png_path.name)

    _prefit_ckpt = OUTPUT_DIR / f"stage1_sdf_prefit_{_prefit_tag}.pth"
    if not _prefit_ckpt.exists() and _prefit_latest.exists():
        shutil.copy2(_prefit_latest, _prefit_ckpt)

    # Persist to Drive immediately (survives Colab disconnect).
    _drive_prefit = DRIVE_OUT / "prefit"
    _drive_prefit.mkdir(parents=True, exist_ok=True)
    _drive_exports = [_prefit_ckpt, ply_norm, glb_norm, ply_can, glb_can, png_path]
    if _prefit_latest.exists():
        _drive_exports.append(_prefit_latest)
    try:
        if Path("/content/drive/MyDrive").exists():
            for p in _drive_exports:
                if p.exists():
                    shutil.copy2(p, _drive_prefit / p.name)
            print("Copied prefit assets to Drive:", _drive_prefit)
        else:
            print("Drive not mounted — skipped Drive copy (local files kept under", _out_dir, ")")
    except Exception as e:
        print("Prefit Drive copy failed:", e)

    s1_glb = _out_dir / "stage1_coarse_mesh.glb"
    if s1_glb.exists():
        s1 = trimesh.load(s1_glb, process=False)
        if isinstance(s1, trimesh.Scene):
            s1 = trimesh.util.concatenate([g for g in s1.geometry.values() if isinstance(g, trimesh.Trimesh)])
        print(
            "Compare with Stage-1 coarse mesh:",
            s1_glb.name,
            f"({len(s1.vertices)} verts) vs sdf prefit ({len(mesh_can.vertices)} verts)",
        )


In [ ]:
# 13.3 — Train BA-NeuS (in-process — shows real tracebacks, prior callback works)
import os, sys, traceback
import torch

os.chdir(BANEUS_ROOT)
os.environ["WANDB_DISABLED"] = "true"
if BANEUS_USE_PRIOR:
    os.environ["BANEUS_SDF_PREFIT"] = str(OUTPUT_DIR / "stage1_sdf_prefit.pth")
else:
    os.environ.pop("BANEUS_SDF_PREFIT", None)  # GT-native: no prefit, SDF geometric_init
if str(BANEUS_ROOT) not in sys.path:
    sys.path.insert(0, str(BANEUS_ROOT))

from optimizers.prior_patch import register_prior
if BANEUS_USE_PRIOR:
    register_prior(lambda x: pseudo_sdf_from_prob(sample_prob(x)))
else:
    register_prior(None)  # GT-native: no Stage-1 topology prior

torch.cuda.set_device(0)
torch.set_default_dtype(torch.float32)
# Repo was written for torch.set_default_tensor_type('torch.cuda.FloatTensor');
# replicate with set_default_device so bare torch.ones/zeros land on CUDA.
torch.set_default_device('cuda')
# Runtime monkeypatch — always fixes CPU/CUDA ray mismatch on Colab + torch 2.x
import importlib
import models.camera as _cammod
importlib.reload(_cammod)  # ensure §11.1 zero-init pose MLP patch is active even in a warm kernel
import dataset.dataset as _dsmod
importlib.reload(_dsmod)  # rebinds LearnPose from reloaded models.camera; resets near_far wraps
from utils.camera_utils import scale_camera

def _gen_random_rays_at_cuda(self, img_idx, batch_size):
    dev = self.device
    ii = int(img_idx.item() if torch.is_tensor(img_idx) else img_idx)

    # CPU indices for image/mask/depth buffers (stored on CPU in Dataset)
    # images/masks/depth live on CPU — index with CPU pixel coords
    pixels_x = torch.randint(0, self.W, (batch_size,), device="cpu")
    pixels_y = torch.randint(0, self.H, (batch_size,), device="cpu")
    color = self.images[ii][(pixels_y, pixels_x)]
    mask = self.masks[ii][(pixels_y, pixels_x)]
    depth = self.depths[ii][(pixels_y, pixels_x)]
    depth = self.undistort_depth(ii, depth)

    Kinv = self.intrinsic_network(inverse=True)[:3, :3].to(dev)
    p = torch.stack([
        pixels_x.to(device=dev, dtype=torch.float32),
        pixels_y.to(device=dev, dtype=torch.float32),
        torch.ones(batch_size, device=dev),
    ], dim=-1)
    p = torch.matmul(Kinv, p.unsqueeze(-1)).squeeze(-1)
    rays_v_norm = torch.linalg.norm(p, ord=2, dim=-1, keepdim=True).clamp_min(1e-8)
    rays_v = p / rays_v_norm

    c2w = self.pose_network(ii).to(dev)
    rays_v = torch.matmul(c2w[:3, :3], rays_v.unsqueeze(-1)).squeeze(-1)
    rays_o = c2w[:3, 3].expand_as(rays_v)

    # Source views: mvs pair.txt when available, else sequential neighbors (stock BA-NeuS fallback)
    if getattr(self, "type_src", "sequential") == "mvs" and hasattr(self, "pair"):
        _id = self.pair["id_list"][ii]
        _src_ids = self.pair[_id]["pair"]
        src_idxs = [self.pair[s]["index"] for s in _src_ids][: self.num_src]
    else:
        src_idxs = [(ii - 1) % self.n_images, (ii + 1) % self.n_images]

    depth_cam = torch.stack([
        self.pose_network(ii).to(dev),
        self.intrinsic_network().to(dev),
    ], dim=0)
    ref_cam = scale_camera(depth_cam, self.pc_scale)
    ref_depth = self.undistort_depth(ii, self.pc_scaled_depths[ii]).to(dev)
    src_cams = torch.stack([
        scale_camera(
            torch.stack([self.pose_network(i).to(dev), self.intrinsic_network().to(dev)], dim=0),
            self.pc_scale,
        )
        for i in src_idxs
    ])
    src_depths = torch.stack([
        self.undistort_depth(i, self.pc_scaled_depths[i]).to(dev) for i in src_idxs
    ])
    pc_input = {
        "ref_cam": ref_cam,
        "ref_depth": ref_depth,
        "src_cams": src_cams,
        "src_depths": src_depths,
        "pc_pixels": self.pc_pixels,
    }

    # VisMVSNet feature loss inputs (when type_sources=mvs and feats were precomputed at Dataset init)
    if getattr(self, "type_src", "sequential") == "mvs" and hasattr(self, "feats"):
        _feat_cam = scale_camera(depth_cam, self.feat_img_scale)
        _feat_src_cams = torch.stack([
            scale_camera(
                torch.stack([self.pose_network(i).to(dev), self.intrinsic_network().to(dev)], dim=0),
                self.feat_img_scale,
            )
            for i in src_idxs
        ])
        feat_input = {
            "depth_cams": depth_cam,
            "size": self.size,
            "center": self.center,
            "feat": self.feats[ii],
            "feat_src": self.feats[src_idxs],
            "cam": _feat_cam,
            "src_cams": _feat_src_cams,
            "rays_v_norm": rays_v_norm,
            "H": self.H,
            "W": self.W,
        }
    else:
        feat_input = None

    data = torch.cat([
        rays_o,
        rays_v,
        color.to(device=dev, dtype=torch.float32),
        mask[:, :1].to(device=dev, dtype=torch.float32),
        depth.to(device=dev, dtype=torch.float32),
    ], dim=-1)
    return data, feat_input, pc_input

if not getattr(_dsmod.Dataset, "_AERIAL_RAYS_CUDA", False):
    _dsmod.Dataset._AERIAL_RAYS_ORIG = getattr(_dsmod.Dataset, "_AERIAL_RAYS_ORIG", _dsmod.Dataset.gen_random_rays_at)
    _dsmod.Dataset.gen_random_rays_at = _gen_random_rays_at_cuda
    _dsmod.Dataset._AERIAL_RAYS_CUDA = True
    print("Applied CUDA ray-generation monkeypatch")
else:
    print("CUDA ray-generation monkeypatch already active")
# --- renderer CUDA: full literal patches (same as §11.1) ---
def _patch_renderer_cuda_literals(rp=BANEUS_ROOT / "models" / "renderer.py"):
    """Patch all CPU-default tensor literals in renderer.py for CUDA training."""
    import re as _re
    p = Path(rp)
    if not p.exists():
        return False
    txt = p.read_text()
    orig = txt

    # Repair earlier bad patch that commented out cumprod(..., -1)
    txt = txt.replace("], -1)  # AERIAL_ONES_CUDA, -1", "], -1), -1")

    txt = _re.sub(
        r"torch\.Tensor\(\[sample_dist\]\)\.expand\(dists\[\.\.\., :1\]\.shape\)",
        "torch.tensor([sample_dist], device=dists.device, dtype=dists.dtype).expand(dists[..., :1].shape)",
        txt,
    )

    subs = [
        (
            "z_vals = torch.linspace(0.0, 1.0, self.n_samples)\n        z_vals = near + (far - near) * z_vals[None, :]",
            "_dev, _dt = rays_o.device, rays_o.dtype\n        near = near.to(device=_dev, dtype=_dt)\n        far = far.to(device=_dev, dtype=_dt)\n        z_vals = torch.linspace(0.0, 1.0, self.n_samples, device=_dev, dtype=_dt)  # AERIAL_ZVALS_CUDA\n        z_vals = near + (far - near) * z_vals[None, :]",
        ),
        (
            "z_vals = torch.linspace(0.0, 1.0, self.n_samples, device=near.device, dtype=near.dtype)  # AERIAL_ZVALS_CUDA\n        z_vals = near + (far - near) * z_vals[None, :]",
            "_dev, _dt = rays_o.device, rays_o.dtype\n        near = near.to(device=_dev, dtype=_dt)\n        far = far.to(device=_dev, dtype=_dt)\n        z_vals = torch.linspace(0.0, 1.0, self.n_samples, device=_dev, dtype=_dt)  # AERIAL_ZVALS_CUDA\n        z_vals = near + (far - near) * z_vals[None, :]",
        ),
        (
            "t_rand = torch.rand([batch_size, 1]) - 0.5",
            "t_rand = torch.rand([batch_size, 1], device=_dev, dtype=_dt) - 0.5  # AERIAL_ZVALS_CUDA",
        ),
        (
            "t_rand = torch.rand([batch_size, z_vals_outside.shape[-1]])",
            "t_rand = torch.rand([batch_size, z_vals_outside.shape[-1]], device=_dev, dtype=_dt)  # AERIAL_ZVALS_CUDA",
        ),
        (
            "[torch.zeros([batch_size, 1]), cos_val[:, :-1]], dim=-1",
            "[torch.zeros([batch_size, 1], device=cos_val.device, dtype=cos_val.dtype), cos_val[:, :-1]], dim=-1",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1]), 1.0 - alpha + 1e-7], -1), -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha.device, dtype=alpha.dtype), 1.0 - alpha + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1]), 1.0 - alpha_in + 1e-7], -1), -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha_in.device, dtype=alpha_in.dtype), 1.0 - alpha_in + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha.device, dtype=alpha.dtype), 1.0 - alpha + 1e-7], -1)  # AERIAL_ONES_CUDA, -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha.device, dtype=alpha.dtype), 1.0 - alpha + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha_in.device, dtype=alpha_in.dtype), 1.0 - alpha_in + 1e-7], -1)  # AERIAL_ONES_CUDA, -1\n            )",
            "torch.cumprod(\n                torch.cat([torch.ones([batch_size, 1], device=alpha_in.device, dtype=alpha_in.dtype), 1.0 - alpha_in + 1e-7], -1),\n                -1,\n            )",
        ),
        (
            "inv_s = deviation_network(torch.zeros([1, 3]))[:, :1].clip(",
            "inv_s = deviation_network(torch.zeros([1, 3], device=pts.device, dtype=pts.dtype))[:, :1].clip(",
        ),
        (
            "torch.arange(batch_size)[:, None]",
            "torch.arange(batch_size, device=z_vals.device)[:, None]",
        ),
        (
            "pc_loss = torch.tensor(0)",
            "pc_loss = torch.tensor(0.0, device=pts.device, dtype=pts.dtype)",
        ),
        (
            "feat_loss = torch.tensor(0)",
            "feat_loss = torch.tensor(0.0, device=pts.device, dtype=pts.dtype)",
        ),
    ]
    for old, new in subs:
        if old in txt:
            txt = txt.replace(old, new)

    for old in [
        "t_rand = torch.rand([batch_size, 1], device=near.device, dtype=near.dtype) - 0.5  # AERIAL_ZVALS_CUDA",
        "t_rand = torch.rand([batch_size, z_vals_outside.shape[-1]], device=near.device, dtype=near.dtype) - 0.5  # AERIAL_ZVALS_CUDA",
    ]:
        if old in txt:
            txt = txt.replace(old, old.replace("near.device", "_dev").replace("near.dtype", "_dt"))

    if txt != orig:
        p.write_text(txt)
        print("patched renderer.py (full CUDA literals)")
        return True
    if "device=alpha.device" in txt and "], -1)  # AERIAL_ONES_CUDA, -1" not in txt:
        print("renderer.py already has full CUDA literal patches")
        return False
    print("WARN: renderer.py CUDA literal patch incomplete — check file manually")
    return False


_patch_renderer_cuda_literals()


# --- geometry_utils.py: sample_pdf u on cdf.device (searchsorted CUDA) ---
_geom = BANEUS_ROOT / "utils" / "geometry_utils.py"
if _geom.exists():
    _gt = _geom.read_text()
    _g_old = """        u = torch.linspace(
            0.0 + 0.5 / n_samples, 1.0 - 0.5 / n_samples, steps=n_samples
        )
        u = u.expand(list(cdf.shape[:-1]) + [n_samples])
    else:
        u = torch.rand(list(cdf.shape[:-1]) + [n_samples])"""
    _g_new = """        u = torch.linspace(
            0.0 + 0.5 / n_samples, 1.0 - 0.5 / n_samples, steps=n_samples,
            device=cdf.device, dtype=cdf.dtype,
        )  # AERIAL_SAMPLEPDF_CUDA
        u = u.expand(list(cdf.shape[:-1]) + [n_samples])
    else:
        u = torch.rand(list(cdf.shape[:-1]) + [n_samples], device=cdf.device, dtype=cdf.dtype)  # AERIAL_SAMPLEPDF_CUDA"""
    if "AERIAL_SAMPLEPDF_CUDA" not in _gt and _g_old in _gt:
        _geom.write_text(_gt.replace(_g_old, _g_new))
        print("patched geometry_utils.py (sample_pdf on CUDA)")
    elif "AERIAL_SAMPLEPDF_CUDA" in _gt:
        print("geometry_utils.py already has sample_pdf CUDA patch")
    else:
        print("WARN: sample_pdf block not found in geometry_utils.py")

# --- fields.py: SingleVarianceNetwork.forward torch.ones on param device ---
_fields = BANEUS_ROOT / "models" / "fields.py"
if _fields.exists():
    _ft = _fields.read_text()
    _f_old = "return torch.ones([len(x), 1]) * torch.exp(self.variance * 10.0)"
    _f_new = "return torch.ones([len(x), 1], device=self.variance.device, dtype=self.variance.dtype) * torch.exp(self.variance * 10.0)  # AERIAL_VAR_CUDA"
    if "AERIAL_VAR_CUDA" not in _ft and _f_old in _ft:
        _fields.write_text(_ft.replace(_f_old, _f_new))
        print("patched fields.py (SingleVarianceNetwork device)")
    elif "AERIAL_VAR_CUDA" in _ft:
        print("fields.py already has SingleVarianceNetwork device patch")
    else:
        print("WARN: SingleVarianceNetwork forward not found in fields.py")


# Guard pc_input / feat_input .cuda() in train loop when sequential/aerial
_opt = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
_ot = _opt.read_text()
_pc_old = """            for attr in [\"ref_cam\", \"ref_depth\", \"src_cams\", \"src_depths\", \"pc_pixels\"]:
                pc_input[attr] = pc_input[attr].cuda()"""
_pc_new = """            if pc_input is not None:  # AERIAL_PC_NONE_PATCH
                for attr in [\"ref_cam\", \"ref_depth\", \"src_cams\", \"src_depths\", \"pc_pixels\"]:
                    pc_input[attr] = pc_input[attr].cuda()"""
if "AERIAL_PC_NONE_PATCH" not in _ot and _pc_old in _ot:
    _ot = _ot.replace(_pc_old, _pc_new)
    _opt.write_text(_ot)
    print("patched ba_neus_optimizer.py (pc_input optional)")

from pyhocon import ConfigFactory
_ts = ConfigFactory.parse_file(str(BANEUS_ROOT / "configs" / "ba_aerial_s1prior.conf")).get_string("dataset.type_sources")
print("Training with dataset.type_sources =", repr(_ts))
_pair_txt = BANEUS_ROOT / "public_data" / CASE_NAME / "cam4feat" / "pair.txt"
if _ts.strip().strip('"') == "mvs":
    assert _pair_txt.is_file(), f"Missing {_pair_txt} — re-run §12.1 (exports cam4feat from BlendedMVS pair.txt)"
    print("cam4feat/pair.txt OK — mvs source selection + VisMVSNet feature loss enabled")
else:
    print("sequential source views — feature loss weight should be 0")


# Patch train loop: feat_input is None in sequential/aerial mode
_opt = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
_ot = _opt.read_text()
_feat_old = """            for attr in [
                \"depth_cams\",
                \"size\",
                \"center\",
                \"feat\",
                \"feat_src\",
                \"cam\",
                \"src_cams\",
                \"rays_v_norm\",
            ]:
                feat_input[attr] = feat_input[attr].cuda()
            for attr in [\"H\", \"W\"]:
                feat_input[attr] = feat_input[attr]"""
_feat_new = """            if feat_input is not None:  # AERIAL_FEAT_NONE_PATCH
                for attr in [
                    \"depth_cams\",
                    \"size\",
                    \"center\",
                    \"feat\",
                    \"feat_src\",
                    \"cam\",
                    \"src_cams\",
                    \"rays_v_norm\",
                ]:
                    feat_input[attr] = feat_input[attr].cuda()
                for attr in [\"H\", \"W\"]:
                    feat_input[attr] = feat_input[attr]"""
if "AERIAL_FEAT_NONE_PATCH" not in _ot and _feat_old in _ot:
    _opt.write_text(_ot.replace(_feat_old, _feat_new))
    print("patched ba_neus_optimizer.py (feat_input optional)")
elif "AERIAL_FEAT_NONE_PATCH" in _ot:
    print("ba_neus_optimizer.py already has feat_input guard")
else:
    print("WARN: feat_input block not found — check ba_neus_optimizer.py")

import importlib
import models.renderer as _rmod
importlib.reload(_rmod)

def _wrap_neus_render_cuda():
    cls = _rmod.NeuSRenderer
    if getattr(cls, "_AERIAL_RENDER_WRAP", False):
        return
    if not hasattr(cls, "_AERIAL_RENDER_ORIG"):
        cls._AERIAL_RENDER_ORIG = cls.render
    _orig_render = cls._AERIAL_RENDER_ORIG
    def _render_cuda(self, rays_o, rays_d, near, far, *args, **kwargs):
        dev, dt = rays_o.device, rays_o.dtype
        return _orig_render(
            self,
            rays_o.to(dev), rays_d.to(dev),
            near.to(device=dev, dtype=dt), far.to(device=dev, dtype=dt),
            *args, **kwargs,
        )
    cls.render = _render_cuda
    cls._AERIAL_RENDER_WRAP = True
    print("Wrapped NeuSRenderer.render (post-reload)")

_wrap_neus_render_cuda()

# --- runtime sample_pdf: device-safe (u follows cdf), injected into both namespaces ---
import utils.geometry_utils as _geomod
importlib.reload(_geomod)

def _sample_pdf_cuda(bins, weights, n_samples, det=False):
    import torch as _t
    weights = weights + 1e-5
    pdf = weights / _t.sum(weights, -1, keepdim=True)
    cdf = _t.cumsum(pdf, -1)
    cdf = _t.cat([_t.zeros_like(cdf[..., :1]), cdf], -1)
    dev, dt = cdf.device, cdf.dtype
    if det:
        u = _t.linspace(0.0 + 0.5 / n_samples, 1.0 - 0.5 / n_samples, steps=n_samples, device=dev, dtype=dt)
        u = u.expand(list(cdf.shape[:-1]) + [n_samples])
    else:
        u = _t.rand(list(cdf.shape[:-1]) + [n_samples], device=dev, dtype=dt)
    u = u.contiguous()
    inds = _t.searchsorted(cdf, u, right=True)
    below = _t.max(_t.zeros_like(inds - 1), inds - 1)
    above = _t.min((cdf.shape[-1] - 1) * _t.ones_like(inds), inds)
    inds_g = _t.stack([below, above], -1)
    matched_shape = [inds_g.shape[0], inds_g.shape[1], cdf.shape[-1]]
    cdf_g = _t.gather(cdf.unsqueeze(1).expand(matched_shape), 2, inds_g)
    bins_g = _t.gather(bins.unsqueeze(1).expand(matched_shape), 2, inds_g)
    denom = cdf_g[..., 1] - cdf_g[..., 0]
    denom = _t.where(denom < 1e-5, _t.ones_like(denom), denom)
    tt = (u - cdf_g[..., 0]) / denom
    return bins_g[..., 0] + tt * (bins_g[..., 1] - bins_g[..., 0])

_geomod.sample_pdf = _sample_pdf_cuda
_rmod.sample_pdf = _sample_pdf_cuda
print("Injected device-safe sample_pdf")

# near/far device: handled by AERIAL_NEAR_CUDA in ba_neus_optimizer.py + render wrapper


def _patch_ba_neus_background_rgb():
    """Fix background_rgb device: train uses rays_o tensor; validate uses rays_o_batch."""
    p = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
    if not p.exists():
        return
    txt = p.read_text()
    orig = txt

    # repair earlier bad global replace
    txt = txt.replace(
        "background_rgb = torch.ones([1, 3], device=rays_o.device, dtype=rays_o.dtype)  # AERIAL_BG_CUDA if self.use_white_bkgd else None",
        "background_rgb = (torch.ones([1, 3], device=rays_o_batch.device, dtype=rays_o_batch.dtype) if self.use_white_bkgd else None)  # AERIAL_BG_CUDA",
    )

    subs = [
        (
            "            background_rgb = None\n            if self.use_white_bkgd:\n                background_rgb = torch.ones([1, 3])",
            "            background_rgb = None\n            if self.use_white_bkgd:\n                background_rgb = torch.ones([1, 3], device=rays_o.device, dtype=rays_o.dtype)  # AERIAL_BG_CUDA",
        ),
        (
            "            background_rgb = torch.ones([1, 3]) if self.use_white_bkgd else None",
            "            background_rgb = (torch.ones([1, 3], device=rays_o_batch.device, dtype=rays_o_batch.dtype) if self.use_white_bkgd else None)  # AERIAL_BG_CUDA",
        ),
    ]
    for old, new in subs:
        if old in txt:
            txt = txt.replace(old, new)

    if txt != orig:
        p.write_text(txt)
        print("patched ba_neus_optimizer.py (background_rgb per-context)")
    elif "AERIAL_BG_CUDA" in txt:
        print("ba_neus_optimizer.py already has background_rgb patch")

_patch_ba_neus_background_rgb()

# train-loop near/far alignment in optimizer source
_opt = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
_ot = _opt.read_text()
_nf_old = "near, far = self.dataset.near_far_from_sphere(rays_o, rays_d)"
_nf_new = (
    "near, far = self.dataset.near_far_from_sphere(rays_o, rays_d)\n"
    "            near = near.to(device=rays_o.device, dtype=rays_o.dtype)  # AERIAL_NEAR_CUDA\n"
    "            far = far.to(device=rays_o.device, dtype=rays_o.dtype)  # AERIAL_NEAR_CUDA"
)
if "AERIAL_NEAR_CUDA" not in _ot and _nf_old in _ot:
    _opt.write_text(_ot.replace(_nf_old, _nf_new))
    print("patched ba_neus_optimizer.py (near/far device)")

for _m in ("models.renderer", "optimizers.ba_neus_optimizer", "optimizers"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])
_wrap_neus_render_cuda()  # re-apply after optimizer reload
import utils.geometry_utils as _geomod2
_geomod2.sample_pdf = _sample_pdf_cuda
_rmod.sample_pdf = _sample_pdf_cuda  # renderer reload rebinds name -> re-inject

os.environ["BANEUS_TRAIN_LOG"] = str(OUTPUT_DIR / "baneus_train.log")

# Log iter/loss/VRAM to file (survives Colab output clears + silent GPU kills)
_opt = BANEUS_ROOT / "optimizers" / "ba_neus_optimizer.py"
_ot = _opt.read_text()
_log_old = """            print(
                "iter:{:8>d} loss={} color_loss={} eikonal_loss={} feat_loss={} depth_loss={} pc_loss={} psnr={} lr={}".format(
                    self.iter_step,
                    loss,
                    color_fine_loss,
                    eikonal_loss,
                    feat_loss,
                    depth_loss,
                    pc_loss,
                    psnr,
                    self.optimizer.param_groups[0]["lr"],
                )
            )"""
_log_new = """            _msg = "iter:{:8>d} loss={} color_loss={} eikonal_loss={} feat_loss={} depth_loss={} pc_loss={} psnr={} lr={} vram={:.2f}GB".format(
                self.iter_step,
                loss,
                color_fine_loss,
                eikonal_loss,
                feat_loss,
                depth_loss,
                pc_loss,
                psnr,
                self.optimizer.param_groups[0]["lr"],
                torch.cuda.memory_allocated() / 1e9,
            )
            print(_msg)
            _lp = os.environ.get("BANEUS_TRAIN_LOG")
            if _lp:
                with open(_lp, "a") as _lf:
                    _lf.write(_msg + "\n")  # AERIAL_TRAIN_LOG"""
if "AERIAL_TRAIN_LOG" not in _ot and _log_old in _ot:
    _opt.write_text(_ot.replace(_log_old, _log_new))
    print("patched ba_neus_optimizer.py (train log file)")

_ckpt_dir = BANEUS_ROOT / "exp" / CASE_NAME / "ba_aerial" / "checkpoints"
_resume = bool(list(_ckpt_dir.glob("ckpt_*.pth"))) if _ckpt_dir.exists() else False
print(f"Checkpoints: {_ckpt_dir}  resume={_resume}")
if _resume:
    _latest = sorted(_ckpt_dir.glob("ckpt_*.pth"))[-1]
    print(f"  latest: {_latest.name}")
print(f"Train log -> {OUTPUT_DIR / 'baneus_train.log'}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("NOTE: blank cell output usually = Colab GPU disconnect or OOM kill (no Python traceback).")
print("      Check baneus_train.log + checkpoints/ after a stop.")

# ---- pose audit during BA-NeuS training ----
import numpy as np

def _baneus_extract_c2ws(dataset):
    out = []
    for i in range(dataset.n_images):
        with torch.no_grad():
            out.append(dataset.pose_network(i).detach().cpu().numpy())
    return np.stack(out)

def _wrap_runner_train_pose_log(runner):
    if getattr(runner, "_pose_log_wrapped", False):
        return
    _orig_train = runner.train

    def _train_with_pose_log():
        _names = list(view_names)
        if HAS_GEO_JSON:
            _ref_gt = np.stack([GEO_REF[n]["c2w"] for n in _names])
            _ref_label = "dataset_gt"
        else:
            _ref_gt = globals().get("NEUS_INIT_C2WS")
            _ref_label = "init_export"
        if _ref_gt is None:
            _ref_gt = _baneus_extract_c2ws(runner.dataset)
        pose_audit_log(
            f"03_baneus_iter{runner.iter_step:08d}_start",
            _baneus_extract_c2ws(runner.dataset), _names,
            ref_c2ws=_ref_gt, ref_label=_ref_label, align_sim3=False,
            extra={"learn_pose": bool(runner.dataset.learn_pose), "iter": int(runner.iter_step)},
        )
        _freq = int(globals().get("POSE_LOG_FREQ", 0) or 0)

        def _maybe_log(tag):
            pose_audit_log(
                tag,
                _baneus_extract_c2ws(runner.dataset), _names,
                ref_c2ws=_ref_gt, ref_label=_ref_label, align_sim3=False,
                extra={"learn_pose": bool(runner.dataset.learn_pose), "iter": int(runner.iter_step)},
            )

        if _freq > 0:
            _orig_save = runner.save_checkpoint
            def _save_and_log(*a, **kw):
                _orig_save(*a, **kw)
                if runner.iter_step > 0 and runner.iter_step % _freq == 0:
                    _maybe_log(f"03_baneus_iter{runner.iter_step:08d}")
            runner.save_checkpoint = _save_and_log

        try:
            _orig_train()
        finally:
            _maybe_log(f"03_baneus_iter{runner.iter_step:08d}_end")
            if HAS_GEO_JSON:
                _ref_gt = np.stack([GEO_REF[n]["c2w"] for n in _names]); _ref_label = "gt"
            else:
                _ref_gt = globals().get("NEUS_INIT_C2WS"); _ref_label = "init_export"
            if _ref_gt is None:
                _ref_gt = _baneus_extract_c2ws(runner.dataset); _ref_label = "self"
            pose_audit_log(
                "04_baneus_final_vs_init" if not HAS_GEO_JSON else "04_baneus_final_vs_geo",
                _baneus_extract_c2ws(runner.dataset), _names,
                ref_c2ws=_ref_gt, ref_label=_ref_label, align_sim3=False,
                extra={"learn_pose": bool(runner.dataset.learn_pose), "iter": int(runner.iter_step)},
            )

    runner.train = _train_with_pose_log
    runner._pose_log_wrapped = True


try:
    from optimizers.ba_neus_optimizer import Runner
    runner = Runner(
        str(BANEUS_ROOT / "configs" / "ba_aerial_s1prior.conf"),
        mode="train",
        case=CASE_NAME,
        is_continue=_resume,
    )
    _wrap_runner_train_pose_log(runner)
    runner.train()
    print("Training finished.")
except Exception:
    traceback.print_exc()
    raise
finally:
    _lp = OUTPUT_DIR / "baneus_train.log"
    if _lp.exists():
        print("--- tail of baneus_train.log ---")
        print("".join(_lp.read_text().splitlines(True)[-5:]))

## 14 — Extract mesh, preview, save to Drive


In [ ]:
# 14.1 — Extract mesh, preview, save (GT-native world OR DA3 Stage-1 comparison)
import os, sys, traceback, shutil
import numpy as np
import trimesh, matplotlib.pyplot as plt
from pathlib import Path
import torch
from pytorch3d.transforms import quaternion_to_matrix, Transform3d


def _as_trimesh(obj):
    """GLB/OBJ may load as trimesh.Scene — collapse to a single Trimesh."""
    import trimesh
    if isinstance(obj, trimesh.Trimesh):
        return obj
    if isinstance(obj, trimesh.Scene):
        geoms = [g for g in obj.geometry.values() if isinstance(g, trimesh.Trimesh)]
        if not geoms:
            raise ValueError("Scene has no mesh geometry")
        return trimesh.util.concatenate(geoms) if len(geoms) > 1 else geoms[0]
    raise TypeError(f"Expected Trimesh or Scene, got {type(obj)}")

def compose_transform(scale, rotation, translation, device="cpu"):
    if translation is None:
        raise ValueError("compose_transform: translation is None")
    dev = torch.device(device)
    scale = torch.as_tensor(scale, dtype=torch.float32, device=dev)
    rotation = torch.as_tensor(rotation, dtype=torch.float32, device=dev)
    translation = torch.as_tensor(translation, dtype=torch.float32, device=dev)
    if translation.dim() == 1:
        translation = translation.unsqueeze(0)
    if scale.dim() == 1:
        scale = scale.unsqueeze(0)
    if rotation.dim() == 2:
        rotation = rotation.unsqueeze(0)
    tfm = Transform3d(device=dev, dtype=torch.float32)
    return tfm.scale(scale).rotate(rotation).translate(translation)

def neus_norm_to_world(v_norm, meta):
    """Inverse of NeuS scale_mat: x_world = x_norm * radius + center."""
    c = np.asarray(meta["center"], np.float64)
    r = float(meta["radius"])
    return np.asarray(v_norm, np.float64) * r + c

def neus_norm_to_canonical(v_norm, meta, device="cpu"):
    """NeuS norm -> DA3 world -> inverse world-align -> inverse M_can2w -> canonical."""
    dev = torch.device(device)
    c = torch.tensor(meta["center"], dtype=torch.float32, device=dev)
    r = float(meta["radius"])
    M = np.asarray(meta.get("M_canonical_to_da3_world", np.eye(4)), np.float64)
    M_inv = torch.tensor(np.linalg.inv(M), dtype=torch.float32, device=dev)
    was = float(meta.get("world_align_s", 1.0))
    waR = torch.tensor(meta.get("world_align_R", np.eye(3)), dtype=torch.float32, device=dev)
    wat = torch.tensor(meta.get("world_align_t", np.zeros(3)), dtype=torch.float32, device=dev)
    v = torch.as_tensor(v_norm, dtype=torch.float32, device=dev)
    v_world = v * r + c
    v_world = (v_world - wat) @ waR.T / was
    ones = torch.ones(v_world.shape[0], 1, device=dev, dtype=torch.float32)
    return (torch.cat([v_world, ones], dim=-1) @ M_inv)[:, :3].cpu().numpy()

def aabb_iou(a_min, a_max, b_min, b_max):
    lo = np.maximum(a_min, b_min)
    hi = np.minimum(a_max, b_max)
    inter = np.maximum(0.0, hi - lo)
    inter_vol = float(np.prod(inter))
    vol_a = float(np.prod(np.maximum(a_max - a_min, 1e-9)))
    vol_b = float(np.prod(np.maximum(b_max - b_min, 1e-9)))
    union = vol_a + vol_b - inter_vol
    return inter_vol, (inter_vol / union if union > 1e-12 else 0.0)

torch.set_default_device("cpu")
os.chdir(BANEUS_ROOT)
os.environ["WANDB_DISABLED"] = "true"
if str(BANEUS_ROOT) not in sys.path:
    sys.path.insert(0, str(BANEUS_ROOT))

# --- load BA-NeuS mesh (always from disk) ---
exp_dir = BANEUS_ROOT / "exp" / CASE_NAME / "ba_aerial"
meshes = sorted(exp_dir.rglob("*.ply"))
assert meshes, f"No BA-NeuS mesh under {exp_dir}"
baneus_mesh = _as_trimesh(trimesh.load(meshes[-1], process=False))
print("BA-NeuS mesh:", meshes[-1], "| verts", len(baneus_mesh.vertices))

meta = np.load(OUTPUT_DIR / "baneus_scene_meta.npz")
v_norm = np.asarray(baneus_mesh.vertices, np.float64)
v_world = neus_norm_to_world(v_norm, meta)

out_dir = OUTPUT_DIR / "baneus"; out_dir.mkdir(parents=True, exist_ok=True)

# Tag exports with SDF pre-fit length (0it on GT-native / no-prior path).
_prefit_tag = "0it"
if BANEUS_USE_PRIOR:
    _prefit_ckpt = OUTPUT_DIR / "stage1_sdf_prefit.pth"
    if _prefit_ckpt.exists():
        import torch as _torch
        _ck = _torch.load(str(_prefit_ckpt), map_location="cpu", weights_only=False)
        _prefit_tag = f"{int(_ck.get('n_iters', globals().get('SDF_PREFIT_ITERS', 500)))}it"
    elif "SDF_PREFIT_ITERS" in globals():
        _prefit_tag = f"{int(SDF_PREFIT_ITERS)}it"
print("Export tag (prefit iters):", _prefit_tag)

def _baneus_export_name(frame, ext="glb"):
    return f"baneus_mesh_{frame}_{_prefit_tag}.{ext}"

baneus_world = trimesh.Trimesh(vertices=v_world.astype(np.float32), faces=baneus_mesh.faces, process=False)
baneus_norm = trimesh.Trimesh(vertices=v_norm.astype(np.float32), faces=baneus_mesh.faces, process=False)
baneus_world.export(out_dir / _baneus_export_name("world"))
baneus_norm.export(out_dir / _baneus_export_name("norm"))

if GT_NATIVE:
    # World frame IS the geo JSON world; canonical == world (identity object pose).
    baneus_world.export(out_dir / _baneus_export_name("canonical"))
    print("AABB world  BA-NeuS:", v_world.min(0).round(3), "->", v_world.max(0).round(3))
    print("AABB norm   BA-NeuS:", v_norm.min(0).round(3), "->", v_norm.max(0).round(3))
    print("Frame center/radius:", np.asarray(meta["center"]).round(3), float(meta["radius"]))
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_trisurf(v_world[:, 0], v_world[:, 1], v_world[:, 2], triangles=baneus_mesh.faces,
                    color="steelblue", linewidth=0, alpha=0.9)
    ax.set_title("BA-NeuS in GT world"); ax.set_box_aspect((1, 1, 1))
    png = out_dir / f"baneus_world_{_prefit_tag}.png"
    plt.tight_layout(); fig.savefig(png, dpi=120, bbox_inches="tight"); plt.show()
    print("Exported:", _baneus_export_name("world"), _baneus_export_name("norm"), _baneus_export_name("canonical"))
    if Path("/content/drive/MyDrive").exists():
        drive_dir = DRIVE_OUT; drive_dir.mkdir(parents=True, exist_ok=True)
        for p in [out_dir / _baneus_export_name("world"), out_dir / _baneus_export_name("norm"),
                  out_dir / _baneus_export_name("canonical"), png, OUTPUT_DIR / "baneus_scene_meta.npz"]:
            if p.exists():
                shutil.copy2(p, drive_dir / p.name)
        print("Copied results to Drive:", drive_dir)
    else:
        print("Outputs in", out_dir)
else:
    s1_path = OUTPUT_DIR / "stage1" / "stage1_coarse_mesh.glb"
    assert s1_path.exists(), f"Missing {s1_path} — re-run §9.3"
    s1_mesh = _as_trimesh(trimesh.load(s1_path, process=False))
    s1_verts = np.asarray(s1_mesh.vertices, np.float64)
    print("Stage-1 mesh:", s1_path, "| verts", len(s1_verts))

    frame = np.load(OUTPUT_DIR / "stage1" / "stage1_frame.npz")
    _s1_match = (np.allclose(meta["center"], frame["center"], rtol=0, atol=1e-2)
                  and abs(float(meta["radius"]) - float(frame["radius"])) < 1e-2)
    if not _s1_match:
        print("NOTE: baneus_scene_meta center/radius differs from stage1_frame.npz")
        print("      (expected when §12.1 fell back to DA3 depth cloud for normalization)")

    v_can = neus_norm_to_canonical(v_norm, meta, device="cpu")
    baneus_aligned = trimesh.Trimesh(vertices=v_can, faces=baneus_mesh.faces, process=False)

    s1_min, s1_max = s1_verts.min(0), s1_verts.max(0)
    ba_min, ba_max = v_can.min(0), v_can.max(0)
    wn_min, wn_max = v_world.min(0), v_world.max(0)
    R = quaternion_to_matrix(torch.tensor(meta["rotation"], dtype=torch.float32))
    sc = torch.tensor(meta["scale"], dtype=torch.float32).reshape(-1)
    if sc.numel() == 1:
        sc = sc.expand(3)
    tfm_l2w = compose_transform(
        scale=sc, rotation=R,
        translation=torch.tensor(meta["translation"], dtype=torch.float32).reshape(3),
        device="cpu",
    )
    s1_world = tfm_l2w.transform_points(torch.tensor(s1_verts, dtype=torch.float32)).numpy()
    sw_min, sw_max = s1_world.min(0), s1_world.max(0)

    print("AABB canonical  Stage-1:", s1_min.round(4), "->", s1_max.round(4))
    print("AABB canonical  BA-NeuS:", ba_min.round(4), "->", ba_max.round(4))
    print("AABB world      Stage-1:", sw_min.round(4), "->", sw_max.round(4))
    print("AABB world      BA-NeuS:", wn_min.round(4), "->", wn_max.round(4))
    print("AABB norm       BA-NeuS:", v_norm.min(0).round(4), "->", v_norm.max(0).round(4))
    print("Frame center/radius:", np.asarray(meta["center"]).round(3), float(meta["radius"]))

    _, iou_can = aabb_iou(ba_min, ba_max, s1_min, s1_max)
    _, iou_world = aabb_iou(wn_min, wn_max, sw_min, sw_max)
    overlap = bool(np.all(ba_max >= s1_min) and np.all(ba_min <= s1_max))
    print(f"Canonical AABB overlap: {overlap} | IoU={iou_can:.3f}")
    print(f"World AABB overlap: {np.all(wn_max >= sw_min) and np.all(wn_min <= sw_max)} | IoU={iou_world:.3f}")

    glb = out_dir / _baneus_export_name("canonical")
    obj = out_dir / _baneus_export_name("canonical", "obj")
    baneus_aligned.export(glb); baneus_aligned.export(obj)
    print("Exported:", _baneus_export_name("world"), _baneus_export_name("norm"), _baneus_export_name("canonical"))

    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot_trisurf(s1_world[:, 0], s1_world[:, 1], s1_world[:, 2], triangles=s1_mesh.faces,
                     color="orange", linewidth=0, alpha=0.9)
    ax1.set_title("Stage-1 in WORLD"); ax1.set_box_aspect((1, 1, 1))
    wlims = np.vstack([np.minimum(sw_min, wn_min), np.maximum(sw_max, wn_max)])
    ax1.set_xlim(wlims[:, 0]); ax1.set_ylim(wlims[:, 1]); ax1.set_zlim(wlims[:, 2])
    ax2 = fig.add_subplot(122, projection="3d")
    ax2.plot_trisurf(v_world[:, 0], v_world[:, 1], v_world[:, 2], triangles=baneus_mesh.faces,
                     color="steelblue", linewidth=0, alpha=0.85)
    ax2.set_title("BA-NeuS in WORLD"); ax2.set_box_aspect((1, 1, 1))
    ax2.set_xlim(wlims[:, 0]); ax2.set_ylim(wlims[:, 1]); ax2.set_zlim(wlims[:, 2])
    plt.tight_layout()
    png = out_dir / f"baneus_vs_stage1_world_{_prefit_tag}.png"
    fig.savefig(png, dpi=120, bbox_inches="tight"); plt.show()

    if Path("/content/drive/MyDrive").exists():
        drive_dir = DRIVE_OUT; drive_dir.mkdir(parents=True, exist_ok=True)
        for p in [glb, obj, out_dir / _baneus_export_name("world"), out_dir / _baneus_export_name("norm"), png,
                  OUTPUT_DIR / "stage1" / "stage1_coarse_mesh.glb",
                  OUTPUT_DIR / "baneus_scene_meta.npz"]:
            if p.exists():
                shutil.copy2(p, drive_dir / p.name)
        print("Copied results to Drive:", drive_dir)
    else:
        print("Outputs in", out_dir)


## 14.2 — Did poses drift during training?

Compare BA-NeuS poses against the **geo-aligned init** from §12 (and JSON camera centers from §2.6).

- **Start** (`03_*_start`) should be ~0 if initialized from exported cameras.
- **End** (`04_baneus_final_vs_geo`) shows training drift vs geo reference.


In [ ]:
# 14.2 — Pose audit: did training return to dataset poses?
import json
from pathlib import Path

log_p = POSE_AUDIT_DIR / "pose_log.jsonl"
rows = []
if log_p.exists():
    for line in log_p.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))

# rows that matter for your experiment (vs geo JSON cams/ GT)
KEY = [e for e in rows if e.get("stage", "").startswith(("02_", "03_", "04_"))]

def _fmt(e):
    vs = e.get("vs_ref") or {}
    return vs.get("rot_deg_max"), vs.get("center_max")

start = next((e for e in rows if e.get("stage", "").endswith("_start")), None)
final = next((e for e in rows if "final_vs_geo" in e.get("stage", "") or "final_vs_init" in e.get("stage", "")), None)

if not rows:
    print("No pose log yet — run §2.6 → §12 → §13.3 first.")
else:
    print("Reference = geo JSON ENU positions (§2.6) or exported init poses\n")
    print("(01_da3_vs_geo compares DA3 centers to jsons/ after geo align.)\n")
    print(f"{'stage':<36} {'rot° max':>10} {'center max':>12}  notes")
    print("-" * 72)
    for e in KEY or rows:
        vs = e.get("vs_ref") or {}
        if not vs:
            continue
        notes = e.get("extra", {}).get("camera_src", "")
        print(f"{e['stage']:<36} {vs.get('rot_deg_max', float('nan')):10.4f} {vs.get('center_max', float('nan')):12.6f}  {notes}")

    exp = next((e.get("extra", {}).get("camera_src") for e in rows if e.get("stage") == "02_neus_export"), None)
    if exp == "da3":
        print("\n⚠ 02_neus_export used DA3 cameras — re-run §12.1 with USE_GEO_REF=True")

    print()
    if start and final:
        sr, sc = _fmt(start)
        fr, fc = _fmt(final)
        print(f"At train START vs geo JSON reference: rot max={sr:.4f}°  center max={sc:.6f}")
        print(f"At train END   vs geo JSON reference: rot max={fr:.4f}°  center max={fc:.6f}")
        if fr is not None and sr is not None:
            if fr < 0.1 and (fc or 0) < 1e-2:
                print("\n✓ Poses match Cathedral dataset ground truth at end.")
            elif (sr or 0) > 90 or (fr or 0) > 90:
                print("\n⚠ ~180° error usually means convention mismatch in this table.")
                print("  Run §14.3 for an honest decode-based comparison.")
            elif fr > sr + 0.5 or (fc or 0) > (sc or 0) * 1.5 + 0.01:
                print("\n✗ Poses drifted away from dataset ground truth during training.")
            else:
                print("\n~ Small movement — inspect table above.")
    print("\nFull log:", log_p)

if Path("/content/drive/MyDrive").exists():
    import shutil
    dst = DRIVE_OUT / "pose_audit"; dst.mkdir(parents=True, exist_ok=True)
    if log_p.exists():
        shutil.copy2(log_p, dst / "pose_log.jsonl")
    snap = POSE_AUDIT_DIR / "snapshots"
    if snap.is_dir():
        for f in snap.glob("*.npz"):
            shutil.copy2(f, dst / f.name)
    print("Copied to Drive:", dst)


## 14.3 — Honest pose check (run anytime)

Run this **after §2.6 + §11.1 + §12** (and **§13** for the trained row). No retrain needed.

**Important:** `pose_network(i)` is **not** the stored camera. BA-NeuS computes
`make_c2w(MLP(i)) @ init_c2w[i]`. Rays use that product. This cell separates:

1. **GT** (`cams/`)
2. **§12 export** / **`cameras_sphere.npz` decoded** — on-disk cameras (should match GT)
3. **`pose_all_gt` / `init_c2w`** — what Dataset loads from the npz
4. **`pose_network()`** — runtime poses (MLP delta applied; ~180° vs GT at init unless zero-init patch ran)
5. **Trained vs @init** — actual learn_pose drift during training
6. **Histograms** — per-view rotation/center error (mean, median, max marked)


In [ ]:
# 14.3 — Honest pose check (run now; needs §2.6 + §11.1 + §11.2 (or §12) in kernel; §13 for checkpoint row)
import os, sys
import numpy as np
import torch
from pathlib import Path

assert 'view_names' in dir(), "Run §2.6 and §3 first"
assert 'summarize_pose_errors' in dir(), "Run §2.6 (pose helpers) first"

sys.path.insert(0, str(BANEUS_ROOT))
os.chdir(BANEUS_ROOT)
from utils.camera_utils import load_K_Rt_from_P

def _decode_c2ws_from_neus_npz(npz_path, n_views):
    cam = np.load(npz_path)
    out = []
    for i in range(n_views):
        P = cam[f"world_mat_{i}"] @ cam[f"scale_mat_{i}"]
        _, c2w = load_K_Rt_from_P(None, P[:3, :4].astype(np.float64))
        out.append(np.asarray(c2w, np.float64))
    return np.stack(out)

def _invert_c2ws(c2ws):
    return np.stack([np.linalg.inv(c) for c in c2ws])

def _report(label, est, ref, ref_name="ref"):
    s, per_view = summarize_pose_errors(est, ref, label=label, align_sim3=False)
    print(
        f"  {label:40s}  rot mean={s['rot_deg_mean']:.4f}°  max={s['rot_deg_max']:.4f}°"
        f"  |  center mean={s['center_mean']:.6f}  max={s['center_max']:.6f}  (vs {ref_name})"
    )
    return s, per_view

def _plot_pose_error_hists(series, out_name="pose_error_histograms.png"):
    """series: [(title, per_view), ...] from summarize_pose_errors."""
    import matplotlib.pyplot as plt
    n = len(series)
    fig, axes = plt.subplots(n, 2, figsize=(10, 3.2 * n), squeeze=False)
    for row, (title, per_view) in enumerate(series):
        rot = np.array([m["rot_deg"] for m in per_view])
        cent = np.array([m["center_l2"] for m in per_view])
        specs = [
            (0, rot, "rotation error (deg)", "#4C72B0"),
            (1, cent, "center error (norm units)", "#55A868"),
        ]
        for col, vals, xlabel, color in specs:
            ax = axes[row, col]
            nb = min(30, max(10, len(vals) // 4))
            ax.hist(vals, bins=nb, color=color, alpha=0.85, edgecolor="white")
            ax.axvline(vals.mean(), color="k", ls="--", lw=1, label=f"mean={vals.mean():.3f}")
            ax.axvline(np.median(vals), color="k", ls=":", lw=1, label=f"median={np.median(vals):.3f}")
            ax.axvline(vals.max(), color="r", ls="-.", lw=1, label=f"max={vals.max():.3f}")
            ax.set_title(f"{title}")
            ax.set_xlabel(xlabel)
            ax.set_ylabel("# views")
            ax.legend(fontsize=8, loc="upper right")
    fig.tight_layout()
    out = POSE_AUDIT_DIR / out_name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved histogram -> {out}")

def _stored_init_c2ws(ds):
    """LearnPose init anchor, or dataset.pose_all when init_c2w is None."""
    ic = getattr(ds.pose_network, "init_c2w", None)
    if ic is not None:
        return ic.detach().cpu().numpy()
    return ds.pose_all.detach().cpu().numpy()

def _stored_gt_c2ws(ds):
    return ds.pose_all_gt.detach().cpu().numpy()

def _find_baseline_npz():
    cands = [
        POSE_AUDIT_DIR / "neus_export_baseline.npz",
        OUTPUT_DIR / "neus_export_baseline.npz",
    ]
    if globals().get("USE_PREPROCESSED") and globals().get("PREPROCESSED_DIR"):
        cands.append(Path(PREPROCESSED_DIR) / "meta" / "neus_export_baseline.npz")
    for p in cands:
        if p.exists():
            return p
    return None

def _to_norm_c2w(c2ws_world, center, radius):
    """World-frame c2w -> NeuS normalized frame (center'=(C-center)/radius, R unchanged)."""
    out = np.asarray(c2ws_world, np.float64).copy()
    out[..., :3, 3] = (out[..., :3, 3] - center) / radius
    return out

n = len(view_names)
npz_path = BANEUS_ROOT / "public_data" / CASE_NAME / "cameras_sphere.npz"
assert npz_path.exists(), f"Missing {npz_path} — run §11.2 (preprocessed) or §12.1"

# scale_mat encodes the normalization: norm->world (diag=radius, translation=+center)
_cam0 = np.load(npz_path)
_sm = _cam0["scale_mat_0"].astype(np.float64)
_radius = float(_sm[0, 0]); _center = _sm[:3, 3]
print(f"Normalization from cameras_sphere: center={_center.round(3)} radius={_radius:.4f}\n")

# GT / reference poses in the SAME normalized frame the network operates in
if HAS_GEO_JSON:
    ref_world = np.stack([GEO_REF[name]["c2w"] for name in view_names])
    ref_norm = _to_norm_c2w(ref_world, _center, _radius)
    _ref_name = "GT_norm"
    print("Reference: BlendedMVS cams/ GT (normalized into NeuS frame)\n")
else:
    bl_path = _find_baseline_npz()
    if bl_path is not None:
        ref_world = np.load(bl_path)["c2ws"]
        ref_norm = _to_norm_c2w(ref_world, _center, _radius)
        _ref_name = "init_export_norm"
        print(f"Reference: exported init poses from {bl_path.name} (va_split has no GT cams/)\n")
    else:
        ref_norm = None
        _ref_name = "decoded"
        print("Reference: decoded cameras_sphere.npz (no GT cams/, no export baseline)\n")

decoded = _decode_c2ws_from_neus_npz(npz_path, n)  # already normalized
if ref_norm is not None:
    s_dec_gt, _ = _report("cameras_sphere decoded", decoded, ref_norm, _ref_name)
else:
    s_dec_gt = {"rot_deg_max": 0.0}
    print("  (skipped decoded-vs-ref — no baseline; run §12.1 first)")

bl_path = _find_baseline_npz()
if bl_path is not None and HAS_GEO_JSON:
    bl = np.load(bl_path)
    export_c2ws = bl["c2ws"]  # stored in world frame
    print(f"  (baseline: {bl_path.name})")
    _report("export baseline (world->norm)", _to_norm_c2w(export_c2ws, _center, _radius), ref_norm, _ref_name)
    _report("export vs decoded npz", _to_norm_c2w(export_c2ws, _center, _radius), decoded, "decoded")
elif bl_path is not None and not HAS_GEO_JSON:
    print(f"  (baseline: {bl_path.name} — used as reference above)")
else:
    print("  (no neus_export_baseline.npz — optional)")

# Fresh Dataset — separates stored cameras from pose_network() forward
import importlib
import models.camera as _cammod
importlib.reload(_cammod)
from pyhocon import ConfigFactory
from dataset import dataset as _dsmod
importlib.reload(_dsmod)
from dataset.dataset import Dataset
conf = ConfigFactory.parse_file(str(BANEUS_ROOT / "configs" / "ba_aerial_s1prior.conf"))
print(f"Config: init_pose={conf.get_bool('dataset.init_pose')}  learn_pose={conf.get_bool('dataset.learn_pose')}  noise={conf.get_float('dataset.noise_magnitude')}")
if conf.get_bool('dataset.learn_pose') and not conf.get_bool('dataset.init_pose'):
    print("⚠ learn_pose=True but init_pose=False — LearnPose has no init_c2w anchor; rays will be random at step 0.")
    print("  Re-run §13.1 with BANEUS_INIT_POSE=True (or set learn_pose=False for frozen-GT baseline).")
ds0 = Dataset(conf["dataset"])
print(f"Dataset.init_pose={ds0.init_pose}  Dataset.learn_pose={ds0.learn_pose}")
_pn = ds0.pose_network
print(f"pose_network.init_c2w is None: {_pn.init_c2w is None}")
if _pn.init_c2w is not None:
    print(f"lin3 |weight|_max: {_pn.lin3.weight.abs().max().item():.3e}  (expect 0 for identity start)")
stored = _stored_init_c2ws(ds0)
pose_gt = _stored_gt_c2ws(ds0)
runtime_init = np.stack([ds0.pose_network(i).detach().cpu().numpy() for i in range(n)])

print("\nStored cameras in Dataset (what cameras_sphere.npz encodes):")
_report("pose_all_gt (decoded, no MLP)", pose_gt, decoded, "decoded")
_report("init base (pose_all)", stored, decoded, "decoded")

print("\npose_network() forward — make_c2w(MLP) @ init_c2w (rays use THIS):")
_, pv_init = _report("pose_network @init", runtime_init, decoded, "decoded")
_report("pose_network @init", runtime_init, stored, "stored init_c2w")

ckpt_dir = BANEUS_ROOT / "exp" / CASE_NAME / "ba_aerial" / "checkpoints"
ckpts = sorted(ckpt_dir.glob("ckpt_*.pth")) if ckpt_dir.exists() else []
pv_trained = None
if ckpts:
    ds = Dataset(conf["dataset"])
    ckpt = torch.load(ckpts[-1], map_location="cuda", weights_only=False)
    ds.pose_network.load_state_dict(ckpt["pose_network"])
    if "intrinsic_network" in ckpt:
        ds.intrinsic_network.load_state_dict(ckpt["intrinsic_network"])
    net_c2ws = np.stack([ds.pose_network(i).detach().cpu().numpy() for i in range(n)])
    print(f"\nCheckpoint: {ckpts[-1].name} (iter {ckpt.get('iter_step', '?')})")
    _, pv_trained = _report("pose_network (trained)", net_c2ws, decoded, "decoded")
    _report("pose_network drift (trained vs @init)", net_c2ws, runtime_init, "@init")
else:
    print("\n(no checkpoints yet — run §13.3 for trained row)")
    net_c2ws = None

# Per-view error distributions (mean / median / max marked on histograms)
_hist = [("@init vs decoded (GT)", pv_init)]
if pv_trained is not None:
    _hist.append((f"trained vs decoded (GT) — {ckpts[-1].name}", pv_trained))
print("\nPer-view pose error histograms (vs decoded / GT):")
_plot_pose_error_hists(_hist)

print()
# --- verdict ---
export_ok = s_dec_gt["rot_deg_max"] < 0.5
stored_ok = summarize_pose_errors(stored, decoded)[0]["rot_deg_max"] < 0.5
runtime_init_ok = summarize_pose_errors(runtime_init, decoded)[0]["rot_deg_max"] < 0.5

if HAS_GEO_JSON:
    if export_ok:
        print("✓ Export / cameras_sphere.npz match GT — on-disk cameras are correct.")
    else:
        print("✗ cameras_sphere.npz still off vs GT — re-run §12.1 with USE_GEO_REF=True.")
else:
    if bl_path is not None and export_ok:
        print("✓ Export / cameras_sphere.npz match init export baseline.")
    elif bl_path is not None:
        print("✗ cameras_sphere.npz off vs init export — re-run §12.1.")
    else:
        print("(va_split: no GT cams — compare decoded vs @init / trained rows above)")

if stored_ok:
    print("✓ Dataset stores decoded cameras in init_c2w / pose_all_gt.")
else:
    print("✗ Dataset init_c2w differs from decoded — check noise_magnitude and §11.1 patches.")

if not runtime_init_ok:
    print("⚠ pose_network() ≠ decoded even before training.")
    if _pn.init_c2w is None:
        if conf.get_bool('dataset.init_pose'):
            print("  Cause: dataset.py pose patch bug (CATHEDRAL_BROKEN_POSE_ELSE) — re-run §11.1 to repair.")
        else:
            print("  Cause: init_pose=False in ba_aerial_s1prior.conf — LearnPose is not anchored to exported cameras.")
            print("  Fix: re-run §13.1 with BANEUS_INIT_POSE=True, or set BANEUS_LEARN_POSE=False for frozen GT cameras.")
    elif _pn.lin3.weight.abs().max().item() > 1e-8:
        print("  Cause: pose MLP lin3 is not zero-initialized (|w|_max > 0).")
        print("  Fix: re-run §11.1 (CATHEDRAL_FORCE_POSE_ZEROINIT), restart kernel, then §14.3 again.")
    else:
        print("  lin3 is zero but forward still wrong — report this output (possible BA-NeuS indexing bug).")
elif net_c2ws is not None:
    drift = summarize_pose_errors(net_c2ws, runtime_init)[0]
    if drift["rot_deg_max"] < 0.5 and drift["center_max"] < 1e-2:
        print("✓ Training barely moved poses (learn_pose had little effect).")
    else:
        print(f"  Training moved poses vs @init: rot_max={drift['rot_deg_max']:.4f}°  center_max={drift['center_max']:.6f}")
